In [1]:
# Cell 1 — Cài đặt thư viện
# !pip install -q bm25s underthesea sentence-transformers==3.* accelerate tqdm peft bitsandbytes
# !pip uninstall -y transformers tokenizers
# !pip install -q "transformers>=4.51.0,<5.0" "tokenizers<0.22"
# !pip install "torch<2.14" --index-url https://download.pytorch.org/whl/cu128
# !pip install -q flash-attn --no-build-isolation  # (tuỳ chọn) cần cho Flash Attention 2, xem USE_FLASH_ATTENTION_2 ở Cell 2

In [ ]:
# Cell 1 - Chạy đầu tiên sau khi Restart Kernel
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
# ĐÃ THÊM (chống OOM): PHẢI set biến này TRƯỚC "import torch" (allocator CUDA của PyTorch chỉ đọc
# biến này 1 LẦN lúc khởi tạo) -- giảm phân mảnh VRAM (fragmentation), đúng gợi ý ngay trong thông
# báo lỗi "torch.OutOfMemoryError" ("If reserved but unallocated memory is large try setting
# PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True"). Không tốn thêm VRAM, chỉ đổi cách allocator
# cấp phát/tái sử dụng bộ nhớ, nên an toàn để luôn bật.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
print(torch.cuda.device_count()) # Lúc này chắc chắn sẽ ra 1

1


/mnt/mmlab2024nas/trantran/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [3]:
import os, torch
print(os.environ.get("CUDA_VISIBLE_DEVICES"))
print(torch.cuda.device_count())          # số GPU mà process THẤY
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

0
1
0 NVIDIA RTX A5000


In [4]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

In [ ]:
# Cell 2 — Imports & Config
import os, json, glob, random, zipfile, re

import numpy as np
import torch
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ---- Flash Attention 2 (tuỳ chọn, tăng tốc + giảm VRAM cho attention nếu GPU hỗ trợ) ----
# Yêu cầu: pip install flash-attn --no-build-isolation (xem Cell 1 phía trên) và GPU kiến trúc
# Ampere trở lên (>= SM80, vd A100/A6000/RTX 30xx/40xx...). Không hỗ trợ trên GPU cũ hơn (Turing/Volta).
# Đặt False nếu GPU không hỗ trợ hoặc chưa cài được flash-attn -- các cell load model bên dưới sẽ tự
# fallback về attention mặc định của transformers (SDPA) nếu bật mà thiếu package.
USE_FLASH_ATTENTION_2 = True

def _attn_impl_kwargs():
    """Trả về {"attn_implementation": "flash_attention_2"} nếu USE_FLASH_ATTENTION_2=True và đã
    cài được package 'flash-attn', ngược lại trả về dict RỖNG (dùng attention mặc định, vd SDPA)
    -- dùng **_attn_impl_kwargs() khi gọi .from_pretrained()/SentenceTransformer(...) để bật/tắt
    đồng loạt ở MỌI nơi load model (embed_model, reranker_model, qa_model) chỉ bằng 1 biến duy nhất."""
    if not USE_FLASH_ATTENTION_2:
        return {}
    try:
        import flash_attn  # noqa: F401
    except ImportError:
        print("[CẢNH BÁO] USE_FLASH_ATTENTION_2=True nhưng chưa cài package 'flash-attn' "
              "(pip install flash-attn --no-build-isolation, xem Cell 1) -> fallback về attention "
              "mặc định (SDPA) cho lần load model này.")
        return {}
    return {"attn_implementation": "flash_attention_2"}

# ---- Chỉnh lại các đường dẫn này cho khớp với dataset trên Kaggle ----
DATA_DIR = "DSC2026-IntelliJQK/data/raw"      # thư mục chứa train.json, warmup.json, ...

# ĐÃ SỬA: corpus giờ CHỈ nhận output ĐÃ TIỀN XỬ LÝ của pipeline 3 file main.py/preprocess.py/
# md_to_json.py -> thư mục "processed-contexts/", mỗi file context_*.json có field
# "passage_segments": list[dict] {"path", "content", "segment_type"}, KHÔNG còn field "passage"
# thô nữa. Corpus context_*.json/selected-contexts.zip CŨ (chưa qua pipeline này) KHÔNG còn
# được dùng trực tiếp — xem load_corpus() ở Cell 4.
CONTEXT_DIR = "DSC2026-IntelliJQK/data/processed/processed-contexts"        # thư mục chứa output của DSC-preprocessing (processed-contexts/context_*.json)
CONTEXTS_ZIP = "DSC2026-IntelliJQK/data/processed/processed-contexts.zip"   # (tuỳ chọn) nếu để dạng zip, sẽ tự giải nén ở Cell 3

TRAIN_PATH = os.path.join(DATA_DIR, "train.json")
WARMUP_PATH = os.path.join(DATA_DIR, "warmup.json")
# ĐÃ SỬA: public-official CŨ (chỉ có document_id gold, dùng để đánh giá Stage-1/Stage-2 retrieval)
# được thay bằng public-official2 (có câu hỏi thật để Stage-3 sinh câu trả lời cuối cùng nộp bài).
# Nếu bộ đề chính thức vẫn dùng tên cũ, chỉ cần đổi lại thành "public-official.json" ở đây.
PUBLIC_TEST_PATH = os.path.join(DATA_DIR, "public-official2.json")
PRIVATE_TEST_PATH = os.path.join(DATA_DIR, "private-official2.json")

# ĐÃ THÊM: train2.json — dữ liệu để fine-tune VietAI/vit5-large (Stage-3), KHÁC với train.json
# (dùng để fine-tune reranker ở Stage-2). Định dạng train2.json:
#   {"<qid>": {"question": "...", "answer": "<câu trả lời tham chiếu, dạng chuỗi>"}, ...}
# (answer ở đây là CÂU TRẢ LỜI THẬT, không phải list document_id như train.json).
TRAIN2_PATH = os.path.join(DATA_DIR, "train2.json")

TOP_K_STAGE1 = 30     # số ứng viên lấy ra ở stage-1 (BM25 / dense) để đưa vào reranker
TOP_K_FINAL = 5       # số document_id trả về cuối cùng (ràng buộc cứng của đề bài)
NEG_PER_POS = 5        # tỉ lệ negative:positive khi mining để fine-tune reranker (5:1)

# ---- Mining negative TRỰC TIẾP trên SHORT-chunk BM25 index đã build sẵn ở Stage-1 (Cell 7/8) ----
# Không còn build BM25 index riêng trên long-chunk, và không còn pre-tokenize (underthesea) toàn
# bộ short-chunk của corpus nữa (đây là nguyên nhân chính gây OOM/CPU nặng trước đây, vì phải
# tokenize long-chunk của TOÀN BỘ corpus dù chỉ một phần nhỏ thực sự được dùng làm training pair).
# Cách mới (xem Section 4): với mỗi câu hỏi, dùng lại đúng bm25_scores_all() (short-chunk, đã có
# sẵn từ Stage-1) để tìm (a) short-chunk điểm cao nhất trong document gold (positive) và (b) các
# document khác có short-chunk điểm cao nhất (hard negative, tối đa NEG_MINING_POOL_K document khác
# nhau). Sau đó dùng trực tiếp short-chunk trúng tuyển
# (Cell 10c) — hàm này gọi thẳng short-chunk trực tiếp (Cell 6b) để ghép short-chunk trúng tuyển
# với path của chính nó ("path | short-chunk"), ĐÚNG Y HỆT cách
# select_relevant_chunks (Cell 6b) chọn short-chunk lúc inference thật (Cell 18) — training pairs và
# inference giờ dùng CHUNG một cách tạo long-chunk, không còn lệch nhau nữa.
# Vì giờ rẻ hơn nhiều, có thể tăng NEG_MINING_POOL_K (vd. 15-20) để có nhiều lựa chọn resample negative
# đa dạng hơn giữa các epoch mà không tốn thêm chi phí tokenize nào.
NEG_MINING_POOL_K = 100  # ĐÃ TĂNG (50 -> 100): pool tĩnh 50 document/câu hỏi bị tái sử
                          # dụng (chỉ re-score ở TẦNG 1.5, KHÔNG re-mine) suốt cả 3 epoch -> không
                          # gian negative quá hẹp so với corpus thật, dễ khiến model học thuộc đặc thù
                          # của đúng 50 candidate đó thay vì ranh giới tổng quát (nghi ngờ là một phần
                          # nguyên nhân dev recall giảm dần qua epoch). Tăng lên 100 hầu như KHÔNG tốn
                          # thêm chi phí ở TẦNG 1 (hybrid score đã tính cho TOÀN BỘ corpus, chỉ cắt
                          # top-K nhiều hơn từ mảng đã sort sẵn) — chi phí tăng thêm chỉ ở TẦNG 1.5
                          # (refresh_pool_scores forward pass thêm ~50 candidate/câu hỏi mỗi epoch),
                          # vẫn rẻ hơn hẳn TẦNG 1 (không có bm25.get_scores()/dense encode nào).
MINING_LAMBDA = 0.4      # trọng số BM25 trong HYBRID score dùng cho TOÀN BỘ candidate mining
                          # Score_mining = 0.4*BM25_norm + 0.6*Dense_norm

# ---- Config TẦNG 1.5 (MỚI) — refresh_pool_scores(): forward pass reranker (rẻ) trên candidate đã
# ---- có sẵn trong neg_pool (TẦNG 1) để cập nhật "độ khó hiện tại" theo checkpoint đang train. ----
POOL_SCORE_BATCH_SIZE = 16   # batch size khi rerank_score() toàn bộ neg_pool của 1 câu hỏi
NEG_HARD_TOP_FRAC = 0.6      # TẦNG 2 chỉ random.sample() negative trong top NEG_HARD_TOP_FRAC
                              # (theo điểm reranker MỚI NHẤT, TẦNG 1.5) của neg_pool -> ưu tiên hard-
                              # negative "khó" dưới góc nhìn hiện tại của model, nhưng vẫn random trong
                              # nhóm đó để có đa dạng giữa các epoch (không học tủ đúng 1 bộ cố định).
                              # Nếu qid chưa có điểm TẦNG 1.5 (chưa refresh) -> TẦNG 2 tự fallback về
                              # random.sample() trên toàn bộ neg_pool như hành vi gốc.

# (MỚI) MIX hard/easy negative thay vì chỉ đẩy khó dần: trước đây 100% n_neg lấy từ top
# NEG_HARD_TOP_FRAC (khó nhất theo điểm reranker HIỆN TẠI) -> có nguy cơ feedback loop (model tự
# chọn negative theo chính điểm số của nó, càng train càng thu hẹp trọng tâm vào 1 vùng hẹp, dễ
# mất khả năng tổng quát trên dev — nghi ngờ là nguyên nhân dev recall epoch cuối luôn thấp hơn).
# NEG_MIX_HARD_FRAC quyết định TỈ LỆ trong n_neg lấy từ hard pool; phần còn lại
# (1 - NEG_MIX_HARD_FRAC) lấy RANDOM ĐỀU trên toàn bộ eligible pool (bao gồm cả candidate "dễ"/hạng
# thấp), giữ model tiếp xúc với cả phổ độ khó thay vì chỉ vùng khó nhất mỗi epoch.
NEG_MIX_HARD_FRAC = 0.4      # 50% negative lấy từ hard pool (top NEG_HARD_TOP_FRAC), 50% còn lại
                              # random đều trên toàn bộ eligible pool (cả dễ lẫn khó).

# ---- Config lọc "nghi ngờ false-negative" bằng XÁC NHẬN CHÉO 2 nguồn ĐỘC LẬP (MỚI) ----
# Vấn đề: nếu chỉ dùng điểm reranker hiện tại (TẦNG 1.5) để loại negative "quá giống positive",
# model đang train sẽ tự quyết định xem candidate nào là false-negative -> feedback loop (model sai
# lại tự "xác nhận" cái sai đó, càng train càng lệch). Cách khắc phục: CHỈ loại 1 candidate ra khỏi
# neg_pool khi CẢ 2 nguồn ĐỘC LẬP cùng đồng thuận nó có khả năng là positive thật:
#   (a) điểm reranker hiện tại (TẦNG 1.5, model ĐANG train) rất cao, VÀ
#   (b) hybrid score gốc (BM25+Dense, TẦNG 1, KHÔNG phụ thuộc model đang train) cũng xếp hạng rất cao
#       trong neg_pool (neg_pool vốn đã được sort giảm dần theo hybrid score lúc mine — Cell 10c/
#       _hybrid_negative_pool — nên chỉ cần dùng VỊ TRÍ trong neg_pool làm proxy cho hybrid rank).
# Nếu chỉ 1 trong 2 nguồn "nghi ngờ" (vd. reranker cao nhưng hybrid rank thấp, hoặc ngược lại) ->
# GIỮ LẠI làm negative bình thường (thận trọng thiên về giữ, không thiên về loại) — tránh việc lọc
# quá tay làm mất hard-negative thật.
# LƯU Ý (ĐÃ SỬA): giá trị 0.9 dưới đây chỉ là NGƯỠNG KHỞI TẠO, dùng ở epoch 1 (lúc chưa có đủ
# known-positive score để tính percentile). Từ epoch 2 trở đi, Cell 14 tự ghi đè biến này bằng
# compute_dynamic_false_neg_threshold() (Cell 10c3) — percentile p10 (làm mượt qua EMA) của
# điểm reranker hiện tại trên known-positive, tự khớp theo độ tự tin thực tế của checkpoint ở
# từng epoch thay vì dùng mãi 1 số cố định.
FALSE_NEG_RERANK_THRESHOLD = 0.9   # (a) điểm reranker hiện tại (TẦNG 1.5) phải >= ngưỡng này
FALSE_NEG_HYBRID_TOP_FRAC = 0.3    # (b) VÀ candidate phải nằm trong top 30% ĐẦU neg_pool theo hybrid
                                    # rank gốc (TẦNG 1) — tức "được cả BM25/Dense lẫn reranker đồng
                                    # thuận là rất liên quan", không chỉ riêng model đang train nghĩ vậy.

# ---- Config Stage-1 HYBRID retrieval (BM25 + Dense, theo mục 2.2.1) ----
EMBED_MODEL_NAME = "AITeamVN/Vietnamese_Embedding_v2"   # dense encoder cho stage-1
EMBED_BATCH_SIZE = 32
LAMBDA_HYBRID = 0.4   # trọng số BM25 trong Score = lambda*BM25_norm + (1-lambda)*Dense_norm
                       # (giá trị mặc định, sẽ được tune lại trên tập dev ở Cell 8c)

# ---- Config Stage-2 filtering (mục 2.2.2): giữ nguyên reranker BAAI/bge-reranker-v2-m3, chỉ ----
# ---- thêm bước lọc theo threshold sau khi rerank.                                        ----
RERANK_TOP_N = 10        # chỉ xét top-10 document điểm reranker cao nhất trước khi lọc
RERANK_THRESHOLD = 0.99  # ngưỡng "tự tin cao": chỉ giữ candidate có điểm reranker > ngưỡng này
RERANK_FALLBACK_K = 2    # nếu không candidate nào vượt ngưỡng -> lấy top-2 theo điểm reranker
                          # (tránh trả về tập rỗng). Cả 3 giá trị trên sẽ được tune lại trên
                          # tập dev ở Cell 19b, sau khi đã có pipeline hoàn chỉnh.

# ---- Config chunking 2 TẦNG theo đúng paper ViDRILL ----
# Corpus có văn bản dài tới hàng triệu ký tự (median ~23.000, xem phân tích EDA), vượt xa context
# 2304 token của BAAI/bge-reranker-v2-m3 -> bắt buộc chunk trước khi đưa vào cả retrieval lẫn reranker.
# Cả 2 tầng đều dựa trên cấu trúc pháp lý phân cấp Điều -> Khoản -> Điểm (giữ nguyên logic cũ),
# chỉ khác nhau về kích thước mục tiêu, có/không overlap, và có/không được phép gộp nhiều đơn vị lại.

# Long chunking (Stage-2 / reranker): các đơn vị Điều/Khoản/Điểm liên tiếp được GỘP lại tới ~2000
# ký tự để cung cấp đủ ngữ cảnh cho cross-encoder. Overlap giữa 2 chunk liên tiếp = ĐÚNG 1 CÂU HOÀN
# CHỈNH (câu cuối của chunk trước lặp lại làm câu đầu chunk sau), không bao giờ cắt đứt giữa câu.
LONG_CHUNK_CHARS = 3000

# Short chunking (Stage-1 / retrieval): mỗi passage_segment (dict {"path", "content", "segment_type"}
# — output của md_to_json.py/preprocess.py, xem load_corpus() ở Cell 4) là 1 đơn vị ĐỘC LẬP, tối đa
# 2000 ký tự (ĐÃ TĂNG từ 1000 -> 2000; path prefix + content, riêng segment "structured" còn được
# gộp thêm root_preamble ở đầu — xem ROOT_PREAMBLE_CHARS và chunk_document_short_from_segments ở
# Cell 4b), KHÔNG gộp nhiều segment lại, KHÔNG overlap. Nội dung nào vượt quá ngân sách còn lại thì
# tách tiếp xuống Khoản rồi Điểm, và nếu xuống tới Điểm vẫn còn dài thì mới cắt theo câu hoàn chỉnh
# (clause-based segmentation, split_legal_unit_recursive — Cell 4b), giống hệt logic cũ, chỉ đổi
# ngân sách ký tự.
SHORT_CHUNK_CHARS = 2000  # NGAN SACH TONG toi da cua 1 short-chunk Stage-1 (content + path + root_preamble
                          # gop them, neu co) -- dung de dat ten file cache (Cell 7) va cho ham
                          # chunk_document_short() CU (khong con duoc goi trong luong chinh, chi giu lai
                          # de tuong thich nguoc). Luong chinh (chunk_document_short_from_segments -- Cell
                          # 4b) gio dung 3 ngan sach RIENG theo segment_type ben duoi, duoc thiet ke sao
                          # cho TONG DO DAI short-chunk cuoi cung cua MOI segment_type deu <= SHORT_CHUNK_CHARS
                          # (2000) nay:
                          #   - "root_preamble": content duoc cat toi da 2000 ky tu, KHONG gan them gi
                          #     (chinh no da la root_preamble cua document, khong can gop root_preamble
                          #     vao chinh no) -> tong toi da 2000.
                          #   - "unstructured_whole": content duoc cat toi da 1500 ky tu, sau do GAN
                          #     THEM ROOT_PREAMBLE_CHARS (500) ky tu root_preamble cua document vao DAU
                          #     -> tong toi da 1500 + 500 = 2000.
                          #   - "structured": content duoc cat toi da 1000 ky tu, GAN path (neu hop le,
                          #     < MAX_SEGMENT_PATH_CHARS = 500 ky tu) roi GAN THEM ROOT_PREAMBLE_CHARS
                          #     (500) ky tu root_preamble cua document vao DAU -> tong toi da
                          #     1000 + 500 (path) + 500 (root_preamble) = 2000. Neu path KHONG hop le
                          #     (>= MAX_SEGMENT_PATH_CHARS) thi path do da duoc GOP THANG vao content
                          #     TRUOC KHI cat (xem chunk_document_short_from_segments -- Cell 4b), nen
                          #     khong cong don them ngoai ngan sach 1000 ky tu content o tren.

# Ngan sach cat CONTENT (KHONG tinh path/root_preamble gan them) o Stage-1, rieng theo tung
# segment_type -- xem giai thich chi tiet o SHORT_CHUNK_CHARS phia tren va
# chunk_document_short_from_segments (Cell 4b).
SHORT_CHUNK_CHARS_ROOT_PREAMBLE = 2000    # segment_type == "root_preamble"
SHORT_CHUNK_CHARS_UNSTRUCTURED = 2000     # segment_type == "unstructured_whole" (+ 500 root_preamble gan them)
SHORT_CHUNK_CHARS_STRUCTURED = 1000       # segment_type == "structured" (+ path + 500 root_preamble gan them)


# Root-preamble dùng khi tạo short-chunk Stage-1: 1000 ký tự ĐẦU TIÊN của đoạn mở đầu văn bản trước
# heading đầu tiên (segment_type == "root_preamble", xem load_corpus() ở Cell 4) của DOCUMENT
# chứa short-chunk đại diện. Nếu có, đã nằm trong short-chunk ở Stage-1 (xem
# short-chunk trực tiếp — Cell 6b), CÙNG với path của short-chunk đại diện đó rồi mới tới
# chính short-chunk đại diện. Nếu document không có segment root_preamble -> chuỗi rỗng.
ROOT_PREAMBLE_CHARS = 500

# (MỚI) Khi short-chunk ĐẠI DIỆN của short-chunk chính là 1 mảnh thuộc segment_type
# "root_preamble" (đoạn mở đầu văn bản trước heading đầu tiên, KHÔNG có path/cấu trúc phân cấp
# Chương/Điều/Khoản), long-chunk không còn ghép root_preamble(1000 ký tự đầu) + path + short-chunk
# như trường hợp thường nữa (vì root_preamble không có path để bám vào) mà MỞ RỘNG NGỮ CẢNH ngay
# trong chính root_preamble: short-chunk = ROOT_PREAMBLE_EXTEND_CHARS ký tự TRƯỚC short-chunk đại
# diện + short-chunk đại diện + ROOT_PREAMBLE_EXTEND_CHARS ký tự SAU nó (xem short-chunk trực tiếp
# — Cell 6b). Các trường hợp short-chunk đại diện KHÔNG thuộc root_preamble giữ nguyên logic cũ.
ROOT_PREAMBLE_EXTEND_CHARS = 500

# "path" (vd "Chương I > Điều 2 > Khoản 1") bình thường chỉ vài chục ký tự. Nếu path DÀI HƠN
# ngưỡng này (bất thường — coi là NHIỄU của bước preprocessing, ví dụ tách sai path/content ở
# md_to_json.py) thì KHÔNG dùng nó làm prefix cấu trúc nữa: gộp thẳng path đó vào phần content
# (coi như text thường), rồi áp dụng ĐÚNG quy tắc cắt như content — chỉ cắt khi TỔNG vượt quá
# SHORT_CHUNK_CHARS (1000), luôn cắt tại ranh giới câu hoàn chỉnh — xem
# chunk_document_short_from_segments ở Cell 4b. Path vẫn được đưa đi retrieve bình thường, chỉ
# không còn được ưu tiên giữ nguyên như 1 prefix cấu trúc nữa.
MAX_SEGMENT_PATH_CHARS = 500

MAX_CHUNKS_PER_DOC_FOR_RERANK = 1   # số short-chunk liên quan nhất/doc được đưa vào reranker khi rerank ứng viên
TOP_K_STAGE1_CHUNKS = 300           # số short-chunk thô lấy từ BM25 trước khi gộp điểm về cấp document

# ---- Giới hạn số tiến trình song song (server DÙNG CHUNG với người khác) ----
# KHÔNG dùng hết toàn bộ CPU (os.cpu_count()) cho các bước tokenize/mining song song hoá
# (Cell 6a, Cell 7, Cell 11) — mặc định chỉ dùng tối đa MAX_WORKERS tiến trình, chừa lại core cho
# các job khác đang chạy chung server. Có thể override bằng biến môi trường MAX_WORKERS trước khi
# start kernel (vd. `export MAX_WORKERS=8`), hoặc chỉnh số bên dưới tuỳ mức độ rảnh của server.
MAX_WORKERS = int(os.environ.get("MAX_WORKERS", 4))


def get_n_workers(cap: int = MAX_WORKERS) -> int:
    """Số tiến trình song song an toàn: không vượt quá cap và không vượt quá (tổng CPU - 1)."""
    return max(1, min(cap, (os.cpu_count() or 1) - 1))


def _lower_priority():
    """Initializer cho worker: hạ priority (nice) của tiến trình con để nhường CPU cho các job
    khác đang chạy cùng server, thay vì cạnh tranh CPU ngang hàng với chúng."""
    try:
        os.nice(10)
    except (AttributeError, OSError):
        pass  # os.nice không có trên Windows, hoặc không đủ quyền chỉnh nice -> bỏ qua


Device: cuda


/mnt/mmlab2024nas/trantran/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load dữ liệu (queries + corpus)

In [6]:
# Cell 3 — Giải nén corpus ĐÃ TIỀN XỬ LÝ (processed-contexts.zip, nếu để dạng zip) và load train.json
if not os.path.isdir(CONTEXT_DIR) and os.path.exists(CONTEXTS_ZIP):
    os.makedirs(CONTEXT_DIR, exist_ok=True)
    with zipfile.ZipFile(CONTEXTS_ZIP, "r") as zf:
        zf.extractall(CONTEXT_DIR)
    print("Đã giải nén processed-context corpus vào", CONTEXT_DIR)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

# train.json có dạng: {"<qid>": {"question": "...", "answer": ["177504", ...]}, ...}
train_raw = load_json(TRAIN_PATH)
print("Số câu hỏi trong train:", len(train_raw))

sample_qid = next(iter(train_raw))
print(sample_qid, "->", train_raw[sample_qid])


Số câu hỏi trong train: 7000
86666 -> {'question': 'Thời hạn cấp đăng ký xe máy của người nước ngoài làm việc tại Việt Nam là bao lâu?', 'answer': ['280282']}


In [ ]:
# Cell 4 — Load toàn bộ corpus ĐÃ TIỀN XỬ LÝ (processed-contexts/*.json, output THẬT của pipeline
# 3 file main.py -> preprocess.py -> md_to_json.py) vào dict: doc_id (str) -> {"name":..., "link":...,
# "passage_segments": [...]}. KHÔNG còn dùng field "passage" (văn bản thô) nữa.
#
# ĐÃ SỬA (khớp đúng build_segments_json/md_to_segments trong md_to_json.py): mỗi phần tử của
# "passage_segments" giờ là 1 DICT {"path": str, "content": str, "segment_type": str}, KHÔNG còn là
# string dạng "path | content" nữa. "segment_type" chỉ nhận 1 trong 3 giá trị mà md_to_json.py sinh
# ra: "structured" (segment thường, có path phân cấp Chương/Mục/Điều/Khoản/Điểm), "root_preamble"
# (đoạn mở đầu văn bản trước heading đầu tiên, path rỗng) và "unstructured_whole" (toàn văn bản không
# có heading nào, path rỗng). Cả 3 loại đều mang nội dung thật (không phải cấu trúc rỗng) nên đều
# được giữ lại để đưa vào Stage-1 — xem ALLOWED_SEGMENT_TYPES bên dưới.

from typing import Optional

ALLOWED_SEGMENT_TYPES = {"structured", "root_preamble", "unstructured_whole"}

_RE_MULTI_SPACE = re.compile(r" {2,}")   # nhiều dấu cách (" ") liên tiếp
_RE_LONG_DOTS = re.compile(r"\.{5,}")    # dãy dấu chấm "." DÀI HƠN 4 ký tự (tức >= 5 dấu liền nhau)
# Dãy dấu gạch ngang "-" DÀI HƠN 3 ký tự liên tiếp (tức >= 4 dấu liền nhau, vd "--------") -> rút
# gọn còn ĐÚNG 3 ký tự ("---"). Ngưỡng >=4 (thay vì >=2) để KHÔNG đụng tới các dấu "-" đơn lẻ hay
# đôi vốn thường xuất hiện trong công thức/phép trừ/khoảng số (vd "a - b", "5-10", "Điều 5-7").
_RE_LONG_DASHES = re.compile(r"-{4,}")
# Chuỗi quốc hiệu/tiêu ngữ lặp lại ở đầu rất nhiều văn bản pháp luật, không mang giá trị retrieval
# -> xoá hẳn. Dùng \s+ giữa các từ (thay vì khoảng trắng literal) để khớp cả khi có xuống dòng hoặc
# nhiều khoảng trắng xen giữa (trước khi _RE_MULTI_SPACE chạy), tránh xoá sót do lệch whitespace.
_RE_NATIONAL_HEADER = re.compile(
    r"CỘNG\s+HÒA\s+XÃ\s+HỘI\s+CHỦ\s+NGHĨA\s+VIỆT\s+NAM\s+"
    r"Độc\s+lập\s*-\s*Tự\s+do\s*-\s*Hạnh\s+phúc\s*",
    re.UNICODE,
)


def _clean_text(text: str) -> str:
    """Tiền xử lý bổ sung cho MỖI field text (path hoặc content) lấy từ processed-context:
    1) Xoá chính xác chuỗi quốc hiệu/tiêu ngữ "CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do -
       Hạnh phúc " (nếu có) — không mang thông tin pháp lý, chỉ là header lặp lại ở đầu văn bản.
    2) Gộp nhiều dấu cách liên tiếp ("    ", ...) thành đúng 1 dấu cách " ".
    3) Dãy dấu gạch ngang dài hơn 3 ký tự (vd "----------", dùng để ngăn cách hình thức trong văn
       bản gốc) -> rút gọn còn ĐÚNG 3 ký tự ("---"). Chỉ áp dụng cho dãy >=4 dấu "-" liền nhau nên
       KHÔNG đụng tới dấu "-" đơn/đôi trong công thức, phép trừ, khoảng số (vd "a - b", "5-10").
    4) Dãy dấu chấm dài hơn 4 ký tự (vd "......") -> rút gọn còn ĐÚNG 3 ký tự ("...").
    """
    text = _RE_NATIONAL_HEADER.sub("", text or "")
    text = _RE_MULTI_SPACE.sub(" ", text)
    text = _RE_LONG_DASHES.sub("---", text)
    text = _RE_LONG_DOTS.sub("...", text)
    return text.strip()


def clean_passage_segment(seg: dict) -> Optional[dict]:
    """Làm sạch 1 passage_segment dict {"path", "content", "segment_type"} (output của
    md_to_json.py). Trả về None nếu segment_type không hợp lệ hoặc content rỗng sau khi clean
    (segment không mang thông tin gì để đi retrieve)."""
    if not isinstance(seg, dict):
        return None
    if seg.get("segment_type") not in ALLOWED_SEGMENT_TYPES:
        return None
    content = _clean_text(seg.get("content", ""))
    if not content:
        return None
    return {
        "path": _clean_text(seg.get("path", "")),
        "content": content,
        "segment_type": seg["segment_type"],
    }


def _extract_root_preamble(segments):
    """Nối nội dung các segment "root_preamble" (đoạn mở đầu văn bản trước heading đầu tiên,
    xem ALLOWED_SEGMENT_TYPES ở trên) của 1 document, rồi cắt lấy ĐÚNG ROOT_PREAMBLE_CHARS
    (1000) ký tự đầu tiên. Nếu có, đã nằm trong short-chunk ở Stage-1 (short-chunk trực tiếp,
    Cell 6b). Trả về "" nếu document không có segment root_preamble nào (vd. văn bản
    "unstructured_whole" hoặc không có đoạn mở đầu trước heading)."""
    parts = [s["content"] for s in (segments or []) if s.get("segment_type") == "root_preamble" and s.get("content")]
    if not parts:
        return ""
    return " ".join(parts)[:ROOT_PREAMBLE_CHARS]


def _extract_root_preamble_full(segments):
    """Giống _extract_root_preamble() ở trên nhưng KHÔNG cắt còn ROOT_PREAMBLE_CHARS — trả về TOÀN
    BỘ nội dung root_preamble (nối các segment segment_type == "root_preamble" bằng dấu cách).
    Dùng làm nguồn để MỞ RỘNG NGỮ CẢNH (500 ký tự trước/sau) khi short-chunk đại diện của long-chunk
    chính là 1 mảnh thuộc root_preamble (xem ROOT_PREAMBLE_EXTEND_CHARS ở Cell 2 và
    short-chunk trực tiếp ở Cell 6b) — root_preamble không có path/cấu trúc phân cấp nên không thể
    bám theo path như các segment "structured" khác, phải mở rộng trực tiếp trong chính văn bản
    root_preamble. Trả về "" nếu document không có segment root_preamble nào."""
    parts = [s["content"] for s in (segments or []) if s.get("segment_type") == "root_preamble" and s.get("content")]
    if not parts:
        return ""
    return " ".join(parts)


def load_corpus(context_dir):
    corpus = {}
    files = sorted(glob.glob(os.path.join(context_dir, "**", "context_*.json"), recursive=True))
    assert len(files) > 0, (
        f"Không tìm thấy file context_*.json (processed-context) trong {context_dir}. "
        f"Corpus giờ CHỈ nhận output của pipeline main.py/preprocess.py/md_to_json.py (thư mục "
        f"processed-contexts/, mỗi file có field \"passage_segments\": list[dict]), KHÔNG còn dùng "
        f"corpus context_*.json thô (field \"passage\") nữa."
    )
    for fp in tqdm(files, desc="Loading processed-context corpus files"):
        data = load_json(fp)
        # một file context_*.json có thể chứa 1 văn bản (dict) hoặc nhiều văn bản (list)
        items = data if isinstance(data, list) else [data]
        for item in items:
            doc_id = str(item["id"])
            raw_segments = item.get("passage_segments", []) or []
            segments = [s for s in (clean_passage_segment(seg) for seg in raw_segments) if s]
            corpus[doc_id] = {
                "name": item.get("name", ""),
                "passage_segments": segments,
                "link": item.get("link", ""),
            }
    return corpus

corpus = load_corpus(CONTEXT_DIR)
print("Số văn bản trong corpus (trước khi lọc):", len(corpus))

# ---- Lọc các văn bản không còn passage_segments nào (rỗng), tương đương bước lọc "passage rỗng" cũ ----
empty_doc_ids = [d for d, v in corpus.items() if not v["passage_segments"]]
if empty_doc_ids:
    gold_ids_all = set()
    for v in train_raw.values():
        gold_ids_all.update(str(a) for a in v["answer"])
    overlap_empty_gold = set(empty_doc_ids) & gold_ids_all
    print(f"Cảnh báo: {len(empty_doc_ids)} văn bản có passage_segments RỖNG -> sẽ loại khỏi corpus.")
    if overlap_empty_gold:
        print(f"  !! {len(overlap_empty_gold)} trong số này lại là gold answer trong train.json: "
              f"{sorted(overlap_empty_gold)[:10]}{'...' if len(overlap_empty_gold) > 10 else ''}")
        print("  -> các câu hỏi có gold answer rỗng này sẽ KHÔNG BAO GIỜ đạt recall=1 dù pipeline đúng, "
              "cần kiểm tra lại nguồn dữ liệu gốc nếu muốn khắc phục.")
    for d in empty_doc_ids:
        del corpus[d]

print("Số văn bản trong corpus (sau khi lọc):", len(corpus))
doc_ids_all = list(corpus.keys())

# ---- doc_id -> 1000 ký tự đầu của root_preamble (đoạn mở đầu văn bản), dùng để ghép
# short-chunk ở Stage-1 (xem short-chunk trực tiếp, Cell 6b) ----
doc_root_preamble_of = {d: _extract_root_preamble(v["passage_segments"]) for d, v in corpus.items()}

Loading processed-context corpus files: 100%|██████████| 8532/8532 [00:37<00:00, 227.51it/s]


Số văn bản trong corpus (trước khi lọc): 8532
Cảnh báo: 21 văn bản có passage_segments RỖNG -> sẽ loại khỏi corpus.
  !! 6 trong số này lại là gold answer trong train.json: ['10533', '131890', '149317', '263763', '288457', '55497']
  -> các câu hỏi có gold answer rỗng này sẽ KHÔNG BAO GIỜ đạt recall=1 dù pipeline đúng, cần kiểm tra lại nguồn dữ liệu gốc nếu muốn khắc phục.
Số văn bản trong corpus (sau khi lọc): 8511


## 1b. Chunk văn bản dài (dành riêng cho Stage-2 / reranker)

BAAI/bge-reranker-v2-m3 chỉ hỗ trợ tối đa 2304 token, trong khi corpus có văn bản dài tới hàng triệu ký tự
(xem phân tích phân bố độ dài — median ~23.000 ký tự, một số file >300.000 ký tự là các bộ luật gộp
nhiều điều/chương vào 1 file).

**Chiến lược 3 tầng ưu tiên khi chunk 1 văn bản:**
1. Nếu văn bản có cấu trúc `"Điều X."` rõ ràng (≥2 lần xuất hiện) → tách theo ranh giới Điều, rồi gộp các
   Điều ngắn lại gần `TARGET_CHUNK_CHARS` để không sinh ra quá nhiều chunk vụn.
2. Nếu không có cấu trúc Điều (văn bản thường: bản án, công văn, quyết định...) → tách theo đoạn văn
   (`\n\n`, fallback `\n`), rồi gộp tương tự.
3. Nếu 1 đơn vị (Điều hoặc đoạn văn) vẫn tự nó dài hơn `HARD_MAX_CHUNK_CHARS` → buộc cắt cứng theo ký tự,
   nhưng ưu tiên cắt tại vị trí gần dấu kết câu (`.`, `?`, `!`, `;`) để tránh đứt giữa câu, có overlap để
   không mất ngữ cảnh ở ranh giới.

**Lưu ý quan trọng:** Stage-1 (BM25) vẫn giữ nguyên hoạt động ở cấp **toàn văn bản** như baseline gốc —
BM25 là phương pháp bag-of-words nên không bị giới hạn context như model transformer, chunking ở đây chỉ
cần thiết cho Stage-2 (reranker).


In [8]:
# Cell 4b — Chunking cho pipeline (đã bỏ nhánh LONG-chunking cố định — xem ghi chú bên dưới):
#  - Short chunking (tối đa SHORT_CHUNK_CHARS=1000 ký tự, KHÔNG overlap) -> dùng cho Stage-1 (BM25/retrieval).
#  - Stage-2 (reranker) dùng trực tiếp short-chunk, không tạo long-chunk
#    (chunk_document_long/chunk_long_with_sentence_overlap/get_doc_chunks_long — đã XOÁ vì không
#    còn nơi nào gọi tới nữa). Stage-2 không ghép short-chunk thành đơn vị mới
#    lúc cần dùng (select_relevant_chunks — Cell 6b — cho pipeline chính/eval; short-chunk trực tiếp
#    — Cell 10c — cho mining training pairs), nên không cần tạo long-chunk ở bất kỳ bước nào.
# Việc tách theo cấu trúc pháp lý Điều -> Khoản -> Điểm (các hàm split_* bên dưới) vẫn dùng chung cho
# short-chunking và cho các bước tách câu khác trong notebook.

DIEU_SPLIT_RE = re.compile(r"(?=\n\s*Điều\s+\d+[\.:])", re.UNICODE)
DIEU_COUNT_RE = re.compile(r"(?:^|\n)\s*Điều\s+\d+[\.:]", re.UNICODE)
KHOAN_SPLIT_RE = re.compile(r"(?=\n\s*\d+\.\s)", re.UNICODE)          # ranh giới Khoản: "\n1. ", "\n2. ", ...
DIEM_SPLIT_RE = re.compile(r"(?=\n\s*[a-zđ]\)\s)", re.UNICODE)        # ranh giới Điểm: "\na) ", "\nb) ", ...
_SENTENCE_SPLIT_RE = re.compile(r"(?<=[\.!?;:])\s+|\n+", re.UNICODE)  # ranh giới câu hoàn chỉnh / dòng

# ---- Chuẩn hoá xuống dòng + dấu chấm/chấm lửng lặp, DÙNG CHUNG CHO CẢ NOTEBOOK ----
# Quy tắc:
#  1) "\r\n", "\r" (đơn lẻ) hoặc "\n" (đơn lẻ) đều CHỈ chuyển thành ĐÚNG 1 khoảng trắng " ".
#  2) Dãy dấu chấm DÀI HƠN 3 dấu (vd "......", tức >=4 dấu "." liền nhau) -> thay bằng " ".
#  3) Dãy dấu chấm lửng Unicode "…" lặp lại, xen khoảng trắng (vd "… … …") -> thay bằng " ".
#     (Đây là 2 dạng KHÁC NHAU: dạng 2 là nhiều ký tự "." liền nhau; dạng 3 là ký tự "…" đơn lặp
#     lại nhiều lần, có thể xen khoảng trắng giữa các "…". Không gộp chung logic.)
#  4) Sau các bước trên, nếu còn nhiều dấu " " liên tiếp thì gộp lại thành 1 dấu " " duy nhất.
# áp dụng ĐỒNG BỘ ở mọi nơi tạo ra chunk text CUỐI CÙNG (short-chunk: chunk_document_short
# ghép từ short-chunk: short-chunk trực tiếp ở Cell 6b, short-chunk trực tiếp ở Cell 10c) để
# chunk text đưa vào BM25/dense/reranker không còn lẫn xuống dòng thô hay dãy chấm/chấm lửng thừa,
# dù được tạo ra từ hàm nào.
# CHỈ dùng hàm này ở bước CUỐI (sau khi đã tách xong theo Điều/Khoản/Điểm/đoạn văn/câu) —
# KHÔNG dùng trước hoặc trong lúc tách, vì các bước tách (split_by_dieu/khoan/diem/paragraph,
# split_into_sentences) vẫn cần \n gốc để nhận diện đúng ranh giới.
_NEWLINE_NORM_RE = re.compile(r"\r\n|\r")
_NEWLINE_RUN_RE = re.compile(r"[ \t]*\n[ \t]*(?:\n[ \t]*)*")
_DOT_RUN_RE = re.compile(r"\.{4,}")                    # dãy "." dài hơn 3 dấu (>=4 dấu liền nhau)
_ELLIPSIS_RUN_RE = re.compile(r"…(?:[ \t]*…)+")         # "… …", "… … …", ... (>=2 lần, xen khoảng trắng)
_MULTI_SPACE_RE = re.compile(r" {2,}")                  # nhiều dấu cách liên tiếp -> 1 dấu cách


def normalize_newlines(text):
    """Chuẩn hoá xuống dòng + dãy dấu chấm/chấm lửng lặp, DÙNG CHUNG CHO CẢ NOTEBOOK:
    1) "\r\n"/"\r" -> "\n" trước, sau đó GỘP cả cụm \n liên tiếp gần nhau (kể cả xen khoảng trắng/tab)
       thành ĐÚNG 1 khoảng trắng " ".
    2) Dãy dấu chấm dài hơn 3 dấu ("......", >=4 dấu "." liền nhau) -> thay bằng " ".
    3) Dãy dấu chấm lửng Unicode "…" lặp lại, có thể xen khoảng trắng ("… … …") -> thay bằng " ".
       (Khác với (2): đây là ký tự "…" đơn lặp lại nhiều lần, không phải nhiều dấu "." liền nhau.)
    4) Gộp mọi dấu " " liên tiếp còn sót lại (do (1)/(2)/(3) sinh ra hoặc đã có sẵn) thành 1 dấu " ".
    Dùng ở bước cuối khi trả về chunk text, KHÔNG dùng trước khi split theo cấu trúc pháp lý."""
    text = _NEWLINE_NORM_RE.sub("\n", text or "")
    text = _NEWLINE_RUN_RE.sub(" ", text)
    text = _DOT_RUN_RE.sub(" ", text)
    text = _ELLIPSIS_RUN_RE.sub(" ", text)
    text = _MULTI_SPACE_RE.sub(" ", text)
    return text.strip()


def split_into_sentences(text):
    """Tách text thành list câu hoàn chỉnh, dựa trên dấu kết câu (. ! ? ; :) và ranh giới dòng."""
    text = (text or "").strip()
    if not text:
        return []
    return [p.strip() for p in _SENTENCE_SPLIT_RE.split(text) if p.strip()]


def split_by_dieu(text):
    return [p.strip() for p in DIEU_SPLIT_RE.split(text) if p.strip()]


def split_by_khoan(text):
    return [p.strip() for p in KHOAN_SPLIT_RE.split(text) if p.strip()]


def split_by_diem(text):
    return [p.strip() for p in DIEM_SPLIT_RE.split(text) if p.strip()]


def split_by_paragraph(text):
    parts = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    if len(parts) < 2:
        parts = [p.strip() for p in text.split("\n") if p.strip()]
    return parts


def split_top_level_units(text):
    """Tách văn bản thành các đơn vị Điều (nếu có cấu trúc Điều rõ ràng, >=2 Điều), nếu không thì
    theo đoạn văn — giống hệt logic gốc, chỉ đổi tên cho rõ vai trò "top-level" trong pipeline 2 tầng."""
    n_dieu = len(DIEU_COUNT_RE.findall(text))
    if n_dieu >= 2:
        return split_by_dieu(text)
    units = split_by_paragraph(text)
    if len(units) < 2:
        units = [text]
    return units


def split_by_sentence_hard(text, max_chars):
    """Cắt 1 đoạn text không còn tách được theo Khoản/Điểm thành các đoạn <= max_chars, LUÔN cắt tại
    ranh giới câu hoàn chỉnh (không bao giờ cắt giữa câu), KHÔNG overlap — dùng làm bước cuối cùng khi
    đã xuống tới cấp thấp nhất của cấu trúc pháp lý mà vẫn còn dài."""
    sentences = split_into_sentences(text)
    if not sentences:
        return [text[:max_chars].strip()] if text.strip() else []

    chunks, buf = [], ""
    for sent in sentences:
        if len(sent) > max_chars:
            if buf:
                chunks.append(buf); buf = ""
            start = 0
            while start < len(sent):
                chunks.append(sent[start:start + max_chars].strip())
                start += max_chars
            continue
        if len(buf) + len(sent) + 1 <= max_chars:
            buf = (buf + " " + sent).strip() if buf else sent
        else:
            chunks.append(buf)
            buf = sent
    if buf:
        chunks.append(buf)
    return [c for c in chunks if c]


def split_legal_unit_recursive(unit_text, max_chars):
    """Tách đệ quy 1 đơn vị (Điều/đoạn văn) xuống Khoản rồi Điểm nếu vẫn dài hơn max_chars. Nếu
    xuống tới Điểm vẫn còn dài -> cắt theo câu hoàn chỉnh (split_by_sentence_hard)."""
    if len(unit_text) <= max_chars:
        return [unit_text]

    khoan_parts = split_by_khoan(unit_text)
    if len(khoan_parts) >= 2:
        out = []
        for kp in khoan_parts:
            out.extend(split_legal_unit_recursive(kp, max_chars))
        return out

    diem_parts = split_by_diem(unit_text)
    if len(diem_parts) >= 2:
        out = []
        for dp in diem_parts:
            out.extend(split_legal_unit_recursive(dp, max_chars))
        return out

    return split_by_sentence_hard(unit_text, max_chars)


# =========================================================================================
# SHORT CHUNKING (Stage-1 / retrieval, và là được dùng trực tiếp cho Stage-2 — xem
# select_relevant_chunks ở Cell 6b, short-chunk trực tiếp ở Cell 10c) — mỗi Điều/Khoản/Điểm
# là 1 đơn vị ĐỘC LẬP, tối đa SHORT_CHUNK_CHARS ký tự, KHÔNG gộp, KHÔNG overlap. Đơn vị dài quá thì
# tách tiếp Khoản -> Điểm -> câu.
# =========================================================================================
def chunk_document_short(doc_id, text):
    text = (text or "").strip()
    if not text:
        return []
    top_units = split_top_level_units(text)
    chunks = []
    for u in top_units:
        chunks.extend(split_legal_unit_recursive(u, SHORT_CHUNK_CHARS))
    # Chuẩn hoá xuống dòng -> khoảng trắng Ở BƯỚC CUỐI (xem normalize_newlines phía trên) — đồng bộ
    # với short-chunk trực tiếp
    # (Cell 10c).
    return [(f"{doc_id}#s{i}", normalize_newlines(c)) for i, c in enumerate(chunks) if c.strip()]


# ĐÃ SỬA: corpus giờ load từ processed-context THẬT (Cell 4, output của main.py/preprocess.py/
# md_to_json.py) nên MỖI văn bản đã có sẵn "passage_segments" — list[dict] {"path", "content",
# "segment_type"} (xem collect_segments()/md_to_segments() trong md_to_json.py), KHÔNG còn là
# string "path | content" nữa. Hàm dưới đây thay thế chunk_document_short() ở Stage-1 (Cell 7 build
# BM25 index): dùng THẲNG từng passage_segment làm 1 đơn vị short-chunk gốc (mọi segment_type trong
# ALLOWED_SEGMENT_TYPES — Cell 4 — đều đã được lọc/giữ lại từ lúc load_corpus).
#
# path (vd "Chương I > Điều 2 > Khoản 1") bình thường chỉ vài chục ký tự. Nếu path DÀI HƠN
# MAX_SEGMENT_PATH_CHARS (Cell 2, mặc định 500) thì coi đây là BẤT THƯỜNG/NHIỄU của bước
# preprocessing (vd md_to_json.py tách sai ranh giới path/content) -> KHÔNG dùng nó làm path riêng
# nữa mà GỘP THẲNG vào content, coi như text thường (path == "" trong trường hợp này).
#
# ĐÃ SỬA (quan trọng, MỚI): việc CẮT content vẫn xảy ra TRƯỚC, CHỈ dựa trên content thuần (KHÔNG
# tính path vào ngân sách cắt) — dùng ĐÚNG logic cũ (Khoản -> Điểm -> câu hoàn chỉnh,
# split_legal_unit_recursive phía trên, LUÔN cắt tại ranh giới câu hoàn chỉnh — không bao giờ cắt
# giữa câu). NHƯNG NGAY SAU KHI CẮT XONG (ở mỗi mảnh con sinh ra), nếu path hợp lệ
# (< MAX_SEGMENT_PATH_CHARS, tức không bị coi là nhiễu) thì path đó được GẮN NGAY vào ĐẦU mảnh con
# bằng " | " ("path | content-đã-cắt") để tạo thành short-chunk text CUỐI CÙNG — chính text này mới
# được đưa vào BM25/dense index ở Stage-1 (Cell 7). Nhờ vậy, khi 1 short-chunk được chọn làm đại
# diện để đưa vào Stage-2/reranker, KHÔNG cần gắn path thêm lần nữa (xem short-chunk trực tiếp ở
# Cell 6b) — path đã có sẵn ngay trong short-chunk text rồi. path gốc vẫn được trả về riêng (xem
# short_chunk_path_of ở Cell 7) để tham khảo/debug, không còn là thành phần bắt buộc ở Stage-2 nữa.

def chunk_document_short_from_segments(doc_id, segments):
    """Trả về list[(chunk_id, text, path, segment_type)]:
      - text: content ĐÃ CẮT theo ngân sách RIÊNG cho từng segment_type (xem
        SHORT_CHUNK_CHARS_ROOT_PREAMBLE/UNSTRUCTURED/STRUCTURED — Cell 2), sau đó GẮN THÊM prefix
        vào ĐẦU bằng " | " tuỳ segment_type — đây là text thật sự đưa vào index Stage-1 (Cell 7) VÀ
        cũng chính là short-chunk dùng thẳng ở Stage-2 (Cell 6b), không cần gắn lại gì nữa.
        ĐÃ SỬA (MỚI, quan trọng): mỗi segment_type có 1 công thức cắt/gắn RIÊNG, được thiết kế để
        TỔNG ĐỘ DÀI short-chunk cuối cùng LUÔN <= 2000 ký tự (SHORT_CHUNK_CHARS) — đây là ngân sách
        chung mà CẢ BM25 lẫn dense index ở Stage-1 đều nhận:
          * "root_preamble": content cắt tối đa 2000 ký tự (SHORT_CHUNK_CHARS_ROOT_PREAMBLE), KHÔNG
            gắn thêm gì (chính segment này đã LÀ root_preamble của document) -> tổng tối đa 2000.
          * "unstructured_whole": content cắt tối đa 1500 ký tự (SHORT_CHUNK_CHARS_UNSTRUCTURED),
            sau đó GẮN THÊM ROOT_PREAMBLE_CHARS (500) ký tự root_preamble của document (đã
            precompute ở doc_root_preamble_of — Cell 4) vào ĐẦU -> tổng tối đa 1500 + 500 = 2000.
          * "structured": content cắt tối đa 1000 ký tự (SHORT_CHUNK_CHARS_STRUCTURED), gắn path
            (nếu hợp lệ, < MAX_SEGMENT_PATH_CHARS) rồi GẮN THÊM ROOT_PREAMBLE_CHARS (500) ký tự
            root_preamble vào ĐẦU (trước path) -> tổng tối đa 1000 + 500 (path) + 500 (root_preamble)
            = 2000. Nếu path KHÔNG hợp lệ (>= MAX_SEGMENT_PATH_CHARS) thì path đó đã được GỘP THẲNG
            vào content TRƯỚC KHI cắt (xem đoạn xử lý MAX_SEGMENT_PATH_CHARS bên dưới) nên vẫn nằm
            trong ngân sách 1000 ký tự content, không cộng dồn thêm.
      - path: path gốc của segment nếu path < MAX_SEGMENT_PATH_CHARS, ngược lại "" (path bất thường
        đã bị gộp thẳng vào content ở trên). Giữ lại riêng để tham khảo/debug.
      - segment_type (MỚI): segment_type gốc của segment sinh ra mảnh này ("structured",
        "root_preamble", "unstructured_whole", hoặc "" nếu seg là string fallback định dạng cũ
        không mang segment_type). Dùng để nhận biết short-chunk nào thuộc root_preamble (không có
        path/cấu trúc phân cấp) -> Stage-2 dùng trực tiếp short-chunk bằng
        ROOT_PREAMBLE_EXTEND_CHARS ký tự trước/sau thay vì dùng root_preamble + short-chunk như
        segment "structured". Logic Cell 6b GIỮ NGUYÊN không đổi.
    """
    # ĐÃ SỬA (MỚI): root_preamble (ROOT_PREAMBLE_CHARS ký tự đầu) của CHÍNH document đang chunk —
    # cần cho việc gộp vào short-chunk "structured" bên dưới. doc_root_preamble_of được precompute
    # ở Cell 4 (chạy trước Cell 7, nơi hàm này thực sự được gọi) nên luôn có sẵn trong session; nếu
    # vì lý do nào đó doc_id chưa có trong dict (vd. gọi hàm này độc lập ngoài luồng chính) thì coi
    # như không có root_preamble ("").
    doc_root_preamble = doc_root_preamble_of.get(doc_id, "") if "doc_root_preamble_of" in globals() else ""

    chunks = []
    for seg in segments or []:
        if not seg:
            continue
        # segments đã là dict {"path", "content", "segment_type"} đã clean sẵn từ load_corpus (Cell 4);
        # vẫn hỗ trợ fallback string "path | content" (định dạng cũ) để không phá vỡ pickle cache cũ.
        if isinstance(seg, dict):
            path, content = seg.get("path", ""), seg.get("content", "")
            seg_type = seg.get("segment_type", "")
        else:
            seg = seg.strip()
            if " | " in seg:
                path, content = seg.split(" | ", 1)
            else:
                path, content = "", seg
            seg_type = ""
        path = (path or "").strip()
        content = (content or "").strip()

        # Path >= MAX_SEGMENT_PATH_CHARS (500) là BẤT THƯỜNG (coi là nhiễu của bước preprocessing,
        # ví dụ md_to_json.py tách sai ranh giới path/content) -> KHÔNG dùng nó làm path riêng nữa,
        # mà GỘP THẲNG vào content (coi như text thường). Vẫn đưa đi retrieve như content bình thường,
        # chỉ khác là không còn giữ nguyên như 1 "path" độc lập -> path = "" từ đây trở đi.
        if len(path) >= MAX_SEGMENT_PATH_CHARS:
            content = f"{path} {content}".strip() if content else path
            path = ""

        if not content:
            continue

        # ĐÃ SỬA (MỚI): ngân sách cắt CONTENT (KHÔNG tính path/root_preamble gắn thêm) giờ RIÊNG
        # theo từng segment_type (xem SHORT_CHUNK_CHARS_ROOT_PREAMBLE/UNSTRUCTURED/STRUCTURED — Cell
        # 2), thay vì dùng chung 1 SHORT_CHUNK_CHARS như trước:
        #   - "root_preamble": budget = SHORT_CHUNK_CHARS_ROOT_PREAMBLE (2000) — không gắn thêm gì ở
        #     bước build text bên dưới nên tổng short-chunk cũng tối đa 2000.
        #   - "unstructured_whole": budget = SHORT_CHUNK_CHARS_UNSTRUCTURED (1500) — sẽ gắn thêm
        #     ROOT_PREAMBLE_CHARS (500) ký tự root_preamble ở bước build text bên dưới -> tổng tối đa
        #     1500 + 500 = 2000.
        #   - "structured" (hoặc "" fallback định dạng cũ): budget = SHORT_CHUNK_CHARS_STRUCTURED
        #     (1000) — sẽ gắn thêm path (< MAX_SEGMENT_PATH_CHARS = 500) và ROOT_PREAMBLE_CHARS (500)
        #     ký tự root_preamble ở bước build text bên dưới -> tổng tối đa 1000 + 500 + 500 = 2000.
        # Nhờ vậy, MỌI short-chunk cuối cùng đưa vào BM25/dense index ở Stage-1 (Cell 7) đều đảm bảo
        # tối đa 2000 ký tự, bất kể segment_type. path, segment_type (và root_preamble gắn lúc build
        # text ở dưới) được gắn NGUYÊN VẸN vào cùng MỌI mảnh con tách ra từ content này.
        if seg_type == "root_preamble":
            _content_budget = SHORT_CHUNK_CHARS_ROOT_PREAMBLE
        elif seg_type == "unstructured_whole":
            _content_budget = SHORT_CHUNK_CHARS_UNSTRUCTURED
        else:
            _content_budget = SHORT_CHUNK_CHARS_STRUCTURED
        for part in split_legal_unit_recursive(content, _content_budget):
            chunks.append((part, path, seg_type))

    out = []
    for i, (text, path, seg_type) in enumerate(chunks):
        if not text.strip():
            continue
        # ĐÃ SỬA (MỚI): root_preamble (ROOT_PREAMBLE_CHARS = 500 ký tự đầu của document, xem
        # doc_root_preamble_of — Cell 4) giờ được gộp thêm vào ĐẦU cho CẢ "structured" LẪN
        # "unstructured_whole" (trước đây chỉ "structured"):
        #   - "structured" -> "root_preamble(500) | path(<500) | content-đã-cắt(<=1000)" (path chỉ
        #     gắn nếu hợp lệ, xem đoạn lọc MAX_SEGMENT_PATH_CHARS phía trên).
        #   - "unstructured_whole" -> "root_preamble(500) | content-đã-cắt(<=1500)" (không có path).
        #   - "root_preamble" -> giữ NGUYÊN "content-đã-cắt(<=2000)", KHÔNG gộp gì thêm (chính segment
        #     này đã LÀ root_preamble của document, gộp thêm chính nó là vô nghĩa).
        prefix_parts = []
        if seg_type in ("structured", "unstructured_whole") and doc_root_preamble:
            prefix_parts.append(doc_root_preamble)
        if path:
            prefix_parts.append(path)
        prefix = " | ".join(prefix_parts)
        final_text = f"{prefix} | {text}" if prefix else text
        # ĐÃ SỬA (MỚI): gắn path (và root_preamble nếu có) vào ĐẦU mảnh con (đã cắt) bằng " | "
        # NGAY TẠI ĐÂY, SAU KHI cắt xong -> đây là short-chunk text cuối cùng, dùng cho CẢ Stage-1
        # (BM25/dense index) LẪN Stage-2 (short-chunk = root_preamble + short-chunk này, xem Cell 6b
        # — logic Cell 6b GIỮ NGUYÊN không đổi), không cần gắn path thêm lần nào nữa ở bước sau.
        out.append((f"{doc_id}#s{i}", normalize_newlines(final_text), path, seg_type))
    return out


In [9]:
# Cell 5 — Chuẩn hoá train.json thành list các sample (qid, question, list[answer_id])
def flatten_queries(raw):
    samples = []
    for qid, v in raw.items():
        samples.append({
            "qid": str(qid),
            "question": v["question"],
            "answers": [str(a) for a in v["answer"]],
        })
    return samples

train_samples = flatten_queries(train_raw)
print(train_samples[0])

# Tách train/dev nội bộ để đánh giá offline (đề bài không cho nhãn của public/private test)
random.shuffle(train_samples)
n_dev = max(1, int(0.1 * len(train_samples)))
dev_samples = train_samples[:n_dev]
fit_samples = train_samples[n_dev:]
print(f"fit: {len(fit_samples)} | dev (offline eval): {len(dev_samples)}")


{'qid': '86666', 'question': 'Thời hạn cấp đăng ký xe máy của người nước ngoài làm việc tại Việt Nam là bao lâu?', 'answers': ['280282']}
fit: 6300 | dev (offline eval): 700


## 2. Stage 1 — Hybrid retrieval (BM25 + Dense)

Bài toán DRiLL (VLSP 2025) được đặt ra như một bài toán truy hồi điều luật: cho một câu
hỏi pháp lý dạng ngôn ngữ tự nhiên (query) và một kho ngữ liệu lớn gồm các điều luật của
Việt Nam (mỗi điều/khoản/điểm tương ứng 1 provision), hệ thống cần trả về danh sách các
điều luật liên quan, có thể trả lời hoặc bao hàm câu hỏi đó.

Hình thức hoá: cho câu hỏi *q* và kho ngữ liệu *C = {a₁, a₂, ..., a_N}* gồm các điều
luật (ở đây là các **short-chunk** — xem mục 1b), hệ thống truy hồi cần học một hàm
chấm điểm *f(q, a)* sao cho các điều luật liên quan nhận điểm cao hơn.

Để tối đa hoá recall, ta dùng **hybrid retrieval**, kết hợp cả tín hiệu sparse (từ vựng)
và dense (ngữ nghĩa):

- **Sparse retrieval (BM25).** BM25 (Robertson & Zaragoza, 2009) tính điểm liên quan dựa
  trên khớp từ vựng chính xác giữa các từ trong câu hỏi và trong điều luật, có điều chỉnh
  theo tần suất từ (TF) và tần suất nghịch đảo văn bản (IDF):

  $$\mathrm{BM25}(q, a) = \sum_{t \in q} \mathrm{IDF}(t) \cdot
  \frac{f(t, a) \cdot (k_1 + 1)}{f(t, a) + k_1 \cdot \left(1 - b + b \cdot \frac{|a|}{\mathrm{avgdl}}\right)}$$

  trong đó *f(t, a)* là tần suất xuất hiện của từ *t* trong điều luật *a*, *|a|* là độ dài
  điều luật, *avgdl* là độ dài trung bình, *k₁, b* là các siêu tham số (dùng mặc định của
  `rank_bm25`).

- **Dense retrieval.** Ta dùng mô hình **Vietnamese_Embedding_v2 (AITeamVN)** để encode cả câu hỏi *q*
  và điều luật *a* thành các vector dense h_q, h_a ∈ ℝ^d. Điểm liên quan là cosine
  similarity giữa 2 vector (tương đương dot-product vì embedding đã được chuẩn hoá về độ
  dài 1):

  $$\mathrm{Dense}(q, a) = \cos(h_q, h_a) = \frac{h_q \cdot h_a}{\lVert h_q \rVert \, \lVert h_a \rVert}$$

- **Kết hợp 2 tín hiệu.** Vì thang điểm của BM25 và cosine similarity rất khác nhau, ta
  min-max normalize từng loại điểm về [0, 1] trên cùng tập ứng viên rồi cộng có trọng số:

  $$\mathrm{Score}(q, a) = \lambda \cdot \mathrm{BM25_{norm}}(q, a) + (1 - \lambda) \cdot \mathrm{Dense_{norm}}(q, a)$$

  với λ ∈ [0, 1] là siêu tham số được **tune trên tập validation (dev)** — xem Cell 8c.
  Chiến lược đơn giản nhưng hiệu quả này cân bằng giữa độ chính xác của sparse retrieval
  và khả năng khái quát hoá ngữ nghĩa của dense retrieval. Hybrid retriever trả về top-k
  điều luật ứng viên cho reranker ở stage-2 (giữ nguyên BAAI/bge-reranker-v2-m3, không đổi).

> **Lưu ý về thứ tự chạy cell:** phần định nghĩa hàm `bm25_retrieve` / `dense_retrieve` /
> `hybrid_retrieve` nằm ở đây (mục 2), nhưng việc **load model dense lên GPU và encode**
> (và bước tune λ đi kèm) được đặt **sau** mục 4 (mining negative) và mục 5 (fine-tune
> reranker) — ngay trước mục 6 (Pipeline) — để tránh xung đột `fork()` (multiprocessing,
> dùng khi mining) với CUDA context (xem lưu ý chi tiết ở cell encode dense).


In [10]:
# Cell 6 — Tokenize tiếng Việt
# CHỈ dùng underthesea cho MỌI nơi tokenize trong notebook (đã bỏ fast tokenizer bằng regex).
# vi_tokenize_short (dùng cho BM25 short-chunk, Cell 7/8) và vi_tokenize (dùng cho phần còn lại:
# chọn short-chunk đại diện, chunk_token_cache, ...) giờ trỏ về CÙNG 1 hàm underthesea bên dưới.
# Giữ nguyên 2 tên biến để không phải sửa các cell downstream đang gọi vi_tokenize_short/vi_tokenize.
from underthesea import word_tokenize as _underthesea_tokenize


def vi_tokenize(text: str):
    return _underthesea_tokenize((text or "").lower())


# ---- Mọi nơi (BM25 short-chunk Stage-1, short-chunk mining, select_best_chunk, chunk_token_cache,
#      ...): LUÔN dùng underthesea, không còn fast/regex tokenizer nữa ----
vi_tokenize_short = vi_tokenize

import bm25s  # thay cho rank_bm25 -> BM25Okapi: bm25s dùng backend numpy/scipy tối ưu (numba/JIT nếu
                 # có sẵn), tính điểm nhanh hơn đáng kể trên corpus lớn, giảm chi phí tính toán ở Stage-1


In [ ]:
# Cell 6a — Tokenize sẵn TẤT CẢ câu hỏi (train_samples), cache ra pickle.
# ĐÃ SỬA: chỉ còn 1 tokenizer (underthesea) cho toàn bộ notebook, nên chỉ cần tokenize câu hỏi
# đúng 1 LẦN duy nhất, lưu ra đĩa theo qid -> list[token]. Cell 11/11b sau đó chỉ cần TRA CACHE,
# không tokenize lại nữa. `_question_token_cache_fast` được GIỮ LẠI làm alias trỏ tới CÙNG dict với
# `_question_token_cache` (không tokenize 2 lần nữa) để không phải sửa tên biến ở các cell downstream
# (mining, Cell 11/11b) vẫn đang tham chiếu `_question_token_cache_fast`.
import pickle

_QUESTION_TOKEN_CACHE_PATH_ACCURATE = "DSC2026-IntelliJQK/cache/question_token_cache_underthesea.pkl"


def _load_or_init_token_cache(path, label):
    if os.path.exists(path):
        with open(path, "rb") as f:
            cache = pickle.load(f)
        print(f"Đã load question token cache ({label}) từ {path}: {len(cache)} câu hỏi.")
    else:
        cache = {}
        print(f"Chưa có question token cache ({label}) tại {path} -> sẽ tokenize mới và lưu lại.")
    return cache


_question_token_cache = _load_or_init_token_cache(_QUESTION_TOKEN_CACHE_PATH_ACCURATE, "underthesea")
# Alias: các cell downstream (Cell 11/11b) vẫn dùng tên `_question_token_cache_fast` -> trỏ về
# CÙNG dict underthesea ở trên, không cần cache/tokenize riêng nữa.
_question_token_cache_fast = _question_token_cache

_qid2question = {s["qid"]: s["question"] for s in train_samples}  # gồm cả fit_samples + dev_samples

_tokenize_fn_for_pool = None

def _tokenize_questions_batch(qid_batch):
    return [(qid, _tokenize_fn_for_pool(_qid2question[qid])) for qid in qid_batch]


def _set_tokenizer_fn(fn):
    global _tokenize_fn_for_pool
    _tokenize_fn_for_pool = fn


def _tokenize_missing_questions(cache, tokenize_fn, cache_path, label, allow_parallel):
    """Tokenize các câu hỏi còn thiếu trong `cache` bằng `tokenize_fn`, lưu lại vào `cache_path`.
    allow_parallel=True dùng multiprocessing.Pool (an toàn ở đây vì CUDA/model lớn chưa init) —
    bật multiprocessing để tăng tốc underthesea (tokenizer duy nhất còn lại, chậm hơn regex)."""
    missing_qids = [qid for qid in _qid2question if qid not in cache]
    if not missing_qids:
        print(f"Tất cả câu hỏi (fit + dev) đã có sẵn trong cache ({label}), không cần tokenize lại.")
        return

    n_workers = min(get_n_workers(), len(missing_qids))  # tránh tạo worker thừa nếu số câu hỏi ít
    if not allow_parallel or n_workers == 1 or len(missing_qids) < 32:
        for qid in tqdm(missing_qids, desc=f"Tokenize câu hỏi ({label})"):
            cache[qid] = tokenize_fn(_qid2question[qid])
    else:
        # An toàn để Pool ở đây (chưa có CUDA/model lớn nào được load trong process này).
        from multiprocessing import Pool

        batch_size = min(64, max(8, len(missing_qids) // (n_workers * 4)))
        qid_batches = [missing_qids[i:i + batch_size] for i in range(0, len(missing_qids), batch_size)]

        with Pool(n_workers, initializer=_set_tokenizer_fn, initargs=(tokenize_fn,)) as pool:
            for batch_result in tqdm(pool.imap_unordered(_tokenize_questions_batch, qid_batches),
                                     total=len(qid_batches), desc=f"Tokenize câu hỏi ({label}, song song)"):
                for qid, toks in batch_result:
                    cache[qid] = toks

    with open(cache_path, "wb") as f:
        pickle.dump(cache, f)
    print(f"Đã tokenize {len(missing_qids)} câu hỏi mới ({label}), lưu cache tại {cache_path} "
          f"(tổng {len(cache)} câu hỏi).")


# Underthesea: tokenizer duy nhất còn lại -> tận dụng multiprocessing để tăng tốc.
_tokenize_missing_questions(_question_token_cache, vi_tokenize,
                             _QUESTION_TOKEN_CACHE_PATH_ACCURATE, "underthesea", allow_parallel=True)


Đã load question token cache (underthesea) từ /mnt/mmlab2024nas/trantran/question_token_cache_underthesea.pkl: 7000 câu hỏi.
Tất cả câu hỏi (fit + dev) đã có sẵn trong cache (underthesea), không cần tokenize lại.


In [12]:
# Cell 6b — Chọn SHORT-chunk liên quan nhất ở Stage-2 bằng CÙNG SCORING với Stage-1.
#
# QUY TẮC PIPELINE MỚI:
# - Stage-1 đã chọn TOP_K_STAGE1 document ở cấp document.
# - Với MỖI document ứng viên, Stage-2 chỉ chọn tối đa MAX_CHUNKS_PER_DOC_FOR_RERANK
#   short-chunk có điểm Stage-1 cao nhất.
# - SHORT-CHUNK ĐƯỢC ĐƯA THẲNG VÀO RERANKER. KHÔNG expand, KHÔNG ghép chunk lân cận,
#   KHÔNG tạo long-chunk.
#
# `chunk_scores` là score của TOÀN BỘ short-chunk cho câu hỏi hiện tại, dùng cùng scoring
# với Stage-1 (HYBRID mặc định). Hàm này chỉ làm nhiệm vụ chọn top-N short-chunk trong
# MỘT document; text trả về chính là `short_chunk_text_of` của chunk đó.

def select_relevant_chunks(doc_id, chunk_scores, top_n=None):
    """Trả về tối đa top_n SHORT-chunk của `doc_id` để đưa thẳng vào Stage-2 reranker.

    Không có bất kỳ bước expand/ghép nào ở đây. `chunk_scores` là score của toàn bộ
    short-chunk và được tính bằng cùng scoring function với Stage-1.

    Return: list[(chunk_id, short_chunk_text)].
    """
    if top_n is None:
        top_n = MAX_CHUNKS_PER_DOC_FOR_RERANK
    idxs = _doc_to_short_chunk_idx.get(doc_id)
    if not idxs or top_n <= 0:
        return []

    ranked_global_idx = sorted(idxs, key=lambda i: -chunk_scores[i])[:top_n]
    out = []
    for global_idx in ranked_global_idx:
        cid = short_chunk_ids[global_idx]
        text = short_chunk_text_of[cid]
        if text and text.strip():
            out.append((cid, text))
    return out


In [ ]:
# Cell 7 — Short-chunk toàn bộ corpus (Stage-1 / retrieval) rồi build BM25 index TRÊN CÁC SHORT-CHUNK
# (song song hoá + cache ra đĩa), thay vì trên toàn văn bản như trước. Đây là thay đổi kiến trúc chính
# để khớp đúng paper ViDRILL: Stage-1 hoạt động ở cấp chunk ngắn (450 ký tự, clause-based), không phải
# cấp document.
import pickle
from multiprocessing import Pool, cpu_count

# ---- Bước 1: short-chunk toàn bộ corpus (mỗi văn bản -> nhiều short-chunk độc lập) ----
# ĐÃ SỬA: chunk_document_short_from_segments (Cell 4b) giờ trả về (chunk_id, text, path, segment_type),
# trong đó text ĐÃ GẮN SẴN path ở đầu (nếu path hợp lệ, gắn NGAY SAU KHI cắt content) -> text này vừa
# là short-chunk index ở Stage-1 (BM25/dense) VỪA là short-chunk dùng thẳng ở Stage-2 (Cell 6b, không
# cần gắn path lại). Đổi hậu tố tên file cache ("_pathinchunk") để không lỡ load nhầm cache ĐỊNH DẠNG
# CŨ (chunk_text CHƯA gắn path, hoặc thiếu segment_type) — nếu có cache cũ, cell này sẽ tự short-chunk
# lại.
_SHORT_CHUNK_CORPUS_PATH = os.path.join(
    "DSC2026-IntelliJQK", "cache",
    f"short_chunk_corpus_typed_{SHORT_CHUNK_CHARS_STRUCTURED}_{SHORT_CHUNK_CHARS_UNSTRUCTURED}_"
    f"{SHORT_CHUNK_CHARS_ROOT_PREAMBLE}_7.pkl",
)  # ĐÃ SỬA: đổi tên cache theo 3 ngân sách RIÊNG từng segment_type (Cell 2) để không lỡ load
   # nhầm cache cũ (tạo bằng 1 SHORT_CHUNK_CHARS chung) — nếu chưa có cache mới, cell này sẽ tự
   # short-chunk lại toàn bộ corpus theo logic mới.

if os.path.exists(_SHORT_CHUNK_CORPUS_PATH):
    with open(_SHORT_CHUNK_CORPUS_PATH, "rb") as f:
        short_chunk_corpus = pickle.load(f)  # list[(chunk_id, doc_id, chunk_text, path, segment_type)]
    print(f"Đã load short-chunk corpus từ {_SHORT_CHUNK_CORPUS_PATH}: {len(short_chunk_corpus)} chunk.")
else:
    short_chunk_corpus = []
    for _doc_id in tqdm(doc_ids_all, desc="Short-chunking toàn bộ corpus (Stage-1)"):
        # ĐÃ SỬA: dùng passage_segments (processed-context, Cell 4) thay vì "passage" thô
        for _cid, _ctext, _path, _segtype in chunk_document_short_from_segments(_doc_id, corpus[_doc_id]["passage_segments"]):
            short_chunk_corpus.append((_cid, _doc_id, _ctext, _path, _segtype))
    with open(_SHORT_CHUNK_CORPUS_PATH, "wb") as f:
        pickle.dump(short_chunk_corpus, f)
    print(f"Đã short-chunk {len(doc_ids_all)} văn bản -> {len(short_chunk_corpus)} chunk, "
          f"lưu tại {_SHORT_CHUNK_CORPUS_PATH}")

short_chunk_ids = [c[0] for c in short_chunk_corpus]
short_chunk_doc_of = {c[0]: c[1] for c in short_chunk_corpus}   # chunk_id -> doc_id
short_chunk_text_of = {c[0]: c[2] for c in short_chunk_corpus}  # chunk_id -> text ĐÃ GẮN path ở đầu
                                                                  # (nếu hợp lệ) — dùng cho CẢ Stage-1
                                                                  # (BM25/dense index) LẪN Stage-2
                                                                  # (short-chunk = root_preamble +
                                                                  # short-chunk này, xem Cell 6b)
short_chunk_path_of = {c[0]: c[3] for c in short_chunk_corpus}  # chunk_id -> path gốc ("" nếu không
                                                                  # có/bất thường) -- CHỈ để tham
                                                                  # khảo/debug, KHÔNG còn dùng lại ở
                                                                  # Stage-2 (path đã nằm sẵn trong
                                                                  # short_chunk_text_of rồi)
short_chunk_segtype_of = {c[0]: c[4] for c in short_chunk_corpus}  # chunk_id -> segment_type gốc
                                                                     # ("structured"/"root_preamble"/
                                                                     # "unstructured_whole"/"") --
                                                                     # dùng để nhận biết short-chunk
                                                                     # thuộc root_preamble (Cell 6b)

# ---- Bước 2: tokenize từng short-chunk (song song + cache ra đĩa) ----
# ĐÃ SỬA: chỉ còn underthesea (đã bỏ fast/regex tokenizer) -> đổi hậu tố tên file cache thành
# "underthesea" để không lỡ load nhầm cache tokenize kiểu regex cũ. ĐÃ SỬA THÊM (MỚI): nội dung
# short_chunk_text_of (short_chunk_corpus[idx][2]) giờ đã gắn path ở đầu (nếu hợp lệ, xem Cell 4b) ->
# đổi thêm hậu tố "_pathinchunk" để không lỡ load nhầm cache tokenize CŨ (tokenize trên text CHƯA
# gắn path), buộc tokenize lại cho khớp text mới.
TOKENIZED_CACHE_PATH = os.path.join(
    "DSC2026-IntelliJQK", "cache",
    f"bm25_tokenized_short_chunks_typed_{SHORT_CHUNK_CHARS_STRUCTURED}_{SHORT_CHUNK_CHARS_UNSTRUCTURED}_"
    f"{SHORT_CHUNK_CHARS_ROOT_PREAMBLE}_underthesea_7.pkl",
)  # ĐÃ SỬA: đổi tên cache tương ứng với _SHORT_CHUNK_CORPUS_PATH ở trên, tránh tokenize nhầm
   # cache cũ (short-chunk text đã đổi theo ngân sách mới -> phải tokenize lại).

def _tokenize_one_chunk(idx):
    # BM25 trên short-chunk dùng vi_tokenize_short, giờ LUÔN là underthesea (xem Cell 6).
    return vi_tokenize_short(short_chunk_corpus[idx][2])

if os.path.exists(TOKENIZED_CACHE_PATH):
    print("Đã tìm thấy cache, load lại thay vì tokenize lại:", TOKENIZED_CACHE_PATH)
    with open(TOKENIZED_CACHE_PATH, "rb") as f:
        bm25_tokenized_corpus = pickle.load(f)
else:
    n_workers = get_n_workers()  # giới hạn bởi MAX_WORKERS (Cell 2) -> không chiếm hết CPU server
    n_chunks = len(short_chunk_corpus)
    print(f"Tokenizing {n_chunks} short-chunk bằng {n_workers} tiến trình song song...")
    if n_workers > 1:
        with Pool(n_workers, initializer=_lower_priority) as pool:
            bm25_tokenized_corpus = list(
                tqdm(pool.imap(_tokenize_one_chunk, range(n_chunks), chunksize=200),
                     total=n_chunks, desc="Tokenizing short-chunk corpus")
            )
    else:
        bm25_tokenized_corpus = [_tokenize_one_chunk(i) for i in tqdm(range(n_chunks), desc="Tokenizing short-chunk corpus")]

    with open(TOKENIZED_CACHE_PATH, "wb") as f:
        pickle.dump(bm25_tokenized_corpus, f)
    print("Đã lưu cache tokenize tại:", TOKENIZED_CACHE_PATH)

# ---- Build BM25 index bằng bm25s (thay cho rank_bm25.BM25Okapi) ----
# bm25s nhận trực tiếp corpus đã tokenize sẵn (list[list[str]]) giống hệt rank_bm25, nên không cần
# tokenize lại. Đây là thay đổi duy nhất cần thiết: bm25s dùng cấu trúc dữ liệu sparse (scipy) +
# tính điểm vector hoá, giúp build index và tính get_scores() nhanh hơn nhiều so với rank_bm25
# (pure Python), giảm đáng kể chi phí tính toán Stage-1 khi corpus lớn.
bm25 = bm25s.BM25()
bm25.index(bm25_tokenized_corpus)
print("BM25 (bm25s) index sẵn sàng với", len(short_chunk_ids), "short-chunk trên", len(doc_ids_all), "văn bản")

# ---- Dọn RAM: bm25_tokenized_corpus và short_chunk_corpus chỉ dùng để build index/dict ở trên,
# không còn được tham chiếu ở bất kỳ cell nào phía sau (bm25 đã tự lưu cấu trúc riêng sau .index(),
# và short_chunk_text_of/short_chunk_doc_of/... đã tách ra từ short_chunk_corpus rồi).
import gc, ctypes

del bm25_tokenized_corpus
del short_chunk_corpus
gc.collect()
ctypes.CDLL("libc.so.6").malloc_trim(0)

print("Đã giải phóng bm25_tokenized_corpus và short_chunk_corpus khỏi RAM.")

Đã load short-chunk corpus từ /mnt/mmlab2024nas/trantran/short_chunk_corpus_typed_1000_2000_2000_7.pkl: 442255 chunk.
Đã tìm thấy cache, load lại thay vì tokenize lại: /mnt/mmlab2024nas/trantran/bm25_tokenized_short_chunks_typed_1000_2000_2000_underthesea_7.pkl


Building index from tokens


BM25 (bm25s) index sẵn sàng với 442255 short-chunk trên 8511 văn bản
Đã giải phóng bm25_tokenized_corpus và short_chunk_corpus khỏi RAM.


In [14]:
# Cell 8 — Hàm retrieve top-k bằng BM25, hoạt động ở cấp SHORT-CHUNK rồi gộp điểm về cấp document.
# ĐÃ SỬA (bỏ bước cắt top-K_STAGE1_CHUNKS thô ở cấp CHUNK trước khi gộp): pipeline CŨ là
#   toàn bộ chunk -> cắt top-300 chunk điểm cao nhất -> gộp max-per-doc TRÊN 300 chunk đó
# có nhược điểm: 1 document có NHIỀU chunk liên quan vừa phải (không chunk nào đủ mạnh để lọt
# top-300 toàn corpus) sẽ bị loại hoàn toàn, dù xét riêng document đó nó xứng đáng có mặt trong
# top-k. Pipeline MỚI:
#   toàn bộ chunk -> gộp max-per-doc TRÊN TOÀN BỘ chunk (không cắt trước) -> sort document theo
#   điểm đã gộp -> cắt top-k document cuối cùng
# tức là bước "cắt top-k" giờ chỉ xảy ra ĐÚNG 1 LẦN, ở CẤP DOCUMENT (sau khi đã gộp), thay vì cắt
# ở cấp CHUNK trước rồi mới gộp. Giao diện hàm (trả về list[(doc_id, score)]) giữ nguyên như
# trước để không phải sửa các cell downstream (mining, eval, pipeline 2 tầng).

def bm25_scores_all(question: str = None, q_tokens=None):
    """Trả về np.array điểm BM25 của TẤT CẢ short-chunk (dùng chung cho bm25_retrieve
    và hybrid_scores_all để không phải tính 2 lần).
    q_tokens (nếu truyền vào) PHẢI được tokenize bằng vi_tokenize_short (underthesea, xem Cell 6),
    để khớp đúng tokenizer đã dùng khi build BM25 index trên short-chunk ở Cell 7.
    bm25.get_scores(...) ở đây là API của bm25s (không phải rank_bm25 nữa, xem Cell 6/7) nhưng
    giữ nguyên chữ ký (nhận list[str] token đơn 1 câu hỏi, trả về np.array điểm cho toàn bộ
    corpus) nên không cần sửa gì thêm ở đây."""
    if q_tokens is None:
        q_tokens = vi_tokenize_short(question)
    return bm25.get_scores(q_tokens)


# ---- Precompute (1 LẦN) ánh xạ short-chunk -> document, dùng để gộp max-per-doc bằng numpy
# ---- (np.maximum.at) trên TOÀN BỘ chunk mỗi câu hỏi, thay vì cắt top-K chunk thô trước rồi
# ---- mới gộp (cách cũ). Vector hoá để việc xét TOÀN BỘ chunk (thay vì chỉ 300 chunk) mỗi
# ---- query vẫn đủ nhanh, không lặp bằng Python thuần trên hàng trăm nghìn chunk.
_doc_id_list_for_agg = sorted(set(short_chunk_doc_of.values()))
_doc_id_to_idx_for_agg = {d: i for i, d in enumerate(_doc_id_list_for_agg)}
_chunk_doc_idx_for_agg = np.array(
    [_doc_id_to_idx_for_agg[short_chunk_doc_of[cid]] for cid in short_chunk_ids], dtype=np.int64
)
_n_docs_for_agg = len(_doc_id_list_for_agg)


def _aggregate_scores_to_doc(scores: np.ndarray, top_k: int):
    """np.array điểm của TOÀN BỘ short-chunk (cùng thứ tự với short_chunk_ids, KHÔNG cắt trước)
    -> top_k document_id: list[(doc_id, score)], score = điểm CHUNK CAO NHẤT/document, gộp trên
    TOÀN BỘ chunk của document đó (không giới hạn trong 1 tập con top-K chunk thô như cách cũ).
    Dùng np.maximum.at để gộp max-per-doc vector hoá (nhanh dù xét toàn bộ chunk mỗi query)."""
    best_score_per_doc = np.full(_n_docs_for_agg, -np.inf, dtype=np.float64)
    np.maximum.at(best_score_per_doc, _chunk_doc_idx_for_agg, scores)
    top_doc_idx = np.argsort(best_score_per_doc)[::-1][:top_k]
    return [(_doc_id_list_for_agg[i], float(best_score_per_doc[i])) for i in top_doc_idx]


def bm25_retrieve(question: str, top_k: int = TOP_K_STAGE1, q_tokens=None):
    """Trả về top_k document_id: list[(doc_id, score)], score = điểm short-chunk cao nhất của
    document đó trên TOÀN BỘ short-chunk corpus (đã bỏ bước cắt top-K_STAGE1_CHUNKS thô ở cấp
    chunk trước khi gộp, xem ghi chú đầu cell).
    q_tokens: nếu đã tokenize sẵn câu hỏi (vd. lấy từ _question_token_cache ở Cell 6a khi mining),
    truyền vào đây để tránh tokenize lại bằng underthesea."""
    scores = bm25_scores_all(question, q_tokens=q_tokens)
    return _aggregate_scores_to_doc(scores, top_k)


In [15]:
# Cell 8b — Hàm dense_retrieve (song song với bm25_retrieve, cùng cấp SHORT-CHUNK) và
# hybrid_retrieve (BM25 + Dense theo công thức mục 2.2.1: min-max normalize rồi weighted sum).
# ĐÃ SỬA (đồng bộ với Cell 8): bỏ bước cắt top-K_STAGE1_CHUNKS thô ở cấp CHUNK trước khi gộp về
# document -- gộp max-per-doc trên TOÀN BỘ chunk rồi mới sort/cắt top-k ở CẤP DOCUMENT (dùng chung
# _aggregate_scores_to_doc, Cell 8).

def _minmax_normalize(scores: np.ndarray) -> np.ndarray:
    lo, hi = float(scores.min()), float(scores.max())
    if hi - lo < 1e-12:
        return np.zeros_like(scores)
    return (scores - lo) / (hi - lo)


def dense_scores_all(question: str = None, q_emb: np.ndarray = None):
    """Trả về np.array điểm cosine similarity (dot product, embeddings đã normalize) của
    câu hỏi với TẤT CẢ short-chunk. Dùng chung cho dense_retrieve và hybrid_scores_all."""
    if q_emb is None:
        # ĐÃ SỬA: Vietnamese_Embedding_v2 (AITeamVN, nền BGE-M3) không định nghĩa sẵn prompt "query"
        # như Qwen3-Embedding -> bỏ prompt_name="query" (nếu giữ lại sẽ lỗi vì model không có
        # prompts config này). Encode câu hỏi trực tiếp, không cần prefix đặc biệt.
        q_emb = embed_model.encode(
            [question], convert_to_numpy=True, normalize_embeddings=True
        )[0]
    return chunk_embeddings @ q_emb


def dense_retrieve(question: str, top_k: int = TOP_K_STAGE1, q_emb=None):
    """Dense-only stage-1 (dùng để ablation / so sánh với BM25 và hybrid). Gộp max-per-doc trên
    TOÀN BỘ short-chunk corpus (không cắt top-K chunk thô trước, xem ghi chú đầu cell)."""
    scores = dense_scores_all(question, q_emb=q_emb)
    return _aggregate_scores_to_doc(scores, top_k)


def hybrid_scores_all(question: str = None, lam: float = None, q_tokens=None, q_emb=None):
    """Trả về hybrid score của TOÀN BỘ short-chunk corpus.
    Dùng CHUNG cho Stage-1 và chọn representative chunk trong Stage-2."""
    if lam is None:
        lam = LAMBDA_HYBRID
    bm25_raw = bm25_scores_all(question, q_tokens=q_tokens)
    dense_raw = dense_scores_all(question, q_emb=q_emb)
    return lam * _minmax_normalize(bm25_raw) + (1 - lam) * _minmax_normalize(dense_raw)


def hybrid_retrieve(question: str, top_k: int = TOP_K_STAGE1, lam: float = None, q_tokens=None, q_emb=None):
    """Stage-1 HYBRID mặc định: kết hợp BM25 (sparse) + Vietnamese_Embedding_v2 (AITeamVN) (dense).
    Score(q, a) = lambda * BM25_norm(q, a) + (1 - lambda) * Dense_norm(q, a), tính trên TOÀN BỘ
    short-chunk corpus rồi min-max normalize từng loại điểm về [0, 1] trước khi cộng (BM25 và
    cosine similarity có thang điểm khác nhau nên không thể cộng trực tiếp). Sau đó gộp max-per-doc
    trên TOÀN BỘ chunk (không cắt top-K chunk thô trước) rồi mới sort/cắt top_k ở CẤP DOCUMENT.
    Giao diện hàm giống hệt bm25_retrieve/dense_retrieve (list[(doc_id, score)]) nên có thể truyền
    thẳng vào two_stage_retrieve(stage1_fn=hybrid_retrieve) không cần sửa gì ở downstream."""
    combined = hybrid_scores_all(question, lam=lam, q_tokens=q_tokens, q_emb=q_emb)
    return _aggregate_scores_to_doc(combined, top_k)


## 3. Hàm đánh giá Recall / Precision (đúng theo công thức của đề bài)

In [16]:
# Cell 9 — Recall@k / Precision@k theo đúng định nghĩa trong đề bài
# (trung bình theo từng câu hỏi; nếu hệ thống không trả về gì thì precision câu đó = 0)
def recall_precision(pred_ids, gold_ids):
    gold = set(gold_ids)
    pred = set(pred_ids)
    if len(pred) == 0:
        precision = 0.0
    else:
        precision = len(gold & pred) / len(pred)
    recall = len(gold & pred) / len(gold) if len(gold) > 0 else 0.0
    return recall, precision

def evaluate(retrieve_fn, samples, top_k=TOP_K_FINAL, desc="eval"):
    # retrieve_fn(question) -> list[doc_id] (đã sắp xếp theo độ liên quan giảm dần)
    recalls, precisions = [], []
    for s in tqdm(samples, desc=desc):
        pred = retrieve_fn(s["question"])[:top_k]   # luôn cắt đúng top_k <= 5
        r, p = recall_precision(pred, s["answers"])
        recalls.append(r); precisions.append(p)
    return {
        "recall": float(np.mean(recalls)),
        "precision": float(np.mean(precisions)),
        "n": len(samples),
    }

In [ ]:
# LƯU Ý (ĐÃ SỬA): mining (Cell 10c) giờ KHÔNG còn dùng multiprocessing/fork nữa (chỉ còn numpy +
# dict lookup), nên ràng buộc "phải load CUDA sau khi fork mining" không còn bắt buộc như trước.
# Nhờ vậy Cell 8c (tune lambda) đã được CHUYỂN LÊN ngay sau cell này (thay vì đặt sau toàn bộ
# mining + fine-tune reranker như bản cũ) — đúng với mặt lý thuyết (mục 2: hybrid retrieval là
# Stage-1), và quan trọng hơn: đảm bảo LAMBDA_HYBRID đã được tune xong TRƯỚC khi training loop
# (Cell 14) dùng nó để eval dev mỗi epoch / chọn best checkpoint (xem note đầu Cell 8c).
# Cell 7b — Encode toàn bộ short-chunk corpus bằng Vietnamese_Embedding_v2 (AITeamVN) (dense, cho stage-1)
# Cùng cấp chunk với BM25 (Cell 7) để 2 tín hiệu có thể kết hợp (hybrid) ở mức chunk, sau đó
# mới gộp điểm về cấp document giống hệt cách BM25 đang làm.
from sentence_transformers import SentenceTransformer

torch.cuda.empty_cache()  # dọn VRAM còn sót lại từ lần chạy OOM trước (nếu có)

embed_model = SentenceTransformer(
    EMBED_MODEL_NAME,
    device=DEVICE,
    model_kwargs={
        "torch_dtype": torch.float16,  # giảm ~50% VRAM, gần như không đổi chất lượng
        **_attn_impl_kwargs(),         # thêm attn_implementation="flash_attention_2" nếu USE_FLASH_ATTENTION_2=True
    },
)
# ĐÃ SỬA: theo đúng khuyến nghị chính thức của AITeamVN/Vietnamese_Embedding_v2 (max sequence length
# huấn luyện = 2048 token) — short-chunk (<= SHORT_CHUNK_CHARS=1000 ký tự) vẫn nằm rất sâu trong
# ngân sách này nên không đổi hành vi, chỉ đảm bảo đúng cấu hình model.
embed_model.max_seq_length = 2048

_CHUNK_EMB_CACHE_PATH = os.path.join(
    "DSC2026-IntelliJQK", "cache", f"vn_embed_v2_short_chunks_{SHORT_CHUNK_CHARS}_7.npy"
)

if os.path.exists(_CHUNK_EMB_CACHE_PATH):
    chunk_embeddings = np.load(_CHUNK_EMB_CACHE_PATH)
    print(f"Đã load dense embeddings từ {_CHUNK_EMB_CACHE_PATH}: shape={chunk_embeddings.shape}")
else:
    short_chunk_texts = [short_chunk_text_of[cid] for cid in short_chunk_ids]
    chunk_embeddings = embed_model.encode(
        short_chunk_texts,
        batch_size=EMBED_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)
    np.save(_CHUNK_EMB_CACHE_PATH, chunk_embeddings)
    print(f"Đã encode {len(short_chunk_ids)} short-chunk -> embeddings shape={chunk_embeddings.shape}, "
          f"lưu tại {_CHUNK_EMB_CACHE_PATH}")


Loading weights: 100%|██████████| 391/391 [00:07<00:00, 49.17it/s]
/mnt/mmlab2024nas/trantran/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


Đã load dense embeddings từ /mnt/mmlab2024nas/trantran/vn_embed_v2_short_chunks_2000_7.npy: shape=(442255, 1024)


In [18]:
# Cell 8c — Tune lambda (trọng số BM25 trong hybrid) trên tập VALIDATION (dev)
# ĐÃ CHUYỂN LÊN ĐÚNG VỊ TRÍ (trước đây cell này nằm SAU toàn bộ mining + fine-tune reranker,
# tức SAU Cell 15 — lý do cũ là lo ngại fork+CUDA khi mining còn dùng multiprocessing, xem note
# đã lỗi thời ở cell encode dense phía trên). Mining giờ không còn multiprocessing/fork nữa nên
# ràng buộc đó không còn, và quan trọng hơn: ĐỂ CELL NÀY SAU Cell 14 (training loop) là 1 BUG
# THẬT — vòng lặp training đánh giá dev sau mỗi epoch bằng hybrid_retrieve(), hàm này dùng
# LAMBDA_HYBRID mặc định (0.4, Cell 2) NẾU lambda chưa được tune, vì Cell 8c (nơi ghi đè
# LAMBDA_HYBRID bằng giá trị tune xong) khi đó CHƯA chạy -> checkpoint 'tốt nhất' (best_epoch,
# BEST_DIR) bị chọn dựa trên Stage-1 CHƯA tune, có thể không thật sự là checkpoint tốt nhất dưới
# Stage-1 đã tune (dùng ở Cell 18 sau đó). Đặt Cell 8c NGAY SAU khi build xong Stage-1 (BM25 +
# Dense, Cell 7b) và TRƯỚC mining/fine-tune reranker đảm bảo LAMBDA_HYBRID đã đúng giá trị tune
# trước khi bất kỳ cell nào (kể cả training loop) dùng tới.
# Grid search theo Recall@TOP_K_STAGE1 của stage-1 hybrid_retrieve trên dev_samples (proxy tốt cho
# chất lượng ứng viên đưa vào reranker — recall càng cao thì reranker càng có cơ hội tìm ra đúng
# điều luật liên quan trong top_k_stage1 ứng viên).
#
# TỐI ƯU: bm25_raw/dense_raw của 1 câu hỏi KHÔNG đổi theo lambda (chỉ cách kết hợp 2 điểm đổi) —
# nhưng bản gốc gọi hybrid_retrieve(question, lam=...) cho MỖI lambda trong LAMBDA_GRID, khiến
# bm25_scores_all()/dense_scores_all() (encode câu hỏi qua GPU + tính điểm toàn corpus) bị tính lại
# 11 LẦN/câu hỏi một cách dư thừa. Ở đây ta PRECOMPUTE bm25_norm/dense_norm 1 LẦN/câu hỏi (min-max
# normalize sẵn), cache lại, rồi mỗi lambda chỉ cần cộng có trọng số (rẻ, không đụng GPU/BM25 nữa).
#
# ĐÃ SỬA (đồng bộ với Cell 8/8b): KHÔNG còn cắt top-K_STAGE1_CHUNKS chunk thô trước khi gộp về
# document nữa. stage1_recall_at_lambda giờ gộp max-per-doc trên TOÀN BỘ short-chunk (dùng chung
# _aggregate_scores_to_doc, Cell 8) rồi mới sort/cắt top_k ở CẤP DOCUMENT — ĐÚNG KHỚP với cách
# hybrid_retrieve (Cell 8b) hoạt động lúc inference thật, nên kết quả tune lambda ở đây phản ánh
# đúng recall thực tế của Stage-1 sau khi sửa.

def _precompute_stage1_norm_scores(samples):
    """Tính trước bm25_norm/dense_norm (đã min-max normalize) cho từng câu hỏi trong `samples`, dùng
    chung cho toàn bộ LAMBDA_GRID bên dưới thay vì tính lại mỗi lambda. Trả về list[(bm25_norm,
    dense_norm)] cùng thứ tự với `samples`."""
    cached = []
    for s in tqdm(samples, desc="Precompute BM25+Dense scores (dùng chung cho mọi lambda)"):
        bm25_raw = bm25_scores_all(s["question"])
        dense_raw = dense_scores_all(s["question"])
        cached.append((_minmax_normalize(bm25_raw), _minmax_normalize(dense_raw)))
    return cached


def stage1_recall_at_lambda(lam, samples, cached_norm_scores, top_k=TOP_K_STAGE1):
    """Chỉ cộng có trọng số 2 mảng điểm ĐÃ NORMALIZE SẴN (cached_norm_scores, TOÀN BỘ short-chunk)
    rồi gộp max-per-doc trên TOÀN BỘ chunk (_aggregate_scores_to_doc, Cell 8) để lấy top_k document
    — không gọi lại bm25_scores_all/dense_scores_all nữa (xem _precompute_stage1_norm_scores), và
    không cắt top-K chunk thô trước khi gộp (khớp đúng hybrid_retrieve, Cell 8b)."""
    recalls = []
    for s, (bm25_norm, dense_norm) in zip(samples, cached_norm_scores):
        combined = lam * bm25_norm + (1 - lam) * dense_norm
        pred = [d for d, _ in _aggregate_scores_to_doc(combined, top_k)]
        r, _ = recall_precision(pred, s["answers"])
        recalls.append(r)
    return float(np.mean(recalls))


# _dev_stage1_norm_scores = _precompute_stage1_norm_scores(dev_samples)

# LAMBDA_GRID = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
# lambda_search_results = {
#     lam: stage1_recall_at_lambda(lam, dev_samples, _dev_stage1_norm_scores)
#     for lam in tqdm(LAMBDA_GRID, desc="Tuning lambda")
# }

# for lam, r in sorted(lambda_search_results.items()):
#     print(f"  lambda={lam:.1f} -> stage-1 recall@{TOP_K_STAGE1}={r:.4f}"
#           + ("  (BM25-only)" if lam == 1.0 else "  (Dense-only)" if lam == 0.0 else ""))

# LAMBDA_HYBRID = max(lambda_search_results, key=lambda_search_results.get)
# MINING_LAMBDA = LAMBDA_HYBRID  # dùng cho mining (Cell 10c) và fine-tune reranker (Cell 14)
# print(f"\n-> Chọn lambda={LAMBDA_HYBRID:.1f} "
#       f"(stage-1 recall@{TOP_K_STAGE1}={lambda_search_results[LAMBDA_HYBRID]:.4f}) cho hybrid_retrieve.")


## 4. Mining positive/negative samples bằng HYBRID để fine-tune cross-encoder (5:1)

Với mỗi câu hỏi: positive và hard-negative đều được chọn từ **cùng một HYBRID scoring** trên
short-chunk, với `MINING_LAMBDA=0.2`: `0.2*BM25_norm + 0.8*Dense_norm`. Như vậy bước mining
không còn dùng BM25-only cho positive hoặc negative. Sau khi chọn short-chunk đại diện, cả positive
và negative đều được giữ nguyên short-chunk bằng đúng `short-chunk trực tiếp` như pipeline
chính.

**Cách mining (short-chunk HYBRID + mở rộng theo Điều), tách 2 tầng để tận dụng cache:**

- **Tầng 1 — `mine_candidate_pools()` (ĐẮT, cache ra đĩa):** với mỗi câu hỏi, tính BM25 + Dense trên
  toàn bộ short-chunk, min-max normalize từng tín hiệu rồi cộng `MINING_LAMBDA * BM25_norm +
  (1-MINING_LAMBDA) * Dense_norm`. Từ cùng mảng điểm hybrid này: (a) chọn short-chunk điểm cao nhất
  trong từng document gold làm positive representative, và (b) chọn short-chunk điểm cao nhất của
  tối đa `NEG_MINING_POOL_K` document khác nhau làm hard-negative pool.
- **Tầng 2 — `build_training_pairs_from_pool()` (RẺ, chạy lại mỗi epoch):** `random.sample()` từ
  negative pool đã mine sẵn ở tầng 1, rồi dùng trực tiếp short-chunk bằng đúng
  `short-chunk trực tiếp` / `short-chunk trực tiếp`, giống pipeline chính.

Do đó positive và negative trong training đều được mining bởi **một scoring function duy nhất**;
không còn trường hợp positive = BM25-only nhưng negative = hybrid.


In [ ]:
# Cell 10c — Mining positive/negative trên SHORT-chunk index — CHỈ TẦNG 1.
#
# Candidate pool lưu (doc_id, global_idx) để có thể tái sử dụng đúng short-chunk đã mine.
# Không còn hàm short-chunk trực tiếp(): training và inference đều dùng THẲNG
# short_chunk_text_of[chunk_id] làm passage cho reranker.
from collections import defaultdict

# ---- Lập chỉ mục doc_id -> list[index vào short_chunk_ids] MỘT LẦN ----
_doc_to_short_chunk_idx = defaultdict(list)
_short_global_to_local_pos = {}
for _idx, _cid in enumerate(short_chunk_ids):
    _doc = short_chunk_doc_of[_cid]
    _short_global_to_local_pos[_idx] = len(_doc_to_short_chunk_idx[_doc])
    _doc_to_short_chunk_idx[_doc].append(_idx)
print(f"Đã lập chỉ mục short-chunk theo document: {len(_doc_to_short_chunk_idx)} document.")


def _hybrid_mining_scores(question, q_tokens=None, q_emb=None, lam=MINING_LAMBDA):
    """Tính HYBRID score trên TOÀN BỘ short-chunk cho mining."""
    bm25_raw = bm25_scores_all(question, q_tokens=q_tokens)
    dense_raw = dense_scores_all(question, q_emb=q_emb)
    return lam * _minmax_normalize(bm25_raw) + (1.0 - lam) * _minmax_normalize(dense_raw)


def _hybrid_negative_pool(question, gold, pool_k, q_tokens=None, q_emb=None):
    """Trả về tối đa pool_k document KHÁC gold, xếp theo HYBRID score.
    Mỗi document giữ global_idx của short-chunk điểm cao nhất."""
    scores = _hybrid_mining_scores(question, q_tokens=q_tokens, q_emb=q_emb, lam=MINING_LAMBDA)
    order = np.argsort(scores)[::-1]
    seen_docs = set()
    pool = []
    for idx in order:
        if len(pool) >= pool_k:
            break
        cid = short_chunk_ids[idx]
        d = short_chunk_doc_of[cid]
        if d in gold or d in seen_docs:
            continue
        seen_docs.add(d)
        pool.append((d, int(idx)))
    return pool, scores


def _best_short_chunk_idx_in_doc(doc_id, scores):
    """global_idx của short-chunk có HYBRID score cao nhất trong document."""
    idxs = _doc_to_short_chunk_idx.get(doc_id)
    if not idxs:
        return None
    return max(idxs, key=lambda i: scores[i])


_CANDIDATE_POOL_CACHE_PATH = os.path.join(
    "DSC2026-IntelliJQK", "cache",
    f"7_vn_embed_v2_mining_candidate_pools_{SHORT_CHUNK_CHARS}_pk{NEG_MINING_POOL_K}_lam{MINING_LAMBDA:.1f}.pkl"
)


def mine_candidate_pools(samples, pool_k=NEG_MINING_POOL_K,
                          cache_path=_CANDIDATE_POOL_CACHE_PATH, force_remine=False):
    """TẦNG 1: mining positive + negative bằng CÙNG HYBRID score trên short-chunk.

    Với mỗi sample: tính BM25 + Dense trên toàn bộ short-chunk; chọn short-chunk điểm cao
    nhất trong mỗi gold document làm positive representative; ngoài gold chọn tối đa pool_k
    document khác nhau, mỗi document lấy short-chunk điểm cao nhất.
    """
    current_qids = {s["qid"] for s in samples}
    if not force_remine and os.path.exists(cache_path):
        with open(cache_path, "rb") as f:
            cached = pickle.load(f)
        if set(cached.keys()) == current_qids:
            print(f"Đã load HYBRID mining candidate pool từ cache: {cache_path} ({len(cached)} câu hỏi).")
            return cached
        print(f"Cache {cache_path} không khớp bộ câu hỏi hiện tại ({len(cached)} vs {len(current_qids)}) -> mine lại.")

    pools = {}
    n_skipped = 0
    for sample in tqdm(samples, desc=f"Mining candidate pool (HYBRID lambda={MINING_LAMBDA:.1f})"):
        qid, question, gold = sample["qid"], sample["question"], set(sample["answers"])
        q_tokens = _question_token_cache_fast.get(qid)
        if q_tokens is None:
            q_tokens = vi_tokenize_short(question)

        neg_pool, hybrid_scores = _hybrid_negative_pool(
            question, gold, pool_k, q_tokens=q_tokens, q_emb=None
        )

        positives = []
        for g in gold:
            if g not in corpus:
                n_skipped += 1
                continue
            global_idx = _best_short_chunk_idx_in_doc(g, hybrid_scores)
            if global_idx is None:
                n_skipped += 1
                continue
            positives.append((g, int(global_idx)))

        pools[qid] = {"positives": positives, "neg_pool": neg_pool}

    if n_skipped:
        print(f"Bỏ qua {n_skipped} document gold không tìm được short-chunk đại diện.")

    with open(cache_path, "wb") as f:
        pickle.dump(pools, f)
    print(f"Đã mine xong HYBRID candidate pool cho {len(pools)} câu hỏi, lưu tại {cache_path}")
    return pools


fit_candidate_pools = mine_candidate_pools(fit_samples, pool_k=NEG_MINING_POOL_K)
print("Số câu hỏi có pool:", len(fit_candidate_pools))

# ---- Dọn RAM: corpus (text gốc chưa chunk) không còn được dùng ở bất kỳ cell nào phía sau ----
del corpus
gc.collect()
ctypes.CDLL("libc.so.6").malloc_trim(0)


Đã lập chỉ mục short-chunk theo document: 8511 document.
Đã load HYBRID mining candidate pool từ cache: /mnt/mmlab2024nas/trantran/7_vn_embed_v2_mining_candidate_pools_2000_pk100_lam0.4.pkl (6300 câu hỏi).
Số câu hỏi có pool: 6300


1

In [20]:
# Cell 10c2 — TẦNG 1.5 (MỚI, RẺ — chạy mỗi epoch, hoặc mỗi N step nếu gọi thủ công với `qids`):
# refresh_pool_scores(pools) — forward pass reranker (điểm "yes") trên TỪNG candidate ĐÃ CÓ SẴN
# trong neg_pool (mine ở TẦNG 1 / Cell 10c, KHÔNG quét lại corpus, KHÔNG gọi lại BM25/dense) để cập
# nhật "độ khó hiện tại" của mỗi candidate DƯỚI GÓC NHÌN của chính checkpoint đang train.
#
# Yêu cầu: Cell 10c đã chạy xong (fit_candidate_pools trong session) VÀ
# Cell 12 (load reranker_model + rerank_score) đã chạy. Ở vị trí notebook hiện tại (trước Cell 12),
# reranker_model CHƯA tồn tại -> cell này chỉ ĐỊNH NGHĨA hàm, KHÔNG gọi ngay; refresh_pool_scores()
# thật sự được gọi mỗi epoch trong training loop (Cell 14), SAU KHI Cell 12/13 đã chạy.
#
# _pool_neg_scores: dict qid -> np.ndarray điểm reranker hiện tại, CÙNG THỨ TỰ với
# fit_candidate_pools[qid]["neg_pool"]. Cell 10d (TẦNG 2) đọc dict này để chọn negative theo rank
# MỚI NHẤT; nếu qid chưa có trong dict (chưa refresh lần nào), Cell 10d tự fallback về
# random.sample() trên toàn bộ neg_pool giống hành vi gốc.

_pool_neg_scores = {}  # qid -> np.ndarray, cùng thứ tự với pools[qid]["neg_pool"]


@torch.no_grad()
def refresh_pool_scores(pools, samples=None, batch_size=POOL_SCORE_BATCH_SIZE, qids=None, desc=None):
    """TẦNG 1.5: forward pass reranker_model (checkpoint HIỆN TẠI — dù đang train dở hay vừa load)
    trên từng candidate trong neg_pool của từng câu hỏi, lưu điểm 'yes' (xác suất liên quan) vào
    _pool_neg_scores[qid]. Chỉ tính lại rerank_score() trên candidate ĐÃ CÓ SẴN trong pool (TẦNG 1)
    -> rẻ hơn hẳn TẦNG 1 (không có bm25.get_scores()/dense encode nào), an toàn để gọi mỗi epoch.

    pools: fit_candidate_pools (hoặc tập con) từ mine_candidate_pools (Cell 10c).
    samples: nếu truyền vào (vd. fit_samples), dùng để tra `question` theo qid; None -> dùng
        qid_to_question (map dựng sẵn từ fit_samples ngay dưới đây).
    qids: giới hạn chỉ refresh một tập con qid (vd. muốn refresh mỗi N step thay vì mỗi epoch, chỉ
        cần truyền các qid vừa xuất hiện trong N step gần nhất) thay vì toàn bộ pools.
    """
    was_training = reranker_model.training
    reranker_model.eval()

    target_qids = list(qids) if qids is not None else list(pools.keys())
    q_lookup = qid_to_question if samples is None else {s["qid"]: s["question"] for s in samples}

    for qid in tqdm(target_qids, desc=desc or "Refresh pool scores (TẦNG 1.5)"):
        pool_entry = pools.get(qid)
        if pool_entry is None or not pool_entry["neg_pool"]:
            continue
        question = q_lookup.get(qid)
        if question is None:
            continue

        neg_pool = pool_entry["neg_pool"]
        texts, valid_idx = [], []
        for i, (d, global_idx) in enumerate(neg_pool):
            cid = short_chunk_ids[global_idx]
            short_text = short_chunk_text_of[cid]
            if not short_text or not short_text.strip():
                continue
            texts.append(short_text)
            valid_idx.append(i)
        if not texts:
            continue

        scores = rerank_score(question, texts, batch_size=batch_size)

        arr = np.full(len(neg_pool), -np.inf, dtype=np.float32)
        for i, sc in zip(valid_idx, scores):
            arr[i] = sc
        _pool_neg_scores[qid] = arr

    if was_training:
        reranker_model.train()


qid_to_question = {s["qid"]: s["question"] for s in fit_samples}

# KHÔNG gọi refresh_pool_scores() ở đây: reranker_model chưa được load ở vị trí này trong notebook
# (Cell 12 nằm SAU). Lần refresh ĐẦU TIÊN diễn ra trong training loop (Cell 14), trước epoch 2 trở
# đi (sau khi checkpoint đã được fine-tune ít nhất 1 epoch) — xem đoạn "refresh_pool_scores(...)"
# được chèn vào đầu vòng lặp epoch ở Cell 14.
print("Đã định nghĩa refresh_pool_scores() (TẦNG 1.5). Sẽ được gọi mỗi epoch trong training loop "
      "(Cell 14), sau khi Cell 12/13 đã chạy.")


Đã định nghĩa refresh_pool_scores() (TẦNG 1.5). Sẽ được gọi mỗi epoch trong training loop (Cell 14), sau khi Cell 12/13 đã chạy.


In [21]:
# Cell 10c3 — TẦNG 1.5 (bổ sung): refresh_pool_pos_scores() + ngưỡng false-negative ĐỘNG
# Vấn đề với FALSE_NEG_RERANK_THRESHOLD tĩnh (Cell 2, = 0.9): ngưỡng "điểm reranker cao" mang ý
# nghĩa khác nhau tuỳ epoch — model mới train 1 epoch thường tự tin thấp hơn hẳn model đã train
# nhiều epoch, nên 1 ngưỡng cố định có thể QUÁ CHẶT ở epoch đầu (không lọc được false-negative nào)
# hoặc QUÁ LỎNG ở epoch cuối (lọc nhầm cả hard-negative thật). Cách khắc phục: lấy percentile thấp
# (mặc định p10) của điểm reranker HIỆN TẠI trên chính các known-positive — tức "ngưỡng đúng" là
# mức điểm mà ~90% positive thật đạt được ở checkpoint đang xét, tự khớp theo độ tự tin thực tế của
# model tại từng epoch, không cần đoán số cố định.
#
# refresh_pool_pos_scores() giống hệt refresh_pool_scores() (Cell 10c2) nhưng chấm điểm reranker
# HIỆN TẠI cho `positives` thay vì `neg_pool` — RẺ hơn nhiều vì mỗi câu hỏi chỉ có 1-3 positive
# (so với NEG_MINING_POOL_K=50 negative), nên gọi thêm mỗi epoch gần như không tốn chi phí đáng kể.
#
# Yêu cầu: giống Cell 10c2, hàm short-chunk trực tiếp/rerank_score/qid_to_question phải đã có
# sẵn trong session (Cell 10c, Cell 10c2, Cell 12). Ở vị trí notebook hiện tại (trước Cell 12),
# reranker_model CHƯA tồn tại -> cell này chỉ ĐỊNH NGHĨA hàm, KHÔNG gọi ngay; refresh_pool_pos_scores()
# thật sự được gọi mỗi epoch trong training loop (Cell 14), NGAY SAU refresh_pool_scores().

_pool_pos_scores = {}  # qid -> np.ndarray điểm reranker hiện tại trên known-positive của câu hỏi đó
                        # (không nhất thiết cùng độ dài với pools[qid]["positives"] nếu có positive
                        # bị bỏ qua do không xử lý được short-chunk — chỉ dùng để gộp lại tính
                        # percentile toàn dataset, không cần giữ alignment vị trí như _pool_neg_scores)


@torch.no_grad()
def refresh_pool_pos_scores(pools, samples=None, batch_size=POOL_SCORE_BATCH_SIZE, qids=None, desc=None):
    """TẦNG 1.5 (positives): forward pass reranker_model (checkpoint HIỆN TẠI) trên từng known-
    positive của từng câu hỏi, lưu điểm 'yes' vào _pool_pos_scores[qid]. Dùng cho
    compute_dynamic_false_neg_threshold() ngay dưới đây. RẺ hơn hẳn refresh_pool_scores (Cell 10c2)
    vì mỗi câu hỏi chỉ có 1-3 positive thay vì cả neg_pool.

    pools: fit_candidate_pools (hoặc tập con) từ mine_candidate_pools (Cell 10c).
    samples/qids: giống hệt refresh_pool_scores (Cell 10c2)."""
    was_training = reranker_model.training
    reranker_model.eval()

    target_qids = list(qids) if qids is not None else list(pools.keys())
    q_lookup = qid_to_question if samples is None else {s["qid"]: s["question"] for s in samples}

    for qid in tqdm(target_qids, desc=desc or "Refresh positive scores (TẦNG 1.5 - positives)"):
        pool_entry = pools.get(qid)
        if pool_entry is None or not pool_entry["positives"]:
            continue
        question = q_lookup.get(qid)
        if question is None:
            continue

        positives = pool_entry["positives"]
        texts = []
        for g, global_idx in positives:
            cid = short_chunk_ids[global_idx]
            short_text = short_chunk_text_of[cid]
            if not short_text or not short_text.strip():
                continue
            texts.append(short_text)
        if not texts:
            continue

        scores = rerank_score(question, texts, batch_size=batch_size)
        _pool_pos_scores[qid] = np.array(scores, dtype=np.float32)

    if was_training:
        reranker_model.train()


_threshold_ema = None  # EMA của threshold động, làm mượt qua các epoch (xem compute_dynamic_false_neg_threshold)


def compute_dynamic_false_neg_threshold(low_percentile=10, min_thr=0.5, max_thr=0.999, min_n=50,
                                         ema_alpha=0.3):
    """Threshold = percentile thấp (mặc định p10) của điểm known-positive TOÀN BỘ dataset ở
    checkpoint HIỆN TẠI (_pool_pos_scores, vừa refresh bởi refresh_pool_pos_scores() ngay phía trên).

    Ý nghĩa: chỉ coi 1 candidate trong neg_pool là "nghi ngờ false-negative" khi điểm reranker của
    nó cao ngang mức mà ~90% positive thật đạt được ở checkpoint hiện tại -> tự khớp theo độ tự tin
    thực tế của model tại epoch đó, thay vì dùng đúng 1 số cố định (FALSE_NEG_RERANK_THRESHOLD=0.9,
    Cell 2) suốt quá trình train.

    min_thr/max_thr: kẹp threshold trong khoảng an toàn (tránh percentile ra giá trị quá thấp/quá
    cao nếu phân bố điểm positive lệch bất thường ở vài epoch đầu).
    min_n: số known-positive score tối thiểu cần có (gộp toàn dataset) mới tin percentile — nếu
    chưa đủ (vd. epoch 1, refresh_pool_pos_scores() chưa chạy lần nào) -> trả về None, nơi gọi hàm
    này (Cell 14) sẽ tự giữ nguyên FALSE_NEG_RERANK_THRESHOLD tĩnh cũ.
    ema_alpha: hệ số làm mượt EMA qua các epoch, tránh threshold nhảy loạn nếu percentile của 1
    epoch nào đó bị nhiễu (vd. ít known-positive xử lý được short-chunk hơn bình thường).
    """
    global _threshold_ema
    all_pos = np.concatenate([v for v in _pool_pos_scores.values() if len(v) > 0]) \
        if _pool_pos_scores else np.array([])
    if len(all_pos) < min_n:
        return None  # chưa đủ dữ liệu -> giữ nguyên threshold tĩnh cũ (Cell 14 tự bỏ qua khi None)

    raw_thr = float(np.clip(np.percentile(all_pos, low_percentile), min_thr, max_thr))
    _threshold_ema = raw_thr if _threshold_ema is None \
        else ema_alpha * raw_thr + (1 - ema_alpha) * _threshold_ema
    return float(_threshold_ema)


print("Đã định nghĩa refresh_pool_pos_scores() và compute_dynamic_false_neg_threshold() (TẦNG 1.5 "
      "- positives). Sẽ được gọi mỗi epoch trong training loop (Cell 14), ngay sau "
      "refresh_pool_scores(), để cập nhật FALSE_NEG_RERANK_THRESHOLD động thay vì giữ cố định 0.9.")

Đã định nghĩa refresh_pool_pos_scores() và compute_dynamic_false_neg_threshold() (TẦNG 1.5 - positives). Sẽ được gọi mỗi epoch trong training loop (Cell 14), ngay sau refresh_pool_scores(), để cập nhật FALSE_NEG_RERANK_THRESHOLD động thay vì giữ cố định 0.9.


In [ ]:
# Cell 10d — TẦNG 2 (RẺ — dùng trực tiếp SHORT-chunk): build_training_pairs_from_pool() -> gọi lại mỗi epoch (Cell 14/rebuild_train_loader)
# để có bộ negative khác nhau, không cần tính lại BM25. Yêu cầu Cell 10c (mine_candidate_pools) và
# Cell 10c2 (refresh_pool_scores, định nghĩa _pool_neg_scores) đã chạy, và Cell 6b đã chạy (định
# nghĩa short-chunk trực tiếp mà short-chunk trực tiếp gọi).
#
# ĐÃ SỬA (so với bản chỉ dùng random.sample() thuần): chọn negative giờ gồm 2 bước.
#   Bước 1 (MỚI) — lọc "nghi ngờ false-negative" bằng XÁC NHẬN CHÉO 2 NGUỒN ĐỘC LẬP
#     (_suspected_false_negatives): chỉ loại candidate khi CẢ hybrid rank GỐC (TẦNG 1, BM25+Dense —
#     KHÔNG phụ thuộc model đang train) LẪN điểm reranker HIỆN TẠI (TẦNG 1.5, checkpoint đang train)
#     đều đồng thuận là rất liên quan -> tránh dùng chính model đang train để tự quyết định false-
#     negative (feedback loop: model lệch rồi tự "xác nhận" cái lệch đó). Nếu chỉ 1 trong 2 nguồn
#     nghi ngờ -> GIỮ LẠI làm negative bình thường (thận trọng thiên về giữ, không thiên về loại).
#   Bước 2 — trong phần còn lại, ưu tiên rank TẦNG 1.5 (_pool_neg_scores — điểm reranker mới nhất):
#     sort giảm dần theo điểm đó, lấy top NEG_HARD_TOP_FRAC candidate khó nhất làm "hard pool", rồi
#     random.sample() trong đó để vẫn có bộ negative khác nhau mỗi epoch (tránh học tủ 1 bộ cố định).
# Nếu qid CHƯA có _pool_neg_scores (chưa refresh lần nào, vd. epoch đầu tiên) -> bỏ qua bước 1 (không
# đủ 2 nguồn để xác nhận chéo) và bước 2 fallback random.sample() trên toàn bộ neg_pool, giống hệt
# hành vi gốc.
#
# Bản streaming: ghi pairs ra đĩa theo batch pickle nối tiếp thay vì giữ toàn bộ list `pairs`
# trong RAM khi build (đỉnh RAM ở bản cũ = build xong + shuffle xong cùng lúc tồn tại 2 bản).
import gc
import pickle


# Thống kê số candidate bị loại vì nghi ngờ false-negative (2 nguồn đồng thuận) — reset mỗi lần
# build_training_pairs_from_pool*/được gọi, chỉ để in ra theo dõi, KHÔNG ảnh hưởng logic.
_false_neg_stats = {"n_checked_qid": 0, "n_suspected_total": 0}


def _suspected_false_negatives(qid, neg_pool):
    """Trả về set các VỊ TRÍ (index vào neg_pool) bị nghi ngờ là false-negative (thật ra có thể là
    positive chưa được gán nhãn), dựa trên XÁC NHẬN CHÉO 2 nguồn ĐỘC LẬP — KHÔNG chỉ dùng điểm của
    model đang train, để tránh feedback loop (model tự lệch rồi tự "xác nhận" cái lệch đó):

      (a) điểm reranker HIỆN TẠI (TẦNG 1.5, _pool_neg_scores — phụ thuộc checkpoint đang train)
          >= FALSE_NEG_RERANK_THRESHOLD (ĐÃ SỬA: từ epoch 2 trở đi, giá trị này được Cell 14
          ghi đè ĐỘNG mỗi epoch bằng compute_dynamic_false_neg_threshold() — Cell 10c3 — thay vì
          cố định 0.9 suốt quá trình train), VÀ
      (b) hybrid rank GỐC (TẦNG 1, BM25+Dense — KHÔNG phụ thuộc model đang train) nằm trong top
          FALSE_NEG_HYBRID_TOP_FRAC đầu của neg_pool. neg_pool đã được _hybrid_negative_pool (Cell
          10c) sort giảm dần theo hybrid score ngay lúc mine, nên chỉ cần dùng VỊ TRÍ trong neg_pool
          làm proxy cho hybrid rank, không cần lưu thêm điểm hybrid nào.

    CHỈ loại khi CẢ (a) VÀ (b) cùng đúng (thận trọng thiên về GIỮ). Nếu chưa có _pool_neg_scores cho
    qid (TẦNG 1.5 chưa refresh) -> không đủ 2 nguồn để xác nhận chéo -> trả về set rỗng (không loại
    gì cả), giữ nguyên hành vi gốc.
    """
    scores_arr = _pool_neg_scores.get(qid)
    if scores_arr is None or len(scores_arr) != len(neg_pool):
        return set()

    hybrid_cutoff = max(1, int(len(neg_pool) * FALSE_NEG_HYBRID_TOP_FRAC))
    suspected = {
        i for i in range(hybrid_cutoff)          # (b) nằm trong top hybrid rank gốc
        if scores_arr[i] >= FALSE_NEG_RERANK_THRESHOLD  # (a) VÀ điểm reranker hiện tại rất cao
    }
    if suspected:
        _false_neg_stats["n_suspected_total"] += len(suspected)
    _false_neg_stats["n_checked_qid"] += 1
    return suspected


def _sample_negatives_for_qid(qid, neg_pool, gold, n_neg):
    """Chọn tối đa n_neg negative (doc_id, global_idx) từ neg_pool cho câu hỏi qid.

    Bước 1 — lọc nghi ngờ false-negative bằng xác nhận chéo 2 nguồn (xem
    _suspected_false_negatives): loại các candidate mà CẢ hybrid rank gốc (TẦNG 1) LẪN điểm reranker
    hiện tại (TẦNG 1.5) đều đồng thuận là rất liên quan, tránh dùng chính unlabeled true-positive làm
    negative để train (feedback loop). Nếu chỉ 1 nguồn nghi ngờ -> vẫn giữ, không loại.

    Bước 2 (ĐÃ SỬA) — MIX hard/easy thay vì lấy 100% từ hard pool: chia n_neg theo NEG_MIX_HARD_FRAC
    (Cell 2). Trước đây toàn bộ n_neg lấy từ top NEG_HARD_TOP_FRAC candidate khó nhất theo điểm
    reranker HIỆN TẠI -> tạo feedback loop (model tự chọn negative theo chính điểm số của nó, càng
    train càng thu hẹp trọng tâm vào 1 vùng hẹp, dễ mất khả năng tổng quát trên dev).
      - n_hard = round(n_neg * NEG_MIX_HARD_FRAC): sample từ top NEG_HARD_TOP_FRAC candidate khó nhất
        (theo điểm reranker TẦNG 1.5 mới nhất) — giữ nguyên vai trò hard-negative như bản gốc.
      - n_easy = n_neg - n_hard: sample RANDOM ĐỀU trên toàn bộ eligible pool (KHÔNG giới hạn top
        hard, loại trừ candidate đã chọn ở phần hard), để model vẫn tiếp xúc với negative "dễ"/hạng
        thấp mỗi epoch thay vì chỉ toàn candidate khó nhất -> giảm nguy cơ feedback loop.
      - Nếu chưa có điểm TẦNG 1.5 cho qid này (vd. epoch đầu tiên, trước lần refresh_pool_scores()
        đầu tiên) -> fallback random.sample() trên toàn bộ eligible pool, giống hệt hành vi gốc
        (chưa có điểm để xếp hạng hard/easy).

    Nếu vẫn thiếu (n_neg > số candidate khả dụng), bổ sung thêm document ngẫu nhiên ngoài gold/đã
    chọn, giống cơ chế fallback gốc."""
    suspected_idx = _suspected_false_negatives(qid, neg_pool)
    eligible_idx = [i for i in range(len(neg_pool)) if i not in suspected_idx]
    if not eligible_idx:
        # Hiếm khi xảy ra (toàn bộ pool bị nghi ngờ) -> không để pool rỗng, dùng lại toàn bộ neg_pool
        # gốc (thận trọng thiên về giữ, thà lẫn vài false-negative còn hơn mất hết negative).
        eligible_idx = list(range(len(neg_pool)))

    scores_arr = _pool_neg_scores.get(qid)
    if scores_arr is not None and len(scores_arr) == len(neg_pool):
        order = sorted(eligible_idx, key=lambda i: scores_arr[i], reverse=True)  # khó nhất trước
        hard_k = max(1, int(len(order) * NEG_HARD_TOP_FRAC))
        hard_pool = order[:hard_k]

        # ---- Bước 2a: phần "hard" — sample trong nhóm khó nhất ----
        n_hard = min(n_neg, round(n_neg * NEG_MIX_HARD_FRAC), len(hard_pool))
        hard_chosen = random.sample(hard_pool, n_hard) if n_hard > 0 else []

        # ---- Bước 2b: phần "easy/mix" — sample random đều trên eligible pool, loại trừ hard đã chọn ----
        n_easy = n_neg - len(hard_chosen)
        hard_chosen_set = set(hard_chosen)
        easy_candidates = [i for i in eligible_idx if i not in hard_chosen_set]
        easy_chosen = random.sample(easy_candidates, min(n_easy, len(easy_candidates))) if n_easy > 0 else []

        chosen_idx = hard_chosen + easy_chosen
    else:
        # Chưa refresh TẦNG 1.5 cho qid này (vd. epoch đầu tiên, trước khi Cell 14 gọi
        # refresh_pool_scores lần đầu) -> fallback rank hybrid ban đầu, giống hệt bản gốc.
        chosen_idx = random.sample(eligible_idx, min(n_neg, len(eligible_idx)))

    negs = [neg_pool[i] for i in chosen_idx]

    if len(negs) < n_neg:
        picked_docs = {d for d, _ in negs}
        extra_docs = [d for d in doc_ids_all if d not in gold and d not in picked_docs]
        random.shuffle(extra_docs)
        for d in extra_docs[: n_neg - len(negs)]:
            idxs = _doc_to_short_chunk_idx.get(d)
            if idxs:
                negs.append((d, idxs[0]))
    return negs


def build_training_pairs_from_pool(pools, samples, neg_per_pos=NEG_PER_POS,
                                    desc="Resample negative từ candidate pool (TẦNG 2 - rẻ)"):
    """Sinh training pairs (question, chunk_id, chunk_text, label) từ candidate pool đã mine sẵn ở
    TẦNG 1 (mine_candidate_pools, Cell 10c). KHÔNG tính lại BM25 — chỉ chọn negative theo rank TẦNG
    1.5 (_pool_neg_scores, xem _sample_negatives_for_qid/Cell 10d) rồi mở rộng short-chunk -> long-
    chunk BẰNG CÁCH GẮN PATH VÀO ĐẦU SHORT-CHUNK ("path | short-chunk", KHÔNG ghép thêm chunk lân cận,
    short-chunk trực tiếp/Cell 10c — ĐÚNG cách select_relevant_chunks dùng trong pipeline
    chính). Gọi lại hàm này mỗi epoch (random.seed khác nhau trước khi gọi, VÀ refresh_pool_scores()
    đã chạy cho epoch đó ở Cell 14) để có bộ negative "khó" khác nhau, cập nhật theo checkpoint mới
    nhất, mà không tốn thêm chi phí BM25 nào.
    Trả về list trong RAM như bản gốc — dùng cho samples nhỏ/vừa. Với samples lớn, dùng
    build_training_pairs_from_pool_streaming bên dưới để tránh đỉnh RAM."""
    pairs = []
    n_skipped = 0
    for sample in tqdm(samples, desc=desc):
        qid, question, gold = sample["qid"], sample["question"], set(sample["answers"])
        pool_entry = pools.get(qid)
        if pool_entry is None:
            continue

        for g, global_idx in pool_entry["positives"]:
            cid = short_chunk_ids[global_idx]
            short_text = short_chunk_text_of[cid]
            if not short_text or not short_text.strip():
                n_skipped += 1
                continue
            pairs.append((question, cid, short_text, 1))

        neg_pool = pool_entry["neg_pool"]
        n_pos = len(pool_entry["positives"]) or len(gold) or 1
        n_neg = neg_per_pos * n_pos
        negs = _sample_negatives_for_qid(qid, neg_pool, gold, n_neg)

        for d, global_idx in negs:
            cid = short_chunk_ids[global_idx]
            short_text = short_chunk_text_of[cid]
            if not short_text or not short_text.strip():
                n_skipped += 1
                continue
            pairs.append((question, cid, short_text, 0))

    random.shuffle(pairs)
    if n_skipped:
        print(f"Bỏ qua {n_skipped} pair do văn bản không chunk được (passage rỗng).")
    return pairs


def build_training_pairs_from_pool_streaming(pools, samples, out_path, neg_per_pos=NEG_PER_POS,
                                              batch_size=2000,
                                              desc="Resample negative từ candidate pool (TẦNG 2 - streaming)"):
    """Giống build_training_pairs_from_pool nhưng ghi ra đĩa theo batch pickle nối tiếp thay vì
    giữ toàn bộ `pairs` trong RAM. Dùng khi số lượng samples/pairs đủ lớn để bản in-RAM gây OOM.
    Negative cũng được chọn theo rank TẦNG 1.5 qua _sample_negatives_for_qid (xem docstring
    build_training_pairs_from_pool). Đọc lại bằng load_pairs_streaming()."""
    buf = []
    n_written = 0
    n_skipped = 0
    with open(out_path, "wb") as fout:
        for sample in tqdm(samples, desc=desc):
            qid, question, gold = sample["qid"], sample["question"], set(sample["answers"])
            pool_entry = pools.get(qid)
            if pool_entry is None:
                continue

            for g, global_idx in pool_entry["positives"]:
                cid = short_chunk_ids[global_idx]
                short_text = short_chunk_text_of[cid]
                if not short_text or not short_text.strip():
                    n_skipped += 1
                    continue
                buf.append((question, cid, short_text, 1))

            neg_pool = pool_entry["neg_pool"]
            n_pos = len(pool_entry["positives"]) or len(gold) or 1
            n_neg = neg_per_pos * n_pos
            negs = _sample_negatives_for_qid(qid, neg_pool, gold, n_neg)

            for d, global_idx in negs:
                cid = short_chunk_ids[global_idx]
                short_text = short_chunk_text_of[cid]
                if not short_text or not short_text.strip():
                    n_skipped += 1
                    continue
                buf.append((question, cid, short_text, 0))

            if len(buf) >= batch_size:
                pickle.dump(buf, fout)
                n_written += len(buf)
                buf = []
                gc.collect()  # ép giải phóng ngay, tránh dồn ứ RAM trước lần dump kế tiếp

        if buf:
            pickle.dump(buf, fout)
            n_written += len(buf)

    if n_skipped:
        print(f"Bỏ qua {n_skipped} pair do văn bản không chunk được (passage rỗng).")
    print(f"Đã ghi {n_written} pairs ra {out_path} (nhiều pickle batch nối tiếp, batch_size={batch_size}).")
    return n_written


def load_pairs_streaming(path):
    """Đọc lại toàn bộ batch đã ghi bởi build_training_pairs_from_pool_streaming, gộp thành 1 list
    và shuffle. Chỉ dùng khi RAM đủ chỗ chứa toàn bộ pairs cùng lúc — nếu không, train trực tiếp
    theo batch từ file thay vì gọi hàm này."""
    pairs = []
    with open(path, "rb") as f:
        while True:
            try:
                pairs.extend(pickle.load(f))
            except EOFError:
                break
    random.shuffle(pairs)
    return pairs


# ---- Sinh training pairs cho epoch đầu ----
# Ở epoch đầu tiên, _pool_neg_scores còn RỖNG (TẦNG 1.5 chưa được gọi lần nào) -> negative epoch 1
# tự động fallback về random.sample() trên neg_pool, giống hệt hành vi gốc. Từ epoch 2 trở đi, Cell
# 14 gọi refresh_pool_scores() trước khi rebuild_train_loader() nên negative sẽ được chọn theo rank
# TẦNG 1.5 (checkpoint vừa fine-tune ở epoch trước).
# Mặc định dùng bản in-RAM (giống code gốc). Nếu kernel từng chết ở đúng bước này (RAM lớn), đổi
# sang nhánh streaming bên dưới: build ra file rồi load_pairs_streaming(PAIRS_CACHE_PATH).
# train_pairs = build_training_pairs_from_pool(fit_candidate_pools, fit_samples, neg_per_pos=NEG_PER_POS)

# --- Nhánh streaming (bật khi cần, comment nhánh trên lại) ---
PAIRS_CACHE_PATH = "DSC2026-IntelliJQK/cache/train_pairs_7task2.pkl"
build_training_pairs_from_pool_streaming(fit_candidate_pools, fit_samples, PAIRS_CACHE_PATH,
                                          neg_per_pos=NEG_PER_POS)
train_pairs = load_pairs_streaming(PAIRS_CACHE_PATH)

print("Tổng số training pairs:", len(train_pairs))
print("Ví dụ:", train_pairs[1][0], "|", train_pairs[1][1], "| label=", train_pairs[1][3])
print("Độ dài chunk ví dụ:", len(train_pairs[1][2]), "ký tự")
if _false_neg_stats["n_checked_qid"]:
    print(f"[Xác nhận chéo false-negative] Đã kiểm tra {_false_neg_stats['n_checked_qid']} câu hỏi "
          f"có điểm TẦNG 1.5, loại tổng cộng {_false_neg_stats['n_suspected_total']} candidate "
          f"nghi ngờ false-negative (cả hybrid rank gốc lẫn điểm reranker hiện tại đều đồng thuận).")
else:
    print("[Xác nhận chéo false-negative] Chưa có điểm TẦNG 1.5 cho câu hỏi nào (bình thường ở lần "
          "build đầu tiên, trước khi refresh_pool_scores() chạy lần đầu ở Cell 14) -> chưa lọc gì.")


Resample negative từ candidate pool (TẦNG 2 - streaming):   0%|          | 0/6300 [00:00<?, ?it/s]

Resample negative từ candidate pool (TẦNG 2 - streaming): 100%|██████████| 6300/6300 [00:11<00:00, 531.66it/s]


Đã ghi 41231 pairs ra /mnt/mmlab2024nas/trantran/train_pairs_7task2.pkl (nhiều pickle batch nối tiếp, batch_size=2000).
Tổng số training pairs: 41231
Ví dụ: Ai có quyền khởi kiện vụ án? | 217329#s35 | label= 0
Độ dài chunk ví dụ: 1389 ký tự
[Xác nhận chéo false-negative] Chưa có điểm TẦNG 1.5 cho câu hỏi nào (bình thường ở lần build đầu tiên, trước khi refresh_pool_scores() chạy lần đầu ở Cell 14) -> chưa lọc gì.


## 5. Fine-tune BAAI/bge-reranker-v2-m3 (cross-encoder)

BAAI/bge-reranker-v2-m3 (kiến trúc XLM-RoBERTa) là model
**sequence-classification** (encoder, không phải causal LM như Qwen3-Reranker trước đây): cặp
(query, document) được đưa thẳng vào tokenizer dưới dạng 1 câu ghép `[query, document]`, model
trả về **1 logit hồi quy duy nhất** thể hiện độ liên quan (không còn 2 logit "yes"/"no" ở token
cuối cùng như trước). Ta fine-tune bằng **binary cross-entropy trên logit này**
(`BCEWithLogitsLoss`, label = 1 nếu document liên quan, 0 nếu không) — sigmoid của logit chính là
xác suất liên quan dùng cho `rerank_score()`/`RERANK_THRESHOLD`.


In [ ]:
# Cell 11 — Load BAAI/bge-reranker-v2-m3, bọc LoRA (train mới) HOẶC load checkpoint đã fine-tune (inference)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

RERANKER_NAME = "BAAI/bge-reranker-v2-m3"
# ĐÃ SỬA: BAAI/bge-reranker-v2-m3 là model SEQUENCE-CLASSIFICATION (kiến
# trúc XLM-RoBERTa/encoder) — KHÁC HẲN Qwen3-Reranker (causal LM chấm điểm qua token "yes"/"no").
# MAX_LEN theo đúng max sequence length huấn luyện chính thức của model: 2304 token (256 cho query +
# 2048 cho passage) — dư dả cho 1 short-chunk (tối đa SHORT_CHUNK_CHARS ký tự
# Việt) cộng câu hỏi.
MAX_LEN = 2304

# ---- Cấu hình LoRA (chỉ áp dụng khi LOAD_PRETRAINED=False, tức train mới) ----
USE_LORA = True          # đặt False nếu muốn quay lại full fine-tune
LORA_R = 16               # rank adapter — 8/16/32 phổ biến, cao hơn = học được nhiều hơn nhưng tốn VRAM hơn
LORA_ALPHA = 32           # thường chọn = 2 * LORA_R
LORA_DROPOUT = 0.05
# ĐÃ SỬA: BAAI/bge-reranker-v2-m3 dùng kiến trúc XLM-RoBERTa (encoder) chứ không phải Qwen/Llama-style
# (decoder) như Qwen3-Reranker -> tên module attention khác hẳn, không còn q_proj/k_proj/v_proj/
# gate_proj/up_proj/down_proj. RobertaSelfAttention đặt tên "query"/"key"/"value" cho 3 phép chiếu
# attention — đây là các module LoRA cần nhắm tới.
LORA_TARGET_MODULES = ["query", "key", "value"]

# ---- Đường dẫn checkpoint đã lưu ở Cell 15 (dùng khi LOAD_PRETRAINED=True) ----
BEST_DIR         = "DSC2026-IntelliJQK/weights/vietnamese-v2-reranker-bge-77-lora-best"
FT_RERANKER_DIR  = "DSC2026-IntelliJQK/weights/vietnamese-v2-reranker-bge-77-lora-ft"
MERGED_DIR       = "DSC2026-IntelliJQK/weights/vietnamese-v2-reranker-bge-77-merged"

# ============================== BẬT/TẮT Ở ĐÂY ==============================
# True  -> KHÔNG train, chỉ load model đã fine-tune sẵn (ưu tiên MERGED_DIR, rồi tới
#          FT_RERANKER_DIR, rồi tới BEST_DIR) để chạy inference/submission.
# False -> khởi tạo LoRA MỚI từ base model để train ở Cell 13/14, Cell 15 sẽ GHI ĐÈ lên các
#          thư mục checkpoint ở trên.
LOAD_PRETRAINED = True
# =============================================================================

reranker_tokenizer = AutoTokenizer.from_pretrained(RERANKER_NAME)
reranker_model = AutoModelForSequenceClassification.from_pretrained(
    RERANKER_NAME, torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    **_attn_impl_kwargs(),
).to(DEVICE)

if LOAD_PRETRAINED:
    # ---- Chỉ inference: load lại checkpoint đã fine-tune, KHÔNG khởi tạo LoRA mới ----
    if os.path.isdir(MERGED_DIR):
        del reranker_model  # bỏ bản base vừa load ở trên, dùng thẳng bản đã merge cho nhẹ VRAM
        torch.cuda.empty_cache() if DEVICE == "cuda" else None
        reranker_model = AutoModelForSequenceClassification.from_pretrained(
            MERGED_DIR, torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
            **_attn_impl_kwargs(),
        ).to(DEVICE)
        print(f"[LOAD_PRETRAINED=True] Đã load MERGED model (không cần peft) từ {MERGED_DIR}")
    elif os.path.isdir(FT_RERANKER_DIR):
        reranker_model = PeftModel.from_pretrained(reranker_model, FT_RERANKER_DIR).to(DEVICE)
        print(f"[LOAD_PRETRAINED=True] Đã load LoRA adapter từ {FT_RERANKER_DIR}")
    elif os.path.isdir(BEST_DIR):
        reranker_model = PeftModel.from_pretrained(reranker_model, BEST_DIR).to(DEVICE)
        print(f"[LOAD_PRETRAINED=True] Đã load LoRA adapter (best checkpoint) từ {BEST_DIR}")
    else:
        raise FileNotFoundError(
            "LOAD_PRETRAINED=True nhưng không tìm thấy checkpoint nào trong "
            f"{MERGED_DIR} / {FT_RERANKER_DIR} / {BEST_DIR}. "
            "Đặt LOAD_PRETRAINED=False nếu muốn train từ đầu."
        )
    reranker_model.eval()
else:
    # ---- Train mới: bọc LoRA lên base model, Cell 13/14/15 sẽ train và GHI ĐÈ checkpoint cũ ----
    if USE_LORA:
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,   # ĐÃ SỬA: sequence-classification, không phải CAUSAL_LM
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            target_modules=LORA_TARGET_MODULES,
            bias="none",
        )
        reranker_model = get_peft_model(reranker_model, lora_config)
        reranker_model.print_trainable_parameters()
    print("[LOAD_PRETRAINED=False] Đã khởi tạo LoRA mới từ base model, sẵn sàng train ở Cell 13/14.")

# ĐÃ SỬA: BAAI/bge-reranker-v2-m3 chấm điểm qua 1 logit hồi quy duy nhất (không còn prompt "yes"/"no"
# kiểu Qwen3-Reranker) -> bỏ toàn bộ PREFIX/SUFFIX/TOKEN_TRUE_ID/TOKEN_FALSE_ID/TASK_INSTRUCTION/
# format_instruction. Input giờ là 1 CẶP [query, document] đưa thẳng vào tokenizer (tokenizer hỗ trợ
# sẵn kiểu input list[[text_a, text_b], ...]), đúng cách dùng chính thức của
# BAAI/bge-reranker-v2-m3.

def process_inputs(pairs, max_length=MAX_LEN):
    """pairs: list[[query, document], ...]. Trả về dict tensor đã chuyển sang DEVICE."""
    inputs = reranker_tokenizer(
        pairs, padding=True, truncation=True, max_length=max_length, return_tensors="pt",
    )
    return {k: v.to(DEVICE) for k, v in inputs.items()}

@torch.no_grad()
def rerank_score(query, docs, batch_size=8):
    # Trả về list điểm liên quan (sigmoid của logit hồi quy) cho từng doc/chunk, KHÔNG tính gradient
    # (dùng khi infer). sigmoid() đưa điểm về [0, 1], giữ tương thích với RERANK_THRESHOLD hiện có
    # (Cell 2) — vốn được thiết kế cho điểm dạng xác suất.
    scores = []
    reranker_model.eval()
    for i in range(0, len(docs), batch_size):
        batch_docs = docs[i:i + batch_size]
        pairs = [[query, d] for d in batch_docs]
        inputs = process_inputs(pairs)
        logits = reranker_model(**inputs).logits.view(-1)
        probs = torch.sigmoid(logits)
        scores.extend(probs.float().cpu().tolist())
    return scores


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 393/393 [00:01<00:00, 376.04it/s]


[LOAD_PRETRAINED=True] Đã load MERGED model (không cần peft) từ /mnt/mmlab2024nas/trantran/vietnamese-v2-reranker-bge-7-merged


In [24]:
# Cell 12 — Dataset / DataLoader cho fine-tuning
from torch.utils.data import Dataset, DataLoader

class RerankDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs  # list[(question, chunk_id, chunk_text, label)]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        question, chunk_id, chunk_text, label = self.pairs[idx]
        # chunk_text là SHORT-chunk gốc; không expand/ghép thành long-chunk.
        passage = chunk_text
        # ĐÃ SỬA: BAAI/bge-reranker-v2-m3 (sequence-classification) nhận input là 1 CẶP [query, passage]
        # đưa thẳng vào tokenizer, không còn ghép thành 1 chuỗi instruct kiểu Qwen3-Reranker nữa.
        return question, passage, label

def collate_fn(batch):
    pairs = [[q, p] for q, p, _ in batch]
    labels = torch.tensor([b[2] for b in batch], dtype=torch.long)
    inputs = process_inputs(pairs)
    return inputs, labels

BATCH_SIZE = 32  # Stage-2 training batch size; giữ effective batch = 16 * 4 = 64.
train_loader = DataLoader(
    RerankDataset(train_pairs), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)
print("Số batch mỗi epoch:", len(train_loader))


Số batch mỗi epoch: 1289


In [25]:
# Cell 13 — Training loop (LoRA fine-tuning, gradient checkpointing để tiết kiệm thêm VRAM)
# + Resample negative mỗi epoch (tránh học tủ 1 bộ negative cố định)
# + ĐÃ THÊM: mỗi epoch (từ epoch 2) refresh known-positive score rồi tính lại
#   FALSE_NEG_RERANK_THRESHOLD ĐỘNG (percentile p10, làm mượt EMA — xem Cell 10c3), thay vì giữ
#   nguyên 1 ngưỡng cố định 0.9 suốt quá trình train.
# + Eval trên dev sau mỗi epoch, chọn checkpoint theo recall trước, precision sau (không dùng F2)
#   (two_stage_retrieve dùng RERANK_THRESHOLD mặc định 0.99 trong lúc train — sẽ tune lại kỹ hơn
#   ở Cell 19b SAU KHI train xong, không ảnh hưởng tới việc chọn checkpoint ở bước này)
# + Flush gradient còn sót lại cuối mỗi epoch (tránh mất tối đa GRAD_ACCUM_STEPS-1 batch cuối epoch)
#
# LƯU Ý khi dùng LoRA:
#   - Chỉ ~0.5-1% tham số được train -> LR cần cao hơn nhiều so với full fine-tune (1e-5 -> 2e-4).
#   - reranker_model.parameters() vẫn truyền cả model (base bị freeze), optimizer sẽ tự bỏ qua
#     các tham số có requires_grad=False vì chúng không có .grad.
import bitsandbytes as bnb

EPOCHS = 3
LR = 2e-4 if USE_LORA else 1e-5   # Giữ LR vì effective batch vẫn = 64 sau khi đổi BATCH_SIZE 8->16 và GRAD_ACCUM 8->4
GRAD_ACCUM_STEPS = 2  # batch hiệu dụng = BATCH_SIZE * GRAD_ACCUM_STEPS = 64

def apply_rerank_filter(ranked, top_k_final=TOP_K_FINAL,
                         rerank_top_n=None, rerank_threshold=None, rerank_fallback_k=None):
    """Lọc kết quả sau rerank theo mục 2.2.2: xét top-N (mặc định 10) document điểm reranker cao
    nhất, chỉ giữ candidate có điểm > threshold (mặc định 0.99). Nếu không candidate nào vượt
    ngưỡng, fallback lấy top-K theo điểm (mặc định 2) để tránh trả về tập rỗng. Luôn cắt thêm
    top_k_final ở cuối để không bao giờ vượt quá ràng buộc cứng của đề bài (tối đa 5).
    ranked: list[(doc_id, score)] đã sort giảm dần theo điểm reranker.
    Đọc rerank_top_n/threshold/fallback_k từ config global (RERANK_*) nếu không truyền vào, để
    Cell 19b (tune threshold) có thể chỉnh lại giá trị mặc định mà không cần định nghĩa lại hàm.

    TẠM THỜI TẮT lọc theo threshold/fallback (xem comment bên dưới): ưu tiên đơn giản hoá, chỉ cắt
    tối đa top_k_final document_id từ top-N ranked theo điểm reranker, không loại candidate nào
    theo ngưỡng nữa. Muốn bật lại threshold thì uncomment khối code phía dưới."""
    if rerank_top_n is None:
        rerank_top_n = RERANK_TOP_N
    if rerank_threshold is None:
        rerank_threshold = RERANK_THRESHOLD
    if rerank_fallback_k is None:
        rerank_fallback_k = RERANK_FALLBACK_K

    top_n_ranked = ranked[:rerank_top_n]

    # ---- TẠM THỜI COMMENT: lọc theo threshold (mục 2.2.2) ----
    # filtered = [d for d, sc in top_n_ranked if sc > rerank_threshold]
    # if not filtered:
    #     filtered = [d for d, _ in top_n_ranked[:rerank_fallback_k]]

    # ---- Ưu tiên hiện tại: bỏ qua threshold/fallback, chỉ cắt tối đa top_k_final ----
    filtered = [d for d, _ in top_n_ranked]
    return filtered[:top_k_final]


# Cache Stage-1 (retrieve + chọn short-chunk) cho eval lặp lại trên CÙNG dev_samples qua
# nhiều epoch: phần này KHÔNG phụ thuộc trọng số reranker đang train (chỉ phụ thuộc BM25 index +
# câu hỏi), nên kết quả giống hệt nhau ở mọi epoch -> tính 1 lần ở epoch đầu tiên, từ epoch 2 trở đi
# chỉ còn phải chạy lại rerank_score() (forward pass GPU, phần DUY NHẤT thực sự cần recompute vì
# dùng trọng số model MỚI). Giúp eval cuối mỗi epoch nhanh hơn hẳn (bỏ được phần BM25 + chọn chunk
# lặp lại vô ích).
_eval_stage1_cache = {}  # (question, stage1_name, top_k_stage1, chunks_per_doc, lambda) -> (flat_chunk_docs, flat_short_texts)

def get_reranked_candidates(question, stage1_fn=hybrid_retrieve, top_k_stage1=TOP_K_STAGE1,
                             use_stage1_cache=False, chunks_per_doc=None, hybrid_lam=None):
    """Stage-1 (stage1_fn) -> Stage-2 rerank (BAAI/bge-reranker-v2-m3), trả về list[(doc_id, score)] đã sort
    giảm dần theo điểm reranker, CHƯA áp threshold/fallback/top_k_final (đó là việc của
    apply_rerank_filter). Tách riêng phần này ra khỏi two_stage_retrieve() để có thể CACHE kết quả
    rerank — bước tốn GPU nhất trong pipeline — và tái sử dụng khi chỉ cần thử nhiều threshold/top_n/
    fallback_k khác nhau (xem Cell 19b: tune threshold) mà không phải chạy lại forward pass reranker
    nhiều lần cho cùng 1 câu hỏi.

    ĐÃ SỬA: representative short-chunk được chọn bằng CÙNG scoring với Stage-1. Với pipeline
    mặc định (hybrid_retrieve), score là hybrid = lambda * BM25_norm + (1-lambda) * Dense_norm.
    Vì vậy training mining, Stage-1 và Stage-2 chunk selection dùng cùng tín hiệu hybrid.

    use_stage1_cache=True: cache (flat_chunk_docs, flat_texts) theo (question, top_k_stage1,
    chunks_per_doc) trong _eval_stage1_cache — dùng khi eval LẶP LẠI cùng stage1_fn/top_k_stage1/
    tập câu hỏi qua nhiều lần gọi (vd. sau mỗi epoch trong training loop): Stage-1 chỉ cần tính
    đúng 1 lần ở lần eval đầu tiên.

    chunks_per_doc: số SHORT-chunk tối đa lấy ra cho MỖI document ứng viên. None -> dùng
    MAX_CHUNKS_PER_DOC_FOR_RERANK. Các short-chunk được đưa THẲNG vào reranker, không expand.
    Giảm số này khi eval để bớt số chunk phải rerank mỗi document."""
    if chunks_per_doc is None:
        chunks_per_doc = MAX_CHUNKS_PER_DOC_FOR_RERANK
    stage1_name = getattr(stage1_fn, "__name__", str(stage1_fn))
    if stage1_fn is hybrid_retrieve:
        if hybrid_lam is None:
            hybrid_lam = LAMBDA_HYBRID
        cache_key = (question, stage1_name, top_k_stage1, chunks_per_doc, float(hybrid_lam))
    else:
        cache_key = (question, stage1_name, top_k_stage1, chunks_per_doc, None)
    if use_stage1_cache and cache_key in _eval_stage1_cache:
        flat_chunk_docs, flat_texts = _eval_stage1_cache[cache_key]
    else:
        candidates = [d for d, _ in stage1_fn(question, top_k=top_k_stage1)]
        if len(candidates) == 0:
            if use_stage1_cache:
                _eval_stage1_cache[cache_key] = ([], [])
            return []

        # Chọn SHORT-chunk bằng cùng scoring với Stage-1.
        if stage1_fn is hybrid_retrieve:
            chunk_scores = hybrid_scores_all(question, lam=hybrid_lam)
        elif stage1_fn is dense_retrieve:
            chunk_scores = dense_scores_all(question)
        else:
            chunk_scores = bm25_scores_all(question)

        flat_chunk_docs = []   # doc_id tương ứng với từng short-chunk
        flat_texts = []       # short-chunk text đưa THẲNG vào reranker
        for d in candidates:
            for cid, ctext in select_relevant_chunks(d, chunk_scores, top_n=chunks_per_doc):
                flat_chunk_docs.append(d)
                flat_texts.append(ctext)

        if use_stage1_cache:
            _eval_stage1_cache[cache_key] = (flat_chunk_docs, flat_texts)

    if not flat_texts:
        return []

    scores = rerank_score(question, flat_texts)

    doc_best_score = {}
    for d, sc in zip(flat_chunk_docs, scores):
        if d not in doc_best_score or sc > doc_best_score[d]:
            doc_best_score[d] = sc

    return sorted(doc_best_score.items(), key=lambda x: x[1], reverse=True)


def two_stage_retrieve(question, stage1_fn=hybrid_retrieve, top_k_stage1=TOP_K_STAGE1, top_k_final=TOP_K_FINAL,
                        rerank_top_n=None, rerank_threshold=None, rerank_fallback_k=None,
                        use_stage1_cache=False, chunks_per_doc=None, hybrid_lam=None):
    """Stage-1: hybrid_retrieve (hoặc bm25_retrieve/dense_retrieve) -> top_k_stage1 candidate doc_id.
    Stage-2: với mỗi candidate document, chọn tối đa chunks_per_doc SHORT-chunk liên quan nhất
    theo cùng scoring với Stage-1 — mặc định MAX_CHUNKS_PER_DOC_FOR_RERANK short-chunk/document.
    KHÔNG expand, KHÔNG ghép chunk lân cận; rerank TẤT CẢ các short-chunk đó cùng lúc (1 batch lớn), lấy điểm CAO NHẤT trong các chunk của mỗi văn
    bản làm điểm đại diện cho văn bản đó, rồi ÁP DỤNG BỘ LỌC THRESHOLD (mục 2.2.2): chỉ giữ lại các
    candidate "tự tin cao" trong top-10 (điểm > 0.99), fallback top-2 nếu không có candidate nào đạt
    ngưỡng, để vừa tăng precision vừa đảm bảo không trả về tập rỗng.
    (Wrapper mỏng quanh get_reranked_candidates() + apply_rerank_filter(), giữ nguyên hành vi/API
    như trước — các cell downstream (eval, ablation, submission) không cần sửa gì.)

    use_stage1_cache=True: truyền thẳng xuống get_reranked_candidates() để cache phần Stage-1 (BM25
    + chọn chunk) qua nhiều lần gọi liên tiếp trên cùng tập câu hỏi (xem eval cuối mỗi epoch bên dưới).
    chunks_per_doc: xem docstring get_reranked_candidates — None giữ nguyên hành vi cũ."""
    ranked = get_reranked_candidates(question, stage1_fn=stage1_fn, top_k_stage1=top_k_stage1,
                                      use_stage1_cache=use_stage1_cache, chunks_per_doc=chunks_per_doc,
                                      hybrid_lam=hybrid_lam)
    return apply_rerank_filter(
        ranked, top_k_final=top_k_final,
        rerank_top_n=rerank_top_n, rerank_threshold=rerank_threshold, rerank_fallback_k=rerank_fallback_k,
    )

# ============================================================================================
# Stage-2 dùng chung đúng một logic cho training/eval/inference: chọn short-chunk và rerank trực tiếp.
# Không còn bất kỳ nhánh EVAL-ONLY hay logic long-chunk nào.
# ============================================================================================


def rebuild_train_loader(epoch_seed):
    # ĐÃ SỬA: dùng lại fit_candidate_pools đã mine/cache sẵn ở Cell 10c (TẦNG 1 - HYBRID, ĐẮT, chỉ
    # chạy 1 lần) -> mỗi epoch chỉ còn gọi TẦNG 1.5 (refresh_pool_scores, RẺ — xem lời gọi ngay
    # TRƯỚC rebuild_train_loader() trong vòng lặp epoch bên dưới) rồi TẦNG 2
    # (build_training_pairs_from_pool, chọn negative theo rank TẦNG 1.5, xem Cell 10d), KHÔNG tính
    # lại bm25.get_scores()/dense cho fit_samples nữa. Nhờ vậy dù restart kernel, chỉ cần
    # mine_candidate_pools() load lại cache TẦNG 1 một lần là đủ cho toàn bộ quá trình train — TẦNG
    # 1.5 sẽ tự "làm nóng lại" difficulty của candidate theo checkpoint mới nhất mỗi epoch.
    random.seed(SEED + epoch_seed)
    pairs = build_training_pairs_from_pool(fit_candidate_pools, fit_samples, neg_per_pos=NEG_PER_POS,
                                            desc=f"Resample negative (epoch seed={epoch_seed})")
    loader = DataLoader(RerankDataset(pairs), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    return loader

# # --- Bật gradient checkpointing để giảm activation memory (vẫn hữu ích dù dùng LoRA) ---
# reranker_model.gradient_checkpointing_enable()
# reranker_model.enable_input_require_grads()  # cần thiết khi dùng checkpointing (kể cả với LoRA)
# # ĐÃ SỬA: bỏ dòng reranker_model.config.use_cache = False — chỉ áp dụng cho causal LM (Qwen3-Reranker
# # cũ dùng KV-cache khi generate); BAAI/bge-reranker-v2-m3 là encoder (sequence-classification), không có
# # khái niệm use_cache nên không cần dòng này nữa.

# # --- Chỉ train các tham số LoRA (requires_grad=True), base model đã bị freeze bởi get_peft_model ---
# trainable_params = [p for p in reranker_model.parameters() if p.requires_grad]
# n_trainable = sum(p.numel() for p in trainable_params)
# n_total = sum(p.numel() for p in reranker_model.parameters())
# print(f"Training {n_trainable:,} / {n_total:,} params ({100*n_trainable/n_total:.2f}%)")

# optimizer = bnb.optim.AdamW8bit(trainable_params, lr=LR)

# steps_per_epoch = len(train_loader) // GRAD_ACCUM_STEPS
# num_training_steps = steps_per_epoch * EPOCHS
# scheduler = torch.optim.lr_scheduler.LinearLR(
#     optimizer, start_factor=1.0, end_factor=0.1, total_iters=max(1, num_training_steps)
# )

# best_recall = -1.0
# best_precision = -1.0
# best_epoch = -1     # epoch (1-indexed) ứng với checkpoint đã lưu ở BEST_DIR — dùng để in/kiểm tra
#                     # lại sau này, tránh mù mờ không rõ BEST_DIR đang tương ứng epoch nào.
# global_step = 0

# for epoch in range(EPOCHS):
#     if epoch == 0:
#         # Epoch đầu tiên: dùng train_loader đã build sẵn ở Cell 13 (negative epoch 1 fallback về
#         # random.sample() vì TẦNG 1.5 chưa refresh lần nào — xem Cell 10d).
#         epoch_loader = train_loader
#     else:
#         # TẦNG 1.5 (MỚI): rerank lại toàn bộ neg_pool bằng checkpoint VỪA fine-tune xong ở epoch
#         # trước — RẺ (không quét lại corpus/BM25/dense), rồi mới TẦNG 2 resample negative theo rank
#         # mới đó (hard-negative "động", bám theo model đang train thay vì rank BM25/dense tĩnh ban đầu).
#         torch.cuda.empty_cache()
#         refresh_pool_scores(fit_candidate_pools, samples=fit_samples,
#                              desc=f"Refresh pool scores TẦNG 1.5 (trước epoch {epoch + 1})")
#         # ĐÃ THÊM: refresh điểm reranker hiện tại trên known-positive (Cell 10c3) rồi tính lại
#         # FALSE_NEG_RERANK_THRESHOLD ĐỘNG theo percentile thấp (p10) của các điểm đó — ngưỡng
#         # tự khớp theo độ tự tin thực tế của checkpoint ở epoch này, thay vì cố định 0.9
#         # (Cell 2) suốt quá trình train. Rẻ (chỉ 1-3 forward pass/câu hỏi), an toàn để gọi mỗi
#         # epoch ngay sau refresh_pool_scores(). None (vd. chưa đủ known-positive) -> tự động
#         # GIỮ NGUYÊN threshold hiện tại, không ghi đè.
#         refresh_pool_pos_scores(fit_candidate_pools, samples=fit_samples,
#                                  desc=f"Refresh positive scores TẦNG 1.5 (trước epoch {epoch + 1})")
#         dynamic_thr = compute_dynamic_false_neg_threshold(low_percentile=10)
#         if dynamic_thr is not None:
#             FALSE_NEG_RERANK_THRESHOLD = dynamic_thr
#             print(f"[Epoch {epoch + 1}] Dynamic FALSE_NEG_RERANK_THRESHOLD (EMA) = "
#                   f"{FALSE_NEG_RERANK_THRESHOLD:.4f} (p10 của "
#                   f"{sum(len(v) for v in _pool_pos_scores.values())} known-positive scores)")
#         epoch_loader = rebuild_train_loader(epoch_seed=epoch)

#     reranker_model.train()
#     running_loss = 0.0
#     optimizer.zero_grad()
#     pbar = tqdm(epoch_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
#     for step, (inputs, labels) in enumerate(pbar):
#         labels = labels.to(DEVICE)

#         # ĐÃ SỬA: BAAI/bge-reranker-v2-m3 (sequence-classification) trả về 1 logit hồi quy duy nhất
#         # (không còn 2 logit "yes"/"no" như Qwen3-Reranker) -> dùng binary cross-entropy trực tiếp
#         # trên logit này thay vì cross-entropy 2 lớp.
#         with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
#             logits = reranker_model(**inputs).logits.view(-1)
#             loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels.float())

#         (loss / GRAD_ACCUM_STEPS).backward()

#         if (step + 1) % GRAD_ACCUM_STEPS == 0:
#             torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
#             optimizer.step()
#             scheduler.step()
#             optimizer.zero_grad()
#             global_step += 1

#         running_loss += loss.item()
#         if step % 20 == 0:
#             pbar.set_postfix(loss=running_loss / (step + 1))

#     if (step + 1) % GRAD_ACCUM_STEPS != 0:
#         torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
#         optimizer.step()
#         scheduler.step()
#         optimizer.zero_grad()
#         global_step += 1

#     reranker_model.eval()
#     # Eval chỉ giảm số SHORT-chunk/document nếu muốn; inference thật mặc định vẫn dùng 3 short-chunk/document.
#     # Short-chunk được đưa thẳng vào reranker, không expand.
#     EVAL_CHUNKS_PER_DOC = 1
#     epoch_metrics = evaluate(
#         lambda q: two_stage_retrieve(q, stage1_fn=hybrid_retrieve, top_k_stage1=TOP_K_STAGE1, top_k_final=TOP_K_FINAL,
#                                       use_stage1_cache=True, chunks_per_doc=EVAL_CHUNKS_PER_DOC),
#         dev_samples, top_k=TOP_K_FINAL, desc=f"eval dev (epoch {epoch+1})"
#     )
#     epoch_recall = epoch_metrics["recall"]
#     epoch_precision = epoch_metrics["precision"]
#     print(f"Epoch {epoch+1}: train_loss(avg)={running_loss/len(epoch_loader):.4f} | "
#           f"dev recall={epoch_recall:.4f} precision={epoch_precision:.4f}")

#     is_better = (epoch_recall, epoch_precision) > (best_recall, best_precision)
#     if is_better:
#         best_recall = epoch_recall
#         best_precision = epoch_precision
#         best_epoch = epoch + 1
#         # Với LoRA, save_pretrained chỉ lưu adapter (vài chục MB), không lưu lại base model
#         reranker_model.save_pretrained(BEST_DIR)
#         reranker_tokenizer.save_pretrained(BEST_DIR)
#         print(f"  -> recall cải thiện (hoặc recall bằng nhưng precision tốt hơn), "
#               f"đã lưu adapter tốt nhất tại {BEST_DIR} (epoch={best_epoch}, recall={best_recall:.4f}, "
#               f"precision={best_precision:.4f})")

# print(f"Fine-tuning xong. Best dev recall = {best_recall} (epoch {best_epoch}) | "
#       f"precision tương ứng = {best_precision} | adapter tốt nhất tại: {BEST_DIR}")
# print(f"LƯU Ý: biến `reranker_model` trong bộ nhớ hiện đang là trọng số EPOCH CUỐI ({EPOCHS}), "
#       f"KHÔNG PHẢI epoch tốt nhất ({best_epoch}) — Cell 15 ngay sau đây sẽ tự nạp lại đúng "
#       f"checkpoint tốt nhất từ {BEST_DIR} trước khi lưu/merge, không cần bạn làm gì thêm ở đây.")


In [26]:
# # Cell 14 — Nạp lại đúng checkpoint TỐT NHẤT (BEST_DIR) rồi lưu/merge, đảm bảo FT_RERANKER_DIR
# # và MERGED_DIR luôn khớp với epoch có dev recall cao nhất, KHÔNG PHẢI epoch cuối cùng của vòng lặp.
# #
# # LÝ DO BẮT BUỘC PHẢI NẠP LẠI: sau khi Cell 14 chạy xong toàn bộ EPOCHS, biến `reranker_model` trong
# # bộ nhớ đang mang trọng số của EPOCH CUỐI CÙNG — vì trong training loop, việc "epoch nào tốt hơn"
# # (biến is_better) chỉ quyết định có GHI FILE ra BEST_DIR hay không, chứ không hề đổi lại trọng số
# # đang có trong biến `reranker_model`. Nếu bỏ qua bước nạp lại này, cell save/merge bên dưới sẽ lưu
# # nhầm epoch cuối (có thể tệ hơn best) vào FT_RERANKER_DIR/MERGED_DIR — đây chính là lỗi gặp phải
# # trước đó khi Cell 18 cho ra recall thấp hơn hẳn "Best dev recall" đã in ra ở Cell 14.
# from peft import PeftModel
# import gc

# print(f"Đang nạp lại checkpoint TỐT NHẤT từ {BEST_DIR} (epoch={best_epoch}, recall={best_recall:.4f}, "
#       f"precision={best_precision:.4f}) vào reranker_model, thay cho trọng số epoch cuối ({EPOCHS}) "
#       f"đang có trong bộ nhớ...")

# # ---- Dọn dẹp triệt để trước khi load lại, tránh VRAM còn sót do optimizer/trainable_params (Cell 14)
# # vẫn giữ tham chiếu tới các tensor LoRA adapter cũ (nhỏ, nhưng dọn cho chắc, đặc biệt nếu VRAM sát
# # giới hạn). `globals().get(...)` để không lỗi NameError nếu biến chưa từng tồn tại (vd. chạy lại
# # Cell 15 nhiều lần trong cùng session). ----
# for _var in ["reranker_model", "optimizer", "trainable_params", "scheduler"]:
#     if _var in globals():
#         del globals()[_var]
# gc.collect()
# torch.cuda.empty_cache() if DEVICE == "cuda" else None

# reranker_model = AutoModelForSequenceClassification.from_pretrained(
#     RERANKER_NAME, torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32
# ).to(DEVICE)

# if USE_LORA:
#     # Trường hợp thường gặp (mặc định notebook): BEST_DIR chỉ chứa adapter LoRA -> gắn vào base model.
#     reranker_model = PeftModel.from_pretrained(reranker_model, BEST_DIR).to(DEVICE)
# else:
#     # Trường hợp full fine-tune (USE_LORA=False): BEST_DIR chứa TOÀN BỘ trọng số model (không phải
#     # adapter) -> load thẳng, không qua PeftModel (PeftModel.from_pretrained sẽ lỗi vì BEST_DIR
#     # không có adapter_config.json trong trường hợp này).
#     del reranker_model
#     torch.cuda.empty_cache() if DEVICE == "cuda" else None
#     reranker_model = AutoModelForSequenceClassification.from_pretrained(
#         BEST_DIR, torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32
#     ).to(DEVICE)

# reranker_model.eval()
# print(f"Đã nạp xong checkpoint tốt nhất (epoch {best_epoch}) — từ đây trở đi, mọi cell dùng "
#       f"`reranker_model`/`rerank_score()` (Cell 18 eval, Cell 20/21 submission, ...) đều dùng ĐÚNG "
#       f"checkpoint tốt nhất, không còn bị lệch sang epoch cuối cùng nữa.")

# # ---- Lưu lại adapter LoRA (giờ đã là adapter TỐT NHẤT, không phải epoch cuối) ----
# reranker_model.save_pretrained(FT_RERANKER_DIR)   # chỉ lưu adapter LoRA (vài chục MB) nếu USE_LORA=True
# reranker_tokenizer.save_pretrained(FT_RERANKER_DIR)
# print("Đã lưu adapter/model fine-tune (best checkpoint) tại:", FT_RERANKER_DIR)

# # ---- Tuỳ chọn: merge LoRA vào base model để có 1 model độc lập, inference nhanh như bản gốc ----
# MERGE_LORA = True
# if USE_LORA and MERGE_LORA:
#     merged_model = reranker_model.merge_and_unload()
#     merged_model.save_pretrained(MERGED_DIR)
#     reranker_tokenizer.save_pretrained(MERGED_DIR)
#     print("Đã merge LoRA vào base model và lưu tại:", MERGED_DIR)
#     print("-> Dùng AutoModelForSequenceClassification.from_pretrained(MERGED_DIR) để load lại, không cần cài peft.")

## 6. Pipeline 2 tầng: Stage-1 (candidates) → Stage-2 (rerank + lọc) → top-5

Stage-1 mặc định dùng **hybrid_retrieve** (BM25 + Vietnamese_Embedding_v2 (AITeamVN), mục 2.2.1) để lấy
top-k ứng viên. Stage-2 dùng **BAAI/bge-reranker-v2-m3** (đã fine-tune LoRA, không đổi) để rerank,
trả về điểm liên quan CE_θ(q, a) ∈ [0, 1] cho từng candidate.

**Lọc theo threshold (mục 2.2.2).** Sau khi rerank, top-k thô của BM25/dense có thể vẫn còn
candidate không liên quan. Để tăng precision, ta áp dụng lọc: chỉ xét **top-10** candidate
điểm reranker cao nhất, giữ lại những candidate có điểm **> 0.99**. Nếu không candidate nào
vượt ngưỡng này, fallback lấy **top-2** theo điểm reranker để tránh trả về tập rỗng. Ngưỡng
0.99, top-10 và fallback top-2 được **tune trên tập validation (dev)** — xem Cell 19b. Kết quả
sau lọc luôn được cắt thêm ở `top_k_final` để đảm bảo không bao giờ vượt quá 5 document_id
(ràng buộc cứng của đề bài). Chiến lược này giúp tăng precision đáng kể chỉ với chi phí giảm
nhẹ recall, cải thiện F2 tổng thể.


In [27]:
# # Cell 18 — Đánh giá pipeline 2 tầng (Hybrid stage-1 + fine-tuned reranker) trên tập dev (offline)
# two_stage_metrics = evaluate(
#     lambda q: two_stage_retrieve(q, stage1_fn=hybrid_retrieve, top_k_stage1=TOP_K_STAGE1, top_k_final=TOP_K_FINAL),
#     dev_samples, top_k=TOP_K_FINAL, desc="2-stage Hybrid(BM25+Dense) + bge-reranker-v2-m3-FT (dev)"
# )

# print("2-stage (fine-tuned):", two_stage_metrics)


In [28]:
# # Cell 18b — Tune bộ lọc Stage-2 (threshold, top-10, fallback top-K) trên tập VALIDATION (dev)
# # Cố định stage-1 = hybrid_retrieve (đã fix lambda ở Cell 8c) + reranker đã fine-tune, chỉ grid
# # search THRESHOLD (giữ rerank_top_n=10, fallback_k=2 cố định theo đúng mục 2.2.2).
# #
# # ĐÃ SỬA tiêu chí chọn threshold: KHÔNG dùng F2-score nữa. Chọn theo ĐÚNG tiêu chí chấm điểm của đề
# # bài — Recall TRƯỚC, Precision SAU (giống hệt cách chọn checkpoint reranker ở Cell 14): so sánh
# # recall trung bình giữa các threshold trước, threshold nào recall cao hơn thắng; nếu recall BẰNG
# # NHAU thì mới so sánh precision để chọn threshold có precision cao hơn. Dùng tuple (recall,
# # precision) so sánh theo thứ tự từ điển (lexicographic) để thể hiện đúng thứ tự ưu tiên này.
# #
# # TỐI ƯU: apply_rerank_filter() (Cell 14) chỉ là hậu xử lý (post-hoc) trên điểm đã rerank — không
# # phụ thuộc gì vào stage-1/GPU. Bản gốc gọi two_stage_retrieve() (chạy lại CẢ stage-1 lẫn forward
# # pass reranker trên GPU) cho MỖI threshold trong THRESHOLD_GRID, tức lặp lại bước tốn GPU nhất của
# # pipeline 8 LẦN/câu hỏi một cách dư thừa. Ở đây ta rerank 1 LẦN/câu hỏi bằng get_reranked_candidates()
# # (Cell 14), cache lại danh sách đã sort theo điểm reranker, rồi mỗi threshold chỉ cần gọi
# # apply_rerank_filter() (rẻ, chạy trên CPU, không đụng GPU/stage-1 nữa).
# #
# # LƯU Ý: hiện tại phần lọc threshold bên trong apply_rerank_filter() đang bị COMMENT (ưu tiên chỉ cắt
# # top_k_final, xem Cell 14) -> threshold_search_results bên dưới sẽ cho kết quả GIỐNG NHAU ở mọi
# # threshold cho tới khi bạn bật lại phần lọc đó. Cell này vẫn hữu ích để tune lại threshold khi cần,
# # nhưng đừng ngạc nhiên nếu các dòng in ra hiện tại trông "phẳng" (không đổi theo threshold).

# # ---- Precompute rerank 1 LẦN cho toàn bộ dev_samples (bước tốn GPU nhất), cache lại rồi mới quét
# #      threshold — xem ghi chú TỐI ƯU ở trên. ----
# _dev_ranked_candidates = [
#     get_reranked_candidates(s["question"], stage1_fn=hybrid_retrieve, top_k_stage1=TOP_K_STAGE1)
#     for s in tqdm(dev_samples, desc="Precompute reranked candidates (dev, dùng chung cho mọi threshold)")
# ]


# def pipeline_metrics_at_threshold(threshold, ranked_lists, samples=dev_samples,
#                                    top_n=RERANK_TOP_N, fallback_k=RERANK_FALLBACK_K):
#     """Tính recall/precision trung bình trên dev_samples ứng với 1 threshold, bằng cách áp
#     apply_rerank_filter() (CPU, rẻ) lên các danh sách đã rerank sẵn (ranked_lists) — không recompute
#     stage-1/rerank. KHÔNG còn tính f2 — lựa chọn threshold dựa trên (recall, precision), không dựa
#     trên f2 nữa (xem ghi chú ĐÃ SỬA ở đầu cell)."""
#     recalls, precisions = [], []
#     for s, ranked in zip(samples, ranked_lists):
#         pred = apply_rerank_filter(
#             ranked, top_k_final=TOP_K_FINAL,
#             rerank_top_n=top_n, rerank_threshold=threshold, rerank_fallback_k=fallback_k,
#         )
#         r, p = recall_precision(pred, s["answers"])
#         recalls.append(r); precisions.append(p)
#     recall, precision = float(np.mean(recalls)), float(np.mean(precisions))
#     return {"recall": recall, "precision": precision}

# THRESHOLD_GRID = [0.5, 0.7, 0.9, 0.95, 0.97, 0.99, 0.995, 0.999]
# threshold_search_results = {
#     t: pipeline_metrics_at_threshold(t, _dev_ranked_candidates) for t in tqdm(THRESHOLD_GRID, desc="Tuning threshold")
# }

# for t, m in sorted(threshold_search_results.items()):
#     print(f"  threshold={t:.3f} -> recall={m['recall']:.4f} precision={m['precision']:.4f}")

# # ĐÃ SỬA: chọn threshold theo (recall, precision) — recall trước, precision sau (không dùng F2).
# # max() trên tuple so sánh recall trước; chỉ khi recall bằng nhau mới xét tới precision.
# RERANK_THRESHOLD = max(
#     threshold_search_results,
#     key=lambda t: (threshold_search_results[t]["recall"], threshold_search_results[t]["precision"]),
# )
# print(f"\n-> Chọn threshold={RERANK_THRESHOLD:.3f} "
#       f"(recall={threshold_search_results[RERANK_THRESHOLD]['recall']:.4f}, "
#       f"precision={threshold_search_results[RERANK_THRESHOLD]['precision']:.4f}) cho bộ lọc Stage-2.")


## 7. Stage 3 — Fine-tune & sinh câu trả lời (QA) bằng `ntphuc149/ViLegalQwen3-1.7B-Base`

Stage-1 (hybrid retrieve) + Stage-2 (reranker đã fine-tune) **KHÔNG train lại** ở đây — chỉ dùng để **inference** (lấy top-5 document liên quan nhất cho một câu hỏi bất kỳ).

Luồng Stage-3 gồm 3 bước:

1. **Chuẩn bị dữ liệu huấn luyện** (`train2.json`): với MỖI câu hỏi trong `train2.json`, chạy Stage-1 + Stage-2 để lấy **top-5 document** (theo điểm reranker), mỗi document lấy đúng **1 chunk điểm reranker cao nhất**, ghép LẦN LƯỢT 5 chunk đó (đã sort giảm dần theo điểm reranker) thành context. Cặp `(question + context) -> answer` (answer lấy từ `train2.json`) là 1 sample huấn luyện.
   - `train.json` dùng để fine-tune **reranker** (Stage-2, mục 4-5).
   - `train2.json` dùng để fine-tune **ViLegalQwen3-1.7B-Base** (Stage-3, sinh câu trả lời) — answer trong `train2.json` là **câu trả lời thật** (chuỗi văn bản), khác với `train.json` (answer là list document_id).
2. **Fine-tune `ntphuc149/ViLegalQwen3-1.7B-Base`** (causal LM, kiến trúc Qwen3, LoRA) — đây là model **Base** (chưa instruction-tuned) đã được continual-pretrain từ Qwen3-1.7B-Base trên kho ngữ liệu pháp luật tiếng Việt (~16GB, bộ ViLegalLM), nên đã có sẵn nền kiến thức pháp luật tiếng Việt, nhưng vẫn cần fine-tune trên các cặp `(question, context) -> answer` này để model học khớp với văn phong/cách trả lời có sẵn trong `train2.json`. Sau mỗi epoch, model được dùng để **sinh câu trả lời thật** (không phải teacher-forcing) trên tập dev nội bộ (10% `train2.json`), rồi tính **điểm METEOR trung bình** so với answer tham chiếu — checkpoint có **METEOR cao nhất** được giữ lại làm checkpoint tốt nhất (KHÔNG còn chọn theo eval loss như trước).
3. **Inference**: dùng checkpoint ViLegalQwen3-1.7B-Base đã fine-tune (METEOR tốt nhất), chạy trên `public-official2.json` (hoặc `PRIVATE_TEST_PATH` khi vào giai đoạn private) để sinh `submission.json` cuối cùng, đúng định dạng:
```json
{
  "147194": {"answer": "Theo quy định tại Điều 37 Luật Doanh nghiệp..."},
  "147195": {"answer": "Người lao động có quyền..."}
}
```
`question_id` (khoá JSON) là mã câu hỏi (string), `answer` (string) là câu trả lời do hệ thống sinh ra — KHÔNG còn là list document_id như submission Stage-2 cũ.


In [30]:
# Cell 21 — Cấu hình + tokenizer cho ntphuc149/ViLegalQwen3-1.7B-Base (Stage-3, sinh câu trả lời từ context)
# ĐÃ SỬA: đổi từ VietAI/vit5-large (seq2seq/encoder-decoder) sang ntphuc149/ViLegalQwen3-1.7B-Base
# (causal LM, kiến trúc Qwen3, bản BASE đã continual-pretrain trên ~16GB văn bản pháp luật tiếng
# Việt (bộ ViLegalLM) nhưng CHƯA instruction-tuned -> vẫn cần fine-tune ở Cell 22d để học cách
# trả lời theo đúng định dạng của train2.json) -> dùng AutoModelForCausalLM thay vì
# AutoModelForSeq2SeqLM, và prompt được ghép trực tiếp vào input (không còn tách
# encoder-input/decoder-target như T5).
#
# ĐÃ SỬA (chống OOM): việc LOAD qa_model (nặng, ~1.7B tham số) bị TÁCH RA khỏi cell này và dời
# xuống cell ngay SAU Cell 22a (build context train2 bằng Stage-1/Stage-2 + xoá embed_model/
# reranker_model) -- xem cell "Cell 21b" phía dưới. Lý do: Cell 22a vẫn cần embed_model (Stage-1,
# dense retrieval) và reranker_model (Stage-2) chạy trên GPU để build context; nếu load thêm
# qa_model NGAY TẠI ĐÂY (tức TRƯỚC Cell 22a) thì trong lúc build context sẽ có CẢ BA model
# (embed_model + reranker_model + qa_model) cùng lúc chiếm VRAM, dễ gây OOM một cách không cần
# thiết vì qa_model lúc đó chưa hề được dùng tới. Cell này chỉ còn giữ phần NHẸ (tokenizer, config,
# hàm build_qa_prompt) -- không tốn VRAM.
from transformers import AutoTokenizer as _AutoTok, AutoModelForCausalLM

QA_MODEL_NAME = "ntphuc149/ViLegalQwen3-1.7B-Base"

# ---- Số document (đã rerank) dùng làm nguồn context; MỖI document chỉ đóng góp ĐÚNG 1 chunk
# (chunk có điểm reranker cao nhất của chính document đó) — KHÔNG lấy nhiều chunk/document. ----
TOP_K_ANSWER_CONTEXT = 5     # ràng buộc của đề bài: đúng 5 document -> đúng 5 chunk

# ĐÃ SỬA (dựa trên số liệu thực đo ở Cell 22a2, chạy trên toàn bộ train2.json, n=7000):
#   Prompt (câu hỏi + 5 chunk context): mean=2414, p50=2424, p90=2788, p95=2885, p99=3073, p100=4050
#   Answer (câu trả lời tham chiếu):    mean=457,  p50=409,  p90=754,  p95=906,  p99=1331, p100=3027
# 4096/2048 (giá trị đoán ban đầu) VƯỢT XA những gì dữ liệu thật cần -> lãng phí VRAM cho tensor
# logits (batch, seq_len, vocab~150k) ở HẦU HẾT các batch, là nguyên nhân chính gây OOM lúc backward.
# DÙNG CHUNG 1 ngân sách answer (QA_MAX_NEW_TOKENS) cho CẢ train (Cell 22c) LẪN generate (Cell 22d
# eval METEOR, Cell 23/24 inference) -- không tách riêng ngân sách train nữa (đã bỏ
# QA_TRAIN_MAX_ANSWER_TOKENS). Việc giảm VRAM lúc train giờ dựa vào kỹ thuật logits_to_keep (xem
# Cell 22d) thay vì cắt bớt answer, nên không cần 2 ngân sách khác nhau nữa.
QA_MAX_INPUT_LEN = 3072       # ngân sách token cho PHẦN PROMPT -- phủ p99 (3073) của dữ liệu thật
QA_MAX_NEW_TOKENS = 1536      # ngân sách token cho ANSWER -- phủ p99 (1331) + đệm, dùng cho CẢ
                              # train (cắt answer_ids khi tokenize) LẪN generate() lúc inference.

qa_tokenizer = _AutoTok.from_pretrained(QA_MODEL_NAME, use_fast=True, trust_remote_code=True)
if qa_tokenizer.pad_token is None:
    # Nhiều model Qwen-family không có sẵn pad_token riêng -> dùng eos_token làm pad_token
    # (chuẩn thực hành phổ biến khi fine-tune causal LM theo batch).
    qa_tokenizer.pad_token = qa_tokenizer.eos_token
qa_tokenizer.padding_side = "right"  # mặc định right-padding lúc TRAIN (xem Cell 22c); sẽ chuyển
                                      # sang "left" khi GENERATE (bắt buộc với causal LM, xem Cell 22d/22e)

print(f"Đã cấu hình tokenizer cho {QA_MODEL_NAME}. qa_model (nặng, tốn VRAM) SẼ được load ở cell "
      f"ngay sau khi build xong context train2 và đã giải phóng embed_model/reranker_model "
      f"(xem 'Cell 21b' bên dưới) -- chưa load ở đây.")


def build_qa_prompt(question: str, context: str) -> str:
    """Ghép (question, context) thành 1 prompt cho ViLegalQwen3-1.7B-Base.

    Dùng chat template của tokenizer nếu model có khai báo sẵn (một số checkpoint Qwen3 vẫn giữ
    chat_template dù là bản "-Base"); nếu không có, fallback về prompt dạng instruction thuần văn
    bản pháp luật, kết thúc bằng "Trả lời:" để model tiếp tục sinh câu trả lời ngay sau đó.
    """
    instruction = (
        "Bạn là trợ lý pháp luật. Dựa vào ngữ cảnh pháp lý dưới đây, hãy trả lời câu hỏi "
        "một cách chính xác, ngắn gọn và đúng trọng tâm.\n\n"
        f"Ngữ cảnh:\n{context}\n\nCâu hỏi: {question}\nTrả lời:"
    )
    if getattr(qa_tokenizer, "chat_template", None):
        messages = [{"role": "user", "content": instruction}]
        return qa_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    return instruction + " "


def build_qa_prompt_fit(question: str, context: str, max_len: int = None) -> str:
    """Như build_qa_prompt nhưng ĐẢM BẢO prompt <= max_len (mặc định QA_MAX_INPUT_LEN) token bằng cách
    CẮT BỚT CONTEXT (từ cuối context), KHÔNG cắt cuối prompt.

    Vì sao: prompt có dạng [chỉ dẫn][context][Câu hỏi + "Trả lời:"][header assistant của chat
    template]. Nếu để tokenizer tự cắt (truncation=True, mặc định cắt BÊN PHẢI) khi prompt quá dài thì
    phần bị mất chính là câu hỏi, "Trả lời:" và header assistant -> model không biết phải trả lời gì.
    Ở đây câu hỏi/chỉ dẫn/header luôn được giữ nguyên vẹn, chỉ phần context bị rút gọn. Context là 5
    chunk đã sort GIẢM dần theo điểm reranker, nên cắt đuôi = bỏ phần kém liên quan nhất trước.

    DÙNG HÀM NÀY Ở MỌI NƠI dựng prompt (train collate Cell 22c, dev generate Cell 22d, inference Cell 23)
    để prompt giữa train/dev/inference luôn giống hệt nhau. Prompt đã vừa ngân sách thì trả về NGUYÊN
    VĂN như build_qa_prompt (không đổi gì)."""
    if max_len is None:
        max_len = QA_MAX_INPUT_LEN

    def _n_tokens(p):
        return len(qa_tokenizer(p, add_special_tokens=False)["input_ids"])

    prompt = build_qa_prompt(question, context)
    n = _n_tokens(prompt)
    if n <= max_len:
        return prompt

    ctx_ids = qa_tokenizer(context, add_special_tokens=False)["input_ids"]
    keep = max(0, len(ctx_ids) - (n - max_len))          # cắt đúng phần token dư
    for _ in range(8):
        # decode nửa chừng 1 ký tự nhiều byte (tiếng Việt) có thể sinh "\ufffd" ở cuối -> bỏ đi.
        ctx = qa_tokenizer.decode(ctx_ids[:keep], skip_special_tokens=True).rstrip("\ufffd").rstrip()
        prompt = build_qa_prompt(question, ctx)
        n2 = _n_tokens(prompt)
        if n2 <= max_len:
            return prompt
        # ranh giới ghép chuỗi có thể làm số token lệch vài đơn vị -> cắt thêm phần còn dư rồi thử lại
        keep = max(0, keep - (n2 - max_len))
    # Không thể vừa ngân sách (vd. chính câu hỏi đã quá dài): trả về bản đã cắt hết mức; tokenizer
    # truncation ở các cell sau chỉ còn là lớp an toàn cuối cùng.
    return prompt


Đã cấu hình tokenizer cho Qwen/Qwen3-1.7B. qa_model (nặng, tốn VRAM) SẼ được load ở cell ngay sau khi build xong context train2 và đã giải phóng embed_model/reranker_model (xem 'Cell 21b' bên dưới) -- chưa load ở đây.


In [31]:
# Cell 22 — Lấy top-5 document (đã rerank) + CHỈ 1 chunk điểm cao nhất/document để làm context.
#
# Hàm này tái sử dụng ĐÚNG pipeline Stage-1 -> Stage-2 (hybrid_retrieve -> select_relevant_chunks
# -> rerank_score) giống hệt get_reranked_candidates() (Cell 13/mục 6), NHƯNG thay vì chỉ giữ lại
# điểm cao nhất của mỗi document (doc_best_score), hàm dưới đây GIỮ LẠI LUÔN chunk text tương ứng
# với điểm cao nhất đó — vì get_reranked_candidates() gốc chỉ trả về (doc_id, score), không có
# text của chunk trúng tuyển.

def get_top_chunks_for_answer(question, stage1_fn=hybrid_retrieve, top_k_stage1=TOP_K_STAGE1,
                               top_k_docs=TOP_K_ANSWER_CONTEXT, chunks_per_doc=None, hybrid_lam=None):
    """Trả về list[(doc_id, score, chunk_text)] — tối đa top_k_docs document (mặc định 5), sort
    giảm dần theo điểm reranker; MỖI document chỉ có ĐÚNG 1 chunk_text (chunk điểm reranker cao
    nhất trong số các short-chunk của chính document đó, được đưa vào rerank ở Stage-2)."""
    if chunks_per_doc is None:
        chunks_per_doc = MAX_CHUNKS_PER_DOC_FOR_RERANK

    candidates = [d for d, _ in stage1_fn(question, top_k=top_k_stage1)]
    if not candidates:
        return []

    # Chọn short-chunk đại diện bằng CÙNG scoring với Stage-1 (giống get_reranked_candidates).
    if stage1_fn is hybrid_retrieve:
        chunk_scores = hybrid_scores_all(question, lam=hybrid_lam)
    elif stage1_fn is dense_retrieve:
        chunk_scores = dense_scores_all(question)
    else:
        chunk_scores = bm25_scores_all(question)

    flat_chunk_docs, flat_texts = [], []
    for d in candidates:
        for cid, ctext in select_relevant_chunks(d, chunk_scores, top_n=chunks_per_doc):
            flat_chunk_docs.append(d)
            flat_texts.append(ctext)

    if not flat_texts:
        return []

    scores = rerank_score(question, flat_texts)

    # doc_id -> (best_score, best_chunk_text): CHỈ giữ chunk điểm cao nhất của mỗi document.
    doc_best = {}
    for d, sc, txt in zip(flat_chunk_docs, scores, flat_texts):
        if d not in doc_best or sc > doc_best[d][0]:
            doc_best[d] = (sc, txt)

    ranked_docs = sorted(doc_best.items(), key=lambda x: x[1][0], reverse=True)[:top_k_docs]
    return [(d, sc, txt) for d, (sc, txt) in ranked_docs]


def build_qa_context(top_chunks):
    """Ghép 5 chunk (mỗi document 1 chunk) thành 1 đoạn context duy nhất cho vit5-large."""
    parts = []
    for i, (doc_id, score, text) in enumerate(top_chunks, start=1):
        parts.append(f"[Văn bản {i} - id {doc_id}] {text}")
    return "\n\n".join(parts)


### 7.0 Tiền xử lý `answer` trong `train2.json` (làm sạch trước khi build context / fine-tune ViLegalQwen3-1.7B-Base)

Trước khi dùng `train2.json` để build context và fine-tune Stage-3, ta làm sạch trường `answer`:

1. **Dọn ký tự rác**: BOM (`\ufeff`), soft-hyphen (`\xad`), NBSP (`\xa0` → dấu cách), chuẩn hoá Unicode **NFC**.
2. **Thay toàn bộ xuống dòng bằng khoảng trắng**: mọi `\n` (kể cả `\r\n`, `\r`) → `" "`; nhiều `\n` liền nhau (kể cả xen khoảng trắng/tab) → chỉ **1** `" "`; sau đó gộp mọi dãy khoảng trắng liên tiếp thành **1** khoảng trắng. Áp dụng cho `answer` và cả `question` (tắt phần `question` bằng `NORMALIZE_QUESTION_NEWLINES=False`). Sau bước này answer là văn bản **1 dòng**, không còn `\n`.
3. **Xoá câu hỏi bị lặp nguyên văn ở đầu answer** (model sẽ học lặp lại câu hỏi nếu để nguyên).
4. **Xoá caption ảnh rác** `(Hình từ Internet)` / `(Hình ảnh từ Internet)` ở bất kỳ vị trí nào.
5. Gộp lại khoảng trắng kép phát sinh sau khi xoá; bỏ các mẫu có answer rỗng sau khi làm sạch.

Kết quả ghi ra `train2_cleaned.json` (cùng thư mục `DATA_DIR`); Cell 22a bên dưới đọc file đã làm sạch này thay cho `train2.json` gốc.

In [ ]:
# Cell 22-pre — Tiền xử lý train2.json (làm sạch `answer`) -> train2_cleaned.json.
# Chỉ dùng CPU/regex, chạy nhanh, KHÔNG cần model nào đã load. Cell 22a (build context + fine-tune
# ViLegalQwen3-1.7B-Base) sẽ đọc TRAIN2_CLEAN_PATH thay vì TRAIN2_PATH gốc.
import json, re, unicodedata

TRAIN2_CLEAN_PATH = "DSC2026-IntelliJQK/data/processed/train2_cleaned.json"
DROP_EMPTY_ANSWER = True   # bỏ mẫu có answer rỗng sau khi làm sạch (target rỗng chỉ còn <eos>, vô nghĩa khi train)
NORMALIZE_QUESTION_NEWLINES = True   # True: áp dụng luôn quy tắc "\n -> ' '" cho trường `question` (không chỉ `answer`)

# ĐÃ THÊM: thay THẾ TOÀN BỘ xuống dòng bằng 1 khoảng trắng. Quy tắc:
#  - Mọi "\n" (kể cả "\r\n", "\r") -> " ".
#  - Nhiều "\n" liền nhau (kể cả xen khoảng trắng/tab giữa các "\n", vd "\n \n\n") -> thay bằng ĐÚNG 1 " ".
#  - Sau đó gộp mọi dãy khoảng trắng liên tiếp còn lại thành ĐÚNG 1 " ".
# Regex `\s*[\r\n]+\s*` khớp cả cụm (khoảng trắng + xuống dòng + khoảng trắng) trong 1 lần nên không
# sinh ra dấu cách kép; bước gộp " +" phía sau chỉ để dọn các dấu cách kép đã có sẵn trong dữ liệu.
_RE_NEWLINE_RUN = re.compile(r"\s*[\r\n]+\s*")


def _newlines_to_space(text: str) -> str:
    """Thay mọi xuống dòng (1 hay nhiều liên tiếp) bằng đúng 1 khoảng trắng, rồi gộp nhiều
    khoảng trắng thành 1. Không strip đầu/cuối chuỗi (để caller tự quyết định)."""
    text = _RE_NEWLINE_RUN.sub(" ", text)
    return re.sub(r" +", " ", text)

# Caption ảnh rác, vd "(Hình từ Internet)", "(Hình ảnh từ internet)" -- ở bất kỳ vị trí nào trong answer.
_RE_IMG_CAPTION = re.compile(r"\(Hình\s*(ảnh)?\s*từ\s*internet\)", flags=re.IGNORECASE)


def _strip_leading_question(text: str, question: str) -> str:
    """Xoá câu hỏi bị lặp NGUYÊN VĂN ở đầu answer (bỏ qua khác biệt về khoảng trắng/xuống dòng và
    dấu '?' cuối câu). Dùng regex khớp theo từng từ của câu hỏi (nối bằng \\s+) thay vì cắt theo độ
    dài chuỗi, nên vẫn đúng khi answer có xuống dòng/khoảng trắng thừa/khoảng trắng đầu chuỗi."""
    q_core = re.sub(r"\s+", " ", question or "").strip().rstrip("?").strip()
    if not q_core:
        return text
    pattern = r"^\s*" + r"\s+".join(re.escape(tok) for tok in q_core.split(" ")) + r"\s*\?*"
    return re.sub(pattern, "", text, count=1).lstrip()


def clean_answer(text: str, question: str) -> str:
    # (Các bước dọn ký tự)
    text = text.replace("\ufeff", "").replace("\xad", "").replace("\xa0", " ")
    text = unicodedata.normalize("NFC", text)

    # ĐÃ THÊM: thay TOÀN BỘ "\n" bằng " " (nhiều "\n" liền nhau -> chỉ 1 " ") và gộp khoảng trắng.
    # Làm NGAY tại đây (trước các bước xoá câu hỏi lặp / caption) để mọi bước sau chỉ còn làm việc
    # trên văn bản 1 dòng. NBSP đã đổi thành " " ở trên nên cũng được gộp chung.
    text = _newlines_to_space(text)

    # xoá câu hỏi bị lặp y nguyên ở đầu answer
    text = _strip_leading_question(text, question)

    # xoá caption ảnh rác "(Hình từ Internet)" ở bất kỳ đâu
    text = _RE_IMG_CAPTION.sub("", text)

    text = re.sub(r" +", " ", text)      # gộp lại dấu cách kép có thể phát sinh sau khi xoá
    return text.strip()                  # (không còn "\n" nào trong text nên không cần dọn cuối dòng nữa)


def preprocess_train2(src_path=TRAIN2_PATH, dst_path=TRAIN2_CLEAN_PATH, drop_empty=DROP_EMPTY_ANSWER):
    raw = load_json(src_path)
    cleaned = {}
    n_changed = n_echo = n_caption = n_dropped = n_nl_answer = n_nl_question = 0
    for qid, v in raw.items():
        v = dict(v)
        # ĐÃ THÊM: question cũng thay "\n" -> " " (tắt bằng NORMALIZE_QUESTION_NEWLINES=False)
        q_old = v.get("question")
        if isinstance(q_old, str) and q_old:
            if "\n" in q_old or "\r" in q_old:
                n_nl_question += 1
            if NORMALIZE_QUESTION_NEWLINES:
                v["question"] = _newlines_to_space(q_old).strip()
        old = v.get("answer")
        if isinstance(old, str) and old:
            if "\n" in old or "\r" in old:
                n_nl_answer += 1
            q = v.get("question", "")
            new = clean_answer(old, q)
            if _RE_IMG_CAPTION.search(old):
                n_caption += 1
            if _strip_leading_question(unicodedata.normalize("NFC", old), q) != unicodedata.normalize("NFC", old).lstrip():
                n_echo += 1
            if new != old:
                n_changed += 1
            v["answer"] = new
        if drop_empty and not (isinstance(v.get("answer"), str) and v["answer"].strip()):
            n_dropped += 1
            continue
        cleaned[qid] = v

    with open(dst_path, "w", encoding="utf-8") as f:
        json.dump(cleaned, f, ensure_ascii=False, indent=2)
    print(f"train2: {len(raw)} mẫu gốc -> {len(cleaned)} mẫu sau làm sạch (bỏ {n_dropped} mẫu answer rỗng)")
    print(f"  answer bị thay đổi: {n_changed} | có câu hỏi lặp ở đầu: {n_echo} | có caption '(Hình từ Internet)': {n_caption}")
    print(f"  có xuống dòng (\\n) -> đã thay bằng ' ': answer {n_nl_answer} | question {n_nl_question}"
          f"{'' if NORMALIZE_QUESTION_NEWLINES else ' (question KHÔNG bị đổi: NORMALIZE_QUESTION_NEWLINES=False)'}")
    print(f"  Đã lưu: {dst_path}")
    return cleaned


train2_cleaned = preprocess_train2()

# Kiểm tra: sau tiền xử lý KHÔNG còn "\n"/"\r" nào trong answer (và question nếu bật cờ).
_bad_ans = [q for q, v in train2_cleaned.items() if "\n" in v["answer"] or "\r" in v["answer"]]
_bad_q = ([q for q, v in train2_cleaned.items() if "\n" in v.get("question", "") or "\r" in v.get("question", "")]
          if NORMALIZE_QUESTION_NEWLINES else [])
assert not _bad_ans and not _bad_q, f"Vẫn còn xuống dòng: answer={_bad_ans[:5]}, question={_bad_q[:5]}"
print("OK: không còn ký tự xuống dòng nào trong train2_cleaned.")

### 7.1 Chuẩn bị dữ liệu huấn luyện cho vit5 từ `train2.json` (inference Stage-1 + Stage-2, KHÔNG train lại retrieval/reranker)

In [ ]:
# Cell 22a — Load train2.json (QA thật) và build context bằng Stage-1 + Stage-2 (chỉ INFERENCE,
# không train lại retrieval/reranker). Định dạng train2.json:
#   {"<qid>": {"question": "...", "answer": "<câu trả lời tham chiếu, dạng chuỗi>"}, ...}

# ĐÃ SỬA: đọc bản train2 ĐÃ LÀM SẠCH (Cell 22-pre -> train2_cleaned.json) thay vì train2.json gốc.
train2_raw = load_json(TRAIN2_CLEAN_PATH)
print("Số mẫu QA trong train2:", len(train2_raw))
sample_qid2 = next(iter(train2_raw))
print(sample_qid2, "->", train2_raw[sample_qid2])

# Cache lại context đã build (Stage-1 + Stage-2 khá tốn GPU) ra file JSON, để không phải chạy lại
# mỗi khi restart kernel -- chỉ cần force_rebuild=True mới build lại từ đầu (vd. khi đổi reranker
# checkpoint / đổi TOP_K_ANSWER_CONTEXT).
VIT5_TRAIN_CACHE_PATH = "DSC2026-IntelliJQK/cache/vit5_train2_context_cache.json"


def build_vit5_training_samples(raw, cache_path=VIT5_TRAIN_CACHE_PATH, top_k_docs=TOP_K_ANSWER_CONTEXT,
                                 stage1_fn=hybrid_retrieve, top_k_stage1=TOP_K_STAGE1, force_rebuild=False):
    """Với MỖI (qid, question, answer) trong train2.json: chạy Stage-1 (stage1_fn) + Stage-2
    (rerank_score) để lấy top_k_docs document liên quan nhất -- MỖI document đúng 1 chunk điểm
    reranker cao nhất (get_top_chunks_for_answer, Cell 22) -- ghép context (build_qa_context) rồi
    tạo 1 sample huấn luyện cho vit5-large: {"qid", "question", "context", "answer", "doc_ids"}.
    """
    if os.path.exists(cache_path) and not force_rebuild:
        cached = load_json(cache_path)
        # ĐÃ THÊM: cache (context Stage-1+Stage-2) có thể được build TỪ TRƯỚC khi có bước tiền xử lý,
        # nên answer trong cache có thể còn bẩn. Context KHÔNG phụ thuộc answer -> giữ nguyên context
        # (khỏi build lại tốn GPU), chỉ ĐỒNG BỘ lại answer từ `raw` (đã làm sạch) và bỏ các qid đã bị
        # loại ở bước tiền xử lý (answer rỗng).
        cached = [s for s in cached if str(s["qid"]) in raw]
        for s in cached:
            s["answer"] = raw[str(s["qid"])]["answer"]
            s["question"] = raw[str(s["qid"])]["question"]   # ĐÃ THÊM: question cũng đã thay "\n" -> " " ở Cell 22-pre
        print(f"Đã nạp {len(cached)} sample huấn luyện vit5 từ cache: {cache_path} (answer đã đồng bộ với bản làm sạch)")
        return cached

    samples = []
    for qid, v in tqdm(raw.items(), desc="Xây dựng context (Stage-1+Stage-2) cho train2"):
        question = v["question"]
        answer = v["answer"]
        top_chunks = get_top_chunks_for_answer(
            question, stage1_fn=stage1_fn, top_k_stage1=top_k_stage1, top_k_docs=top_k_docs
        )
        context = build_qa_context(top_chunks)
        samples.append({
            "qid": str(qid),
            "question": question,
            "context": context,
            "answer": answer,
            "doc_ids": [d for d, _, _ in top_chunks],
        })

    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(samples, f, ensure_ascii=False, indent=2)
    print(f"Đã build và lưu {len(samples)} sample huấn luyện vit5 vào cache: {cache_path}")
    return samples


vit5_train_samples_all = build_vit5_training_samples(train2_raw)
print(vit5_train_samples_all[0])

# ---- ĐÃ SỬA (chống OOM): giải phóng TRIỆT ĐỂ 2 model của Stage-1/Stage-2 (embed_model,
# reranker_model) ngay sau khi build xong context -- từ đây cho tới hết Stage-3 TRAINING
# (cell "Cell 21b" load qa_model + Cell 22c/22d train), pipeline KHÔNG còn cần forward pass nào
# qua embed_model/reranker_model nữa (toàn bộ context train2 đã được cache ở samples/JSON phía
# trên), nên không có lý do giữ chúng trên VRAM trong lúc train qa_model (1.7B tham số).
# reranker_model/embed_model sẽ được NẠP LẠI ở Cell 22e (sau khi train xong) để dùng cho
# inference/submission (Cell 23/24) -- xem cell đó.
import gc

_freed = [n for n in ("reranker_model", "embed_model") if n in globals()]
for _name in ("reranker_model", "embed_model"):
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache() if DEVICE == "cuda" else None
print(f"Đã giải phóng khỏi VRAM: {_freed if _freed else '(không còn model nào cần giải phóng)'} "
      f"-- qa_model (Stage-3) sẽ được load ở cell ngay sau đây, không còn phải \"gánh\" thêm "
      f"embed_model/reranker_model nữa.")
if DEVICE == "cuda":
    print(f"VRAM đang cấp phát: {torch.cuda.memory_allocated() / 1024**3:.2f} GB "
          f"(đã reserve: {torch.cuda.memory_reserved() / 1024**3:.2f} GB)")


Số mẫu QA trong train2: 7000
82051 -> {'question': 'Vận chuyển động vật ra khỏi địa bàn cấp tỉnh mà không có Giấy chứng nhận kiểm dịch động vật, sản phẩm động vật thì bị xử phạt thế nào?', 'answer': 'Căn cứ khoản 3, khoản 5 Điều 17 Nghị định 90/2017/NĐ-CP, được sửa đổi bởi điểm a khoản 9 Điều 3 Nghị định 07/2022/NĐ-CP, khoản 7 Điều 2 Nghị định 04/2020/NĐ-CP quy định về vi phạm quy định chung về Giấy chứng nhận kiểm dịch động vật, sản phẩm động vật vận chuyển ra khỏi địa bàn cấp tỉnh như sau:\nVi phạm quy định chung về Giấy chứng nhận kiểm dịch động vật, sản phẩm động vật vận chuyển ra khỏi địa bàn cấp tỉnh\n1. Phạt tiền từ 4.000.000 đồng đến 5.000.000 đồng đối với hành vi mua bán, tẩy xóa, sửa chữa Giấy chứng nhận kiểm dịch động vật, sản phẩm động vật.\n2. Phạt tiền từ 5.000.000 đồng đến 6.000.000 đồng đối với hành vi cho thuê, cho mượn, thuê, mượn Giấy chứng nhận kiểm dịch động vật, sản phẩm động vật.\n3. Phạt tiền từ 6.000.000 đồng đến 8.000.000 đồng đối với hành vi không có Giấy chứ

In [ ]:
# # Cell 22a2 — CHẨN ĐOÁN độ dài token thực tế của prompt/context và answer trong train2.json
# # (chạy SAU khi build xong vit5_train_samples_all ở Cell 22a). Dùng để chọn QA_MAX_INPUT_LEN /
# # QA_MAX_NEW_TOKENS (Cell 21) SÁT với dữ liệu thật thay vì đoán 4096/2048 -- đây chính là 2 con số
# # quyết định kích thước tensor logits (batch, seq_len, vocab~150k), nguồn OOM chính lúc backward
# # (xem log lỗi: "Tried to allocate 3.75 GiB" khớp gần đúng với 1 tensor logits fp32 kích thước này).
# # Sau khi xem kết quả in ra, quay lại Cell 21, đổi QA_MAX_INPUT_LEN/QA_MAX_NEW_TOKENS cho hợp lý
# # (thường chọn quanh mốc p95-p99 + đệm nhỏ, KHÔNG cần lấy max tuyệt đối vì vài outlier hiếm không
# # đáng để tất cả các batch khác phải trả giá VRAM), rồi chạy lại Cell 21 -> 22c -> 22d.
# import numpy as _np_diag

# _prompt_lens, _answer_lens = [], []
# for _s in tqdm(vit5_train_samples_all, desc="Đo độ dài token (prompt/context + answer)"):
#     _prompt = build_qa_prompt(_s["question"], _s["context"])
#     _prompt_lens.append(len(qa_tokenizer(_prompt, add_special_tokens=False)["input_ids"]))
#     _answer_lens.append(len(qa_tokenizer(_s["answer"].strip(), add_special_tokens=False)["input_ids"]))

# _prompt_lens = _np_diag.array(_prompt_lens)
# _answer_lens = _np_diag.array(_answer_lens)


# def _print_len_stats(name, arr):
#     pcts = [50, 90, 95, 99, 100]
#     vals = _np_diag.percentile(arr, pcts)
#     stats_str = ", ".join(f"p{p}={int(v)}" for p, v in zip(pcts, vals))
#     print(f"{name}: mean={arr.mean():.0f}, {stats_str} (n={len(arr)})")


# print("---- Độ dài token thực tế trong train2.json ----")
# _print_len_stats("Prompt (câu hỏi + context 5 chunk)", _prompt_lens)
# _print_len_stats("Answer (câu trả lời tham chiếu)", _answer_lens)
# print(f"\nGợi ý: đặt QA_MAX_INPUT_LEN quanh p95-p99 của Prompt (hiện tại đang set = {QA_MAX_INPUT_LEN}), "
#       f"QA_MAX_NEW_TOKENS quanh p95-p99 của Answer + đệm ~10-20% (hiện tại đang set = {QA_MAX_NEW_TOKENS}). "
#       f"Nếu p95-p99 thực tế thấp hơn NHIỀU so với giá trị đang set -> đây chính là chỗ đang \"lãng phí\" "
#       f"VRAM cho những token không cần thiết ở HẦU HẾT các batch.")


Đo độ dài token (prompt/context + answer): 100%|██████████| 7000/7000 [00:42<00:00, 162.87it/s]


---- Độ dài token thực tế trong train2.json ----
Prompt (câu hỏi + context 5 chunk): mean=2414, p50=2424, p90=2788, p95=2885, p99=3073, p100=4050 (n=7000)
Answer (câu trả lời tham chiếu): mean=457, p50=409, p90=754, p95=906, p99=1331, p100=3027 (n=7000)

Gợi ý: đặt QA_MAX_INPUT_LEN quanh p95-p99 của Prompt (hiện tại đang set = 3072), QA_MAX_NEW_TOKENS quanh p95-p99 của Answer + đệm ~10-20% (hiện tại đang set = 1536). Nếu p95-p99 thực tế thấp hơn NHIỀU so với giá trị đang set -> đây chính là chỗ đang "lãng phí" VRAM cho những token không cần thiết ở HẦU HẾT các batch.


In [34]:
# Cell 21b — Load qa_model (ntphuc149/ViLegalQwen3-1.7B-Base) LÊN GPU.
# ĐÃ SỬA (chống OOM): cell này CHỦ Ý được đặt SAU cell giải phóng embed_model/reranker_model
# (Cell 22a) -- tại thời điểm này VRAM chỉ còn phải chứa DUY NHẤT qa_model (+ optimizer/LoRA
# adapter sẽ thêm ở Cell 22d), không còn phải "gánh" thêm embed_model (Stage-1) hay
# reranker_model (Stage-2) như trước nữa (2 model đó CHỈ cần cho việc build context ở Cell 22a,
# không cần trong lúc train Stage-3).
#
# ĐÃ THÊM (chống OOM lúc .backward()): load base model ở QLoRA 4-bit (nf4) thay vì bfloat16 đầy
# đủ. Với GPU nhỏ (~15-16GB), phần tốn VRAM nhất khi train causal LM KHÔNG PHẢI base weights (chỉ
# ~3.4GB ở bf16) mà là ACTIVATION + LOGITS: logits có shape
# (batch, seq_len, vocab_size) -- ViLegalQwen3-1.7B-Base có vocab ~151.9k token (cùng tokenizer họ Qwen3),
# nên dù batch nhỏ, 1 forward
# pass vẫn tốn vài GB cho riêng logits (+ bản fp32 dùng để tính loss, + gradient của nó lúc
# backward). Giảm base weights xuống 4-bit (từ ~3.4GB bf16 xuống ~1GB) nhường lại VRAM cho phần
# logits/activation này, kết hợp cùng gradient checkpointing (Cell 22d) và batch nhỏ + grad-accum
# (Cell 22c) để không còn OOM ở bước backward nữa.
QA_LOAD_IN_4BIT = False    # đặt False nếu muốn quay lại load bfloat16 đầy đủ như trước

if QA_LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    _qa_bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
        bnb_4bit_use_double_quant=True,
    )
    qa_model = AutoModelForCausalLM.from_pretrained(
        QA_MODEL_NAME,
        quantization_config=_qa_bnb_config,
        device_map={"": 0} if DEVICE == "cuda" else None,
        trust_remote_code=True,
        **_attn_impl_kwargs(),
    )
else:
    qa_model = AutoModelForCausalLM.from_pretrained(
        QA_MODEL_NAME,
        torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
        trust_remote_code=True,
        **_attn_impl_kwargs(),
    ).to(DEVICE)
qa_model.config.pad_token_id = qa_tokenizer.pad_token_id
qa_model.eval()
print(f"Đã load {QA_MODEL_NAME} (causal LM, kiến trúc Qwen3, "
      f"{'4-bit QLoRA' if QA_LOAD_IN_4BIT else 'bfloat16'}) cho Stage-3 (sinh câu trả lời) -- "
      f"VRAM hiện tại chỉ còn model này (embed_model/reranker_model đã được giải phóng ở cell trước).")
if DEVICE == "cuda":
    print(f"VRAM đang cấp phát: {torch.cuda.memory_allocated() / 1024**3:.2f} GB "
          f"(đã reserve: {torch.cuda.memory_reserved() / 1024**3:.2f} GB)")


Loading weights: 100%|██████████| 311/311 [00:01<00:00, 264.11it/s]


Đã load Qwen/Qwen3-1.7B (causal LM, kiến trúc Qwen3, bfloat16) cho Stage-3 (sinh câu trả lời) -- VRAM hiện tại chỉ còn model này (embed_model/reranker_model đã được giải phóng ở cell trước).
VRAM đang cấp phát: 3.20 GB (đã reserve: 3.22 GB)


In [35]:
# Cell 22b — Tách train/dev cho fine-tune vit5 (giống cách tách dev_samples ở Cell 5, mục 1) --
# dev ở đây dùng để chọn checkpoint tốt nhất theo eval loss (teacher-forcing), KHÔNG liên quan tới
# dev_samples của reranker (Stage-2).
random.shuffle(vit5_train_samples_all)
n_dev_vit5 = max(1, int(0.1 * len(vit5_train_samples_all)))
vit5_dev_samples = vit5_train_samples_all[:n_dev_vit5]
vit5_fit_samples = vit5_train_samples_all[n_dev_vit5:]
print(f"vit5 fit: {len(vit5_fit_samples)} | vit5 dev (offline eval loss): {len(vit5_dev_samples)}")


vit5 fit: 6300 | vit5 dev (offline eval loss): 700


In [ ]:
# Cell 22c — Dataset/DataLoader cho fine-tune ntphuc149/ViLegalQwen3-1.7B-Base (causal LM)
# ĐÃ SỬA (so với vit5-large seq2seq): input giờ là 1 CHUỖI DUY NHẤT = prompt(question, context) nối
# trực tiếp với answer + eos_token (không còn cặp encoder-input/decoder-labels riêng biệt). Loss chỉ
# tính trên phần ANSWER (+eos): phần prompt bị mask bằng -100 trong labels, đúng chuẩn causal-LM
# instruction fine-tuning (không học lại việc "sinh ra" ngữ cảnh/câu hỏi, chỉ học sinh câu trả lời).
from torch.utils.data import Dataset, DataLoader

# ViLegalQwen3-1.7B-Base (~1.7B tham số) nặng hơn vit5-large per-sample khi train causal LM full-
# sequence (prompt + answer đều nằm trong 1 forward) -> giảm batch size, tăng grad-accum tương ứng
# để GIỮ NGUYÊN effective batch = 32 như cấu hình vit5-large cũ.
# ĐÃ SỬA (chống OOM lúc backward): GPU thực tế chỉ có ~15-16GB (không phải 24GB như dự kiến ban
# đầu) -> giảm BATCH_SIZE về 1 (logits (batch, seq_len, vocab~150k) là phần tốn VRAM nhất mỗi
# forward/backward, giảm batch = giảm thẳng theo tỉ lệ; vocab ViLegalQwen3-1.7B-Base ~151.9k token, cùng
# bậc độ lớn), tăng GRAD_ACCUM 8 -> 32 để GIỮ NGUYÊN
# effective batch = 32 (không đổi tốc độ học/số lần update, chỉ đổi VRAM peak mỗi micro-batch).
# LƯU Ý: giá trị này TỪNG bị ghi nhầm là 2 trong 1 bản trước (comment nói đã giảm 2->1 nhưng code
# vẫn là 2) -- đây là fix cho đúng khớp giữa comment và code.
VIT5_BATCH_SIZE = 1
VIT5_GRAD_ACCUM_STEPS = 32   # effective batch = 1 * 32 = 32 (giữ nguyên như trước)


class QADataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        # ĐÃ SỬA: cắt bớt CONTEXT (không cắt cuối prompt) nếu prompt > QA_MAX_INPUT_LEN, xem Cell 21.
        prompt = build_qa_prompt_fit(s["question"], s["context"])
        answer = s["answer"].strip()
        return prompt, answer


def vit5_collate_fn(batch):
    # ĐÃ SỬA: dùng CHUNG QA_MAX_NEW_TOKENS (Cell 21) cho cả train và generate -- không tách ngân
    # sách answer riêng cho train nữa. VRAM lúc train được kiểm soát bằng kỹ thuật logits_to_keep
    # (Cell 22d, chỉ tính logits phần answer) thay vì bằng cách cắt bớt answer ở đây.
    max_total = QA_MAX_INPUT_LEN + QA_MAX_NEW_TOKENS + 1   # +1 cho eos nối sau answer
    input_ids_list, labels_list, answer_ids_list = [], [], []
    for prompt, answer in batch:
        prompt_ids = qa_tokenizer(
            prompt, truncation=True, max_length=QA_MAX_INPUT_LEN, add_special_tokens=False
        )["input_ids"]
        answer_ids = qa_tokenizer(
            answer, truncation=True, max_length=QA_MAX_NEW_TOKENS, add_special_tokens=False
        )["input_ids"] + [qa_tokenizer.eos_token_id]

        # ĐÃ SỬA: prompt đã được build_qa_prompt_fit (Cell 21) rút gọn CONTEXT cho vừa QA_MAX_INPUT_LEN
        # (câu hỏi/"Trả lời:"/header assistant còn nguyên), và answer_ids <= QA_MAX_NEW_TOKENS + eos, nên
        # điều kiện dưới KHÔNG bao giờ xảy ra -- chỉ giữ làm lớp an toàn. Trước đây tokenizer cắt BÊN PHẢI
        # prompt (mất câu hỏi) và nhánh "giữ phần cuối" này gần như không bao giờ chạy.
        if len(prompt_ids) + len(answer_ids) > max_total:
            prompt_ids = prompt_ids[-(max_total - len(answer_ids)):]

        ids = prompt_ids + answer_ids
        labels = [-100] * len(prompt_ids) + answer_ids  # mask phần prompt, chỉ học phần answer(+eos)
        input_ids_list.append(ids)
        labels_list.append(labels)
        answer_ids_list.append(answer_ids)

    max_len = max(len(ids) for ids in input_ids_list)
    pad_id = qa_tokenizer.pad_token_id
    input_ids, attention_mask, labels = [], [], []
    for ids, labs in zip(input_ids_list, labels_list):
        pad_len = max_len - len(ids)
        input_ids.append(ids + [pad_id] * pad_len)
        attention_mask.append([1] * len(ids) + [0] * pad_len)
        labels.append(labs + [-100] * pad_len)

    # ĐÃ THÊM: answer_len (số token THẬT của answer, tính cả eos, KHÔNG tính padding) -- dùng ở
    # Cell 22d để bật kỹ thuật "logits_to_keep" (chỉ tính logits cho phần answer, bỏ qua phần prompt
    # bị mask -100 -- không đổi loss, chỉ đổi cách tính cho rẻ hơn). CHỈ đúng ý nghĩa khi
    # VIT5_BATCH_SIZE == 1 (mỗi batch đúng 1 sample, answer luôn nằm ở cuối chuỗi, không lẫn với
    # sample khác/padding) -- Cell 22d tự kiểm tra điều kiện này trước khi dùng.
    answer_lens = [len(a) for a in answer_ids_list]

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "answer_len": answer_lens[0] if len(answer_lens) == 1 else None,
    }


vit5_train_loader = DataLoader(
    QADataset(vit5_fit_samples), batch_size=VIT5_BATCH_SIZE, shuffle=True, collate_fn=vit5_collate_fn
)


In [ ]:
# Cell 22d — Fine-tune ntphuc149/ViLegalQwen3-1.7B-Base trên train2.json (LoRA + gradient
# checkpointing để tiết kiệm VRAM).
# ĐÃ SỬA (tiêu chí chọn checkpoint TỐT NHẤT): KHÔNG còn dùng eval loss (teacher-forcing) nữa. Sau
# mỗi epoch, model SINH CÂU TRẢ LỜI THẬT (generate(), giống hệt lúc inference) trên vit5_dev_samples,
# rồi tính ĐIỂM METEOR trung bình so với answer tham chiếu (nltk.translate.meteor_score) — checkpoint
# có dev_meteor CAO NHẤT được lưu làm checkpoint tốt nhất, vì METEOR phản ánh sát chất lượng câu trả
# lời sinh ra hơn nhiều so với loss teacher-forcing (loss thấp không đảm bảo câu sinh ra ở chế độ tự
# hồi quy thực tế là tốt).
#
# LƯU Ý khi dùng LoRA:
#   - Chỉ ~0.5-1% tham số được train -> LR cần cao hơn nhiều so với full fine-tune (1e-5 -> 2e-4).
#   - reranker_model.parameters() vẫn truyền cả model (base bị freeze), optimizer sẽ tự bỏ qua
#     các tham số có requires_grad=False vì chúng không có .grad.
from peft import LoraConfig as _LoraConfig, get_peft_model as _get_peft_model, TaskType as _TaskType

VIT5_USE_LORA = True
VIT5_LORA_R = 16
VIT5_LORA_ALPHA = 32
VIT5_LORA_DROPOUT = 0.05
# ĐÃ SỬA: ViLegalQwen3-1.7B-Base dùng kiến trúc Qwen3 (decoder-only) chứ không phải T5 (encoder-
# decoder) như vit5-large -> tên module khác hẳn (không còn q/k/v/o), gồm cả attention lẫn MLP.
VIT5_LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

VIT5_EPOCHS = 3
VIT5_LR = 2e-4 if VIT5_USE_LORA else 1e-5

VIT5_BEST_DIR = "DSC2026-IntelliJQK/weights/vilegalqwen3-1.7b-best"

# ĐÃ THÊM: khi base model đã bị quantize 4-bit (QA_LOAD_IN_4BIT, xem Cell 21b), PHẢI chạy
# prepare_model_for_kbit_training() TRƯỚC KHI gắn LoRA -- hàm này tự làm mọi việc cần thiết cho
# QLoRA: cast layer-norm/embedding sang fp32 (ổn định số học), freeze base weights, BẬT gradient
# checkpointing (kèm use_reentrant=False) và enable_input_require_grads() -- nên KHÔNG cần gọi lại
# 3 dòng đó thêm 1 lần nữa như trước (chỉ cần cho trường hợp KHÔNG quantize, xem nhánh else).
if QA_LOAD_IN_4BIT:
    from peft import prepare_model_for_kbit_training as _prepare_kbit
    qa_model = _prepare_kbit(
        qa_model, use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )

if VIT5_USE_LORA:
    lora_config_vit5 = _LoraConfig(
        task_type=_TaskType.CAUSAL_LM,
        r=VIT5_LORA_R, lora_alpha=VIT5_LORA_ALPHA, lora_dropout=VIT5_LORA_DROPOUT,
        target_modules=VIT5_LORA_TARGET_MODULES, bias="none",
    )
    qa_model = _get_peft_model(qa_model, lora_config_vit5)
    qa_model.print_trainable_parameters()

# ---- Gradient checkpointing: giảm activation memory (đổi lại chậm hơn ~20-30% do phải
# recompute forward trong backward) — càng quan trọng hơn với causal LM 1.7B (prompt+answer nằm
# chung 1 forward dài) so với vit5-large trước đây. Khi QA_LOAD_IN_4BIT=True, việc này ĐÃ được
# prepare_model_for_kbit_training() ở trên xử lý xong -- chỉ còn cần gọi lại cho nhánh KHÔNG
# quantize (bfloat16 đầy đủ). ----
if not QA_LOAD_IN_4BIT:
    qa_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    # use_reentrant=False: cách checkpointing mới hơn của PyTorch, tương thích tốt hơn với LoRA/PEFT
    # (tránh lỗi "None of the inputs have requires_grad=True" và tiết kiệm thêm chút activation
    # memory so với use_reentrant=True mặc định của bản cũ).
    qa_model.enable_input_require_grads()   # bắt buộc khi dùng LoRA + checkpointing (input
                                             # embedding bị freeze -> cần ép require_grad để
                                             # backward chạy qua được)
qa_model.config.use_cache = False       # use_cache không tương thích với checkpointing lúc train;
                                         # được bật/tắt lại quanh mỗi lần generate() để eval METEOR
                                         # (xem trong training loop) và trước inference (Cell 22e)

# ĐÃ SỬA (chống OOM): AdamW thường giữ 2 moment (m, v) ở fp32 cho MỖI tham số trainable --
# PagedAdamW8bit (bitsandbytes) nén optimizer state xuống 8-bit + dùng "paged" memory (tự động
# swap ra CPU khi VRAM căng, giống paging của hệ điều hành) -> giảm đáng kể VRAM baseline, tăng
# thêm chút đầu (headroom) cho tensor logits lớn lúc backward (xem note OOM ở Cell 22c).
import bitsandbytes as bnb
optimizer_vit5 = bnb.optim.PagedAdamW8bit(
    [p for p in qa_model.parameters() if p.requires_grad], lr=VIT5_LR
)

# ĐÃ THÊM (chống OOM, KHÔNG đổi loss/chất lượng training): kỹ thuật "logits_to_keep" -- CausalLM
# forward() (transformers bản mới, đã xác nhận đúng với kiến trúc Qwen3) hỗ trợ tham số
# logits_to_keep=k để CHỈ tính logits ở k vị trí CUỐI của chuỗi, thay vì toàn bộ seq_len. Vì
# phần PROMPT trong labels bị mask -100 (không đóng góp gì vào loss), tensor logits
# (batch, seq_len_TOÀN_BỘ, vocab~150k) hiện tại đang lãng phí phần lớn VRAM để tính logits cho
# những vị trí prompt KHÔNG BAO GIỜ được dùng tới. Với batch_size=1 (VIT5_BATCH_SIZE, xem Cell 22c),
# answer luôn nằm Ở CUỐI chuỗi -- ta chỉ cần logits_to_keep = answer_len + 1 (phần "+1" để có đủ
# logit dự đoán TOKEN ANSWER ĐẦU TIÊN từ vị trí ngay trước nó), rồi tự tính cross-entropy trên đúng
# answer_len vị trí đó bằng tay -- LOSS TÍNH RA GIỐNG HỆT cách cũ (cùng công thức, cùng token, cùng
# ignore_index), chỉ khác ở chỗ không materialize logits cho phần prompt bị bỏ đi.
# Chỉ áp dụng được khi: (a) forward() của qa_model hỗ trợ logits_to_keep, VÀ (b) VIT5_BATCH_SIZE==1
# (batch>1 sẽ có nhiều answer ở nhiều vị trí khác nhau trong chuỗi đã pad -- không còn đơn giản là
# "N vị trí cuối" nữa). Nếu 1 trong 2 điều kiện không thỏa, tự động FALLBACK về cách tính cũ (full
# forward + labels -100), không lỗi, không đổi hành vi.
import inspect as _inspect
import torch.nn.functional as _F

_qa_forward_fn = qa_model.get_base_model().forward if hasattr(qa_model, "get_base_model") else qa_model.forward
_SUPPORTS_LOGITS_TO_KEEP = "logits_to_keep" in _inspect.signature(_qa_forward_fn).parameters
_USE_LOGITS_TO_KEEP = _SUPPORTS_LOGITS_TO_KEEP and VIT5_BATCH_SIZE == 1
print(f"[Tối ưu VRAM] logits_to_keep: forward() hỗ trợ={_SUPPORTS_LOGITS_TO_KEEP}, "
      f"VIT5_BATCH_SIZE==1={VIT5_BATCH_SIZE == 1} -> "
      f"{'BẬT (chỉ tính logits phần answer)' if _USE_LOGITS_TO_KEEP else 'TẮT (dùng full-forward như cũ, fallback an toàn)'}.")


def _qa_forward_step(batch_on_device, answer_len):
    """Trả về loss của 1 micro-batch, giống hệt qa_model(**batch).loss về mặt giá trị, nhưng nếu
    _USE_LOGITS_TO_KEEP=True thì KHÔNG materialize logits cho phần prompt bị mask -100 -- tiết
    kiệm VRAM tỉ lệ với (seq_len_toàn_bộ / answer_len), thường là 5-7 lần dựa trên số liệu thực đo
    ở Cell 22a2 (prompt p50~2424 token so với answer p50~409 token)."""
    if _USE_LOGITS_TO_KEEP and answer_len is not None and answer_len > 0:
        out = qa_model(
            input_ids=batch_on_device["input_ids"],
            attention_mask=batch_on_device["attention_mask"],
            logits_to_keep=answer_len + 1,
        )
        # logits: (1, answer_len+1, vocab) -- vị trí cuối cùng (predict token KHÔNG tồn tại) bị bỏ.
        shift_logits = out.logits[:, :-1, :].contiguous()
        # answer_len token CUỐI của labels chính là answer_ids thật (không có -100 ở đoạn này,
        # xem cách build labels ở Cell 22c) -- dùng thẳng làm target, không cần shift/pad thêm.
        shift_targets = batch_on_device["labels"][:, -answer_len:].contiguous()
        loss = _F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)).float(),
            shift_targets.view(-1),
            ignore_index=-100,
        )
        del out
        return loss
    else:
        out = qa_model(
            input_ids=batch_on_device["input_ids"],
            attention_mask=batch_on_device["attention_mask"],
            labels=batch_on_device["labels"],
        )
        loss = out.loss
        del out
        return loss


# ---- METEOR (nltk) — thay thế hoàn toàn eval loss làm tiêu chí chọn checkpoint ----
import nltk
for _pkg in ["wordnet", "omw-1.4", "punkt", "punkt_tab"]:
    try:
        nltk.download(_pkg, quiet=True)
    except Exception as _e:
        print(f"Cảnh báo: không tải được gói nltk '{_pkg}' ({_e}) -- METEOR vẫn chạy được (match từ "
              f"chính xác), chỉ mất phần match synonym tiếng Anh (không ảnh hưởng nhiều vì answer "
              f"là tiếng Việt).")
from nltk.translate.meteor_score import meteor_score


def _meteor_tokenize(text: str):
    # Dùng lại underthesea word_tokenize (vi_tokenize, Cell 6) để tách từ/từ ghép tiếng Việt cho
    # METEOR, chính xác hơn split() theo khoảng trắng đơn thuần.
    return vi_tokenize(text or "")


@torch.no_grad()
def _generate_dev_answers(samples, max_new_tokens=QA_MAX_NEW_TOKENS, batch_size=16):
    """Sinh câu trả lời THẬT bằng generate() (KHÔNG teacher-forcing) cho 1 tập sample — dùng để
    tính METEOR trên dev mỗi epoch. Causal LM BẮT BUỘC left-padding khi generate theo batch (để các
    token sinh ra mới luôn nằm liền sau phần prompt, không bị lẫn với pad ở giữa).

    ĐÃ SỬA (giảm VRAM + thời gian, vì đây chỉ dùng để SO SÁNH TƯƠNG ĐỐI giữa các epoch/checkpoint,
    không phải sinh submission cuối cùng):
      1) num_beams=2 -> 1 (greedy, do_sample=False): bỏ beam search -- với num_beams=2, GPU phải
         giữ ĐỒNG THỜI 2 chuỗi/sample suốt quá trình decode (KV-cache x2, compute gần x2). Greedy
         đủ để xếp hạng tương đối giữa các epoch, và VRAM tiết kiệm được dùng để tăng batch_size.
      2) Length bucketing: sort sample theo ĐỘ DÀI TOKEN của prompt TRƯỚC khi chia batch, thay vì
         giữ nguyên thứ tự ngẫu nhiên như trước. Trước đây 1 batch có thể vừa chứa prompt ~2000
         token vừa chứa prompt ~4000 token (xem thống kê Cell 22a2) -- cả batch phải pad theo prompt
         DÀI NHẤT trong batch đó, lãng phí compute cho các prompt ngắn hơn. Sort trước giúp mỗi
         batch chỉ gồm các prompt có độ dài gần nhau -> giảm đáng kể padding thừa. Kết quả được map
         lại đúng THỨ TỰ GỐC của `samples` để khớp với answer tham chiếu ở vit5_eval_meteor.
      3) batch_size 8 -> 16: nhờ (1)+(2) giải phóng VRAM, đo thực tế còn dư ngân sách nên tăng batch
         mặc định lên 16 để giảm thêm số vòng lặp.
    """
    qa_model.eval()
    old_padding_side = qa_tokenizer.padding_side
    qa_tokenizer.padding_side = "left"

    # ---- Bucket theo độ dài token của prompt (tokenize CPU, không đụng GPU, rẻ) ----
    # ĐÃ SỬA: build_qa_prompt_fit (Cell 21) cắt bớt CONTEXT thay vì để tokenizer cắt phải (mất câu hỏi).
    prompts_all = [build_qa_prompt_fit(s["question"], s["context"]) for s in samples]
    prompt_lens = [len(qa_tokenizer(p, add_special_tokens=False)["input_ids"]) for p in prompts_all]
    order = sorted(range(len(samples)), key=lambda i: prompt_lens[i])

    answers_by_orig_idx = [None] * len(samples)
    for i in tqdm(range(0, len(order), batch_size), desc="Generate dev answers (METEOR)"):
        batch_idx = order[i:i + batch_size]
        prompts = [prompts_all[j] for j in batch_idx]
        inputs = qa_tokenizer(
            prompts, return_tensors="pt", padding=True, truncation=True,
            max_length=QA_MAX_INPUT_LEN, add_special_tokens=False,
        ).to(DEVICE)
        output_ids = qa_model.generate(
            **inputs, max_new_tokens=max_new_tokens, num_beams=1, do_sample=False,  # greedy (ĐÃ SỬA: 2 -> 1)
            pad_token_id=qa_tokenizer.pad_token_id,
        )
        gen_only = output_ids[:, inputs["input_ids"].shape[1]:]  # bỏ phần prompt echo lại
        for j, ids in zip(batch_idx, gen_only):
            answers_by_orig_idx[j] = qa_tokenizer.decode(ids, skip_special_tokens=True).strip()

    qa_tokenizer.padding_side = old_padding_side
    qa_model.train()
    return answers_by_orig_idx


def vit5_eval_meteor(samples, batch_size=16):  # 8 -> 16: đồng bộ batch_size mặc định với _generate_dev_answers
    """Sinh câu trả lời thật cho `samples` (dev) rồi tính điểm METEOR trung bình so với answer
    tham chiếu -- dùng để CHỌN CHECKPOINT TỐT NHẤT (điểm CÀNG CAO càng tốt), thay cho eval loss."""
    preds = _generate_dev_answers(samples, batch_size=batch_size)
    scores = []
    for s, pred in zip(samples, preds):
        hyp_tokens = _meteor_tokenize(pred)
        if not hyp_tokens:
            scores.append(0.0)
            continue
        ref_tokens = _meteor_tokenize(s["answer"])
        scores.append(meteor_score([ref_tokens], hyp_tokens))
    return float(np.mean(scores)) if scores else 0.0


best_vit5_meteor = -1.0
best_vit5_epoch = -1

for epoch in range(VIT5_EPOCHS):
    qa_model.train()
    qa_tokenizer.padding_side = "right"  # train dùng right-padding (generate ở dưới tự chuyển sang
                                          # "left" rồi trả lại "right" khi xong, xem _generate_dev_answers)
    running_loss = 0.0
    optimizer_vit5.zero_grad()
    pbar = tqdm(vit5_train_loader, desc=f"Fine-tune ViLegalQwen3-1.7B-Base epoch {epoch+1}/{VIT5_EPOCHS}")
    step = 0
    for step, batch in enumerate(pbar):
        # answer_len là số nguyên Python thuần (không phải tensor, xem Cell 22c) -- tách riêng
        # TRƯỚC khi .to(DEVICE), nếu không sẽ lỗi vì int không có .to().
        answer_len = batch.pop("answer_len", None)
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        loss = _qa_forward_step(batch, answer_len)
        (loss / VIT5_GRAD_ACCUM_STEPS).backward()

        # Xoá reference tới loss NGAY sau khi lấy .item() -- logits (batch, seq_len, vocab~150k)
        # là tensor lớn nhất giữ trong bộ nhớ mỗi step (được _qa_forward_step tự del rồi); nếu để
        # biến `loss` sống tới vòng lặp kế tiếp (bị ghi đè) thì có 1 khoảng ngắn CẢ 2 bộ activation
        # (step cũ + step mới) cùng tồn tại -> dễ đẩy VRAM vượt ngưỡng ngay lúc GPU đã gần đầy.
        running_loss += loss.item()
        del loss

        if (step + 1) % VIT5_GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_([p for p in qa_model.parameters() if p.requires_grad], max_norm=1.0)
            optimizer_vit5.step()
            optimizer_vit5.zero_grad()
            # ĐÃ THÊM (chống OOM): dọn cache ngay sau mỗi lần optimizer.step() thật (không phải
            # mỗi micro-batch, để không làm chậm training quá nhiều) -- giảm phân mảnh VRAM tích
            # tụ qua nhiều bước accumulate liên tiếp.
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

        if step % 20 == 0:
            pbar.set_postfix(loss=running_loss / (step + 1))

    if (step + 1) % VIT5_GRAD_ACCUM_STEPS != 0:
        torch.nn.utils.clip_grad_norm_([p for p in qa_model.parameters() if p.requires_grad], max_norm=1.0)
        optimizer_vit5.step()
        optimizer_vit5.zero_grad()

    qa_model.config.use_cache = True    # bật lại KV-cache trong lúc generate() để tính METEOR
    # ĐÃ THÊM (chống OOM lúc eval): dọn VRAM NGAY TRƯỚC generate() beam-search (batch_size x
    # num_beams sequences cùng lúc trong prefill) -- VRAM có thể còn phân mảnh/chưa giải phóng
    # hết từ optimizer.step()/backward() của epoch vừa train xong.
    if DEVICE == "cuda":
        gc.collect()
        torch.cuda.empty_cache()
    dev_meteor = vit5_eval_meteor(vit5_dev_samples, batch_size=16)  # 8 -> 16: đã dư VRAM sau khi bỏ beam search + bucketing
    qa_model.config.use_cache = False   # tắt lại ngay để train tiếp epoch sau (tương thích gradient checkpointing)
    print(f"Epoch {epoch+1}: train_loss(avg)={running_loss/len(vit5_train_loader):.4f} | dev_meteor={dev_meteor:.4f}")

    if dev_meteor > best_vit5_meteor:
        best_vit5_meteor = dev_meteor
        best_vit5_epoch = epoch + 1
        qa_model.save_pretrained(VIT5_BEST_DIR)
        qa_tokenizer.save_pretrained(VIT5_BEST_DIR)
        print(f"  -> dev_meteor cải thiện, đã lưu checkpoint tốt nhất tại {VIT5_BEST_DIR} "
              f"(epoch={best_vit5_epoch}, dev_meteor={best_vit5_meteor:.4f})")

print(f"Fine-tune ViLegalQwen3-1.7B-Base xong. Best dev_meteor={best_vit5_meteor:.4f} (epoch {best_vit5_epoch}), "
      f"checkpoint tại: {VIT5_BEST_DIR}")


trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030
[Tối ưu VRAM] logits_to_keep: forward() hỗ trợ=True, VIT5_BATCH_SIZE==1=True -> BẬT (chỉ tính logits phần answer).


/mnt/mmlab2024nas/trantran/.venv/lib/python3.10/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/trantran/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):
Fine-tune Qwen3-1.7B epoch 1/3:   6%|▌         | 377/6300 [09:31<2:29:40,  1.52s/it, loss=0.86] 


KeyboardInterrupt: 

In [ ]:
# Cell 22e — Nạp lại checkpoint ViLegalQwen3-1.7B-Base TỐT NHẤT (VIT5_BEST_DIR, chọn theo METEOR
# CAO NHẤT trên dev -- xem Cell 22d) vào qa_model trước khi dùng để inference -- biến qa_model
# trong bộ nhớ hiện đang là trọng số EPOCH CUỐI, KHÔNG PHẢI epoch có METEOR tốt nhất.
import gc as _gc

if VIT5_USE_LORA:
    from peft import PeftModel as _PeftModel
    del qa_model
    _gc.collect()
    torch.cuda.empty_cache() if DEVICE == "cuda" else None
    if QA_LOAD_IN_4BIT:
        from transformers import BitsAndBytesConfig as _BnbConfigE
        _qa_bnb_config_e = _BnbConfigE(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
            bnb_4bit_use_double_quant=True,
        )
        qa_model = AutoModelForCausalLM.from_pretrained(
            QA_MODEL_NAME, quantization_config=_qa_bnb_config_e,
            device_map={"": 0} if DEVICE == "cuda" else None, trust_remote_code=True,
            **_attn_impl_kwargs(),
        )
    else:
        qa_model = AutoModelForCausalLM.from_pretrained(
            QA_MODEL_NAME, torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
            trust_remote_code=True,
            **_attn_impl_kwargs(),
        ).to(DEVICE)
    qa_model = _PeftModel.from_pretrained(qa_model, VIT5_BEST_DIR)
    if QA_LOAD_IN_4BIT:
        # ĐÃ SỬA: KHÔNG merge_and_unload() khi base là 4-bit -- merge cần dequantize tạm thời về
        # full-precision (tốn thêm VRAM ngay lúc merge, dễ OOM lần nữa) và không ổn định với mọi
        # phiên bản peft/bitsandbytes. Giữ nguyên PeftModel wrapper: generate()/forward() vẫn hoạt
        # động đúng như bình thường (adapter LoRA được cộng vào output on-the-fly), chỉ chậm hơn
        # KHÔNG đáng kể so với model đã merge.
        qa_model.config.use_cache = True
    else:
        # Merge LoRA vào base model để generate() không cần đi qua PeftModel wrapper nữa (đơn giản/nhanh hơn)
        qa_model = qa_model.merge_and_unload()
        qa_model.config.use_cache = True   # bật lại KV-cache cho generate() (Stage-3 inference), đã tắt lúc train
else:
    del qa_model
    _gc.collect()
    torch.cuda.empty_cache() if DEVICE == "cuda" else None
    qa_model = AutoModelForCausalLM.from_pretrained(
        VIT5_BEST_DIR, torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
        trust_remote_code=True,
        **_attn_impl_kwargs(),
    ).to(DEVICE)

qa_model.config.pad_token_id = qa_tokenizer.pad_token_id
qa_tokenizer.padding_side = "left"   # generate() với causal LM BẮT BUỘC left-padding
qa_model.eval()
print(f"Đã nạp checkpoint ViLegalQwen3-1.7B-Base TỐT NHẤT (epoch {best_vit5_epoch}, "
      f"dev_meteor={best_vit5_meteor:.4f}) vào qa_model -- từ đây trở đi generate_answer()/submission "
      f"(Cell 23/24) đều dùng checkpoint này.")

# ---- ĐÃ THÊM: NẠP LẠI embed_model (Stage-1) + reranker_model (Stage-2) -- 2 model này đã bị
# xoá khỏi VRAM ở Cell 22a (ngay trước khi train Stage-3) để tiết kiệm VRAM lúc train. Train
# Stage-3 xong rồi -> generate_answer()/generate_submission() (Cell 23/24) vẫn cần gọi lại
# hybrid_retrieve() (dùng embed_model) và rerank_score() (dùng reranker_model) để lấy context cho
# câu hỏi MỚI (không có trong cache train2) -- nếu không nạp lại ở đây, Cell 23/24 sẽ lỗi
# NameError vì 2 biến này không còn tồn tại.
if "embed_model" not in globals():
    torch.cuda.empty_cache() if DEVICE == "cuda" else None
    embed_model = SentenceTransformer(
        EMBED_MODEL_NAME, device=DEVICE,
        model_kwargs={"torch_dtype": torch.float16, **_attn_impl_kwargs()},
    )
    embed_model.max_seq_length = 2048
    print(f"Đã nạp lại embed_model ({EMBED_MODEL_NAME}) cho Stage-1 (hybrid_retrieve) lúc inference.")

if "reranker_model" not in globals():
    reranker_tokenizer = AutoTokenizer.from_pretrained(RERANKER_NAME)
    _base_reranker = AutoModelForSequenceClassification.from_pretrained(
        RERANKER_NAME, torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
        **_attn_impl_kwargs(),
    ).to(DEVICE)
    if os.path.isdir(MERGED_DIR):
        del _base_reranker
        torch.cuda.empty_cache() if DEVICE == "cuda" else None
        reranker_model = AutoModelForSequenceClassification.from_pretrained(
            MERGED_DIR, torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
            **_attn_impl_kwargs(),
        ).to(DEVICE)
    elif os.path.isdir(FT_RERANKER_DIR):
        reranker_model = PeftModel.from_pretrained(_base_reranker, FT_RERANKER_DIR).to(DEVICE)
    elif os.path.isdir(BEST_DIR):
        reranker_model = PeftModel.from_pretrained(_base_reranker, BEST_DIR).to(DEVICE)
    else:
        reranker_model = _base_reranker
    reranker_model.eval()
    print(f"Đã nạp lại reranker_model ({RERANKER_NAME}, fine-tuned) cho Stage-2 (rerank_score) lúc inference.")

if DEVICE == "cuda":
    print(f"VRAM đang cấp phát (qa_model + embed_model + reranker_model, mode inference): "
          f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB "
          f"(đã reserve: {torch.cuda.memory_reserved() / 1024**3:.2f} GB)")


### 7.2 Inference: sinh câu trả lời bằng vit5 đã fine-tune, xuất `submission.json`

In [ ]:
# Cell 23 — Sinh câu trả lời bằng ViLegalQwen3-1.7B-Base (causal LM) THEO BATCH, cùng cách làm với
# _generate_dev_answers (Cell 22d): left-padding + length bucketing + greedy (num_beams=1).
# Context vẫn là 5 chunk (1 chunk/document, theo điểm reranker cao nhất -- get_top_chunks_for_answer, Cell 22).
#
# ĐÃ SỬA: trước đây generate_answer() sinh TỪNG câu hỏi một (batch=1) -> GPU chạy decode với
# batch=1 (memory-bound) rất phí. Giờ tách làm 2 pha:
#   Pha 1 (retrieval): với MỖI câu hỏi chạy Stage-1 + Stage-2 (get_top_chunks_for_answer) -> prompt.
#                      Pha này KHÔNG batch được theo câu hỏi (BM25/dense/rerank chạy riêng từng câu).
#   Pha 2 (generation): gom TẤT CẢ prompt, sort theo độ dài token, chia batch rồi generate().
#                       Đây mới là phần tốn thời gian nhất (tới QA_MAX_NEW_TOKENS token/câu) nên
#                       batch ở đây cho tốc độ tăng rõ rệt.
#
# KHÁC _generate_dev_answers (có chủ đích, vì đây là inference thật chứ không phải eval trong lúc train):
#   1) KHÔNG gọi qa_model.train() ở cuối -- trả lại đúng chế độ ban đầu (eval). Nếu ép train(),
#      lora_dropout sẽ bật lại lúc generate (khi qa_model còn là PeftModel, vd QA_LOAD_IN_4BIT=True).
#   2) Sort GIẢM dần theo độ dài prompt (batch nặng nhất chạy TRƯỚC) -> nếu OOM thì lộ ra ngay ở
#      batch đầu thay vì sau nhiều giờ chạy.
#   3) Có FALLBACK OOM: batch nào hết VRAM thì tự chia đôi rồi chạy lại. Lý do: ở đây embed_model +
#      reranker_model + qa_model cùng nằm trên GPU (Cell 22e nạp lại), trong khi lúc eval dev ở
#      Cell 22d thì 2 model kia đã bị giải phóng -> VRAM còn dư ÍT HƠN, nên batch mặc định = 8 (không phải 16).
import gc

INFER_BATCH_SIZE = 16


@torch.no_grad()
def _generate_from_prompts(prompts, max_new_tokens=QA_MAX_NEW_TOKENS, batch_size=INFER_BATCH_SIZE,
                            desc="Generate answers", progress=True):
    """Sinh câu trả lời (greedy) cho list prompt theo batch. Trả về list[str] CÙNG THỨ TỰ với `prompts`."""
    if not prompts:
        return []

    was_training = qa_model.training
    old_padding_side = qa_tokenizer.padding_side
    qa_model.eval()
    qa_tokenizer.padding_side = "left"   # causal LM BẮT BUỘC left-padding khi generate theo batch
    try:
        # ---- Bucket theo độ dài token của prompt (CPU, rẻ). Chỉ giữ len, không giữ list id để đỡ RAM. ----
        prompt_lens = []
        for i in range(0, len(prompts), 256):
            enc = qa_tokenizer(prompts[i:i + 256], add_special_tokens=False,
                               truncation=True, max_length=QA_MAX_INPUT_LEN)
            prompt_lens.extend(len(ids) for ids in enc["input_ids"])
        order = sorted(range(len(prompts)), key=lambda i: prompt_lens[i], reverse=True)

        def _run(idx_list):
            inputs = qa_tokenizer(
                [prompts[j] for j in idx_list], return_tensors="pt", padding=True, truncation=True,
                max_length=QA_MAX_INPUT_LEN, add_special_tokens=False,
            ).to(DEVICE)
            output_ids = qa_model.generate(
                **inputs, max_new_tokens=max_new_tokens, num_beams=1, do_sample=False,  # greedy
                use_cache=True, pad_token_id=qa_tokenizer.pad_token_id,
            )
            gen_only = output_ids[:, inputs["input_ids"].shape[1]:]  # bỏ phần prompt echo lại
            return [qa_tokenizer.decode(ids, skip_special_tokens=True).strip() for ids in gen_only]

        state = {"bs": batch_size}   # cỡ batch hiện hành -- tự giảm (và GIỮ NGUYÊN mức đã giảm) sau mỗi lần OOM

        def _run_safe(idx_list):
            try:
                return _run(idx_list)
            except torch.cuda.OutOfMemoryError:
                if len(idx_list) == 1:
                    raise   # 1 prompt đơn lẻ cũng OOM -> không còn gì để chia nhỏ nữa
            # Tới đây tức là OOM và còn chia nhỏ được. Dọn VRAM SAU KHI đã thoát khỏi khối except
            # (nếu dọn bên trong, traceback vẫn còn giữ tham chiếu tới các tensor của lần chạy hỏng).
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            mid = len(idx_list) // 2
            state["bs"] = max(1, mid)   # các batch SAU dùng luôn cỡ nhỏ hơn, khỏi phải OOM lại từng batch
            print(f"[OOM] batch {len(idx_list)} prompt hết VRAM -> chia đôi ({mid} + {len(idx_list) - mid}), "
                  f"các batch sau dùng batch_size={state['bs']}.")
            return _run_safe(idx_list[:mid]) + _run_safe(idx_list[mid:])

        answers = [None] * len(prompts)
        pbar = tqdm(total=len(order), desc=desc, disable=not progress)
        i = 0
        while i < len(order):
            batch_idx = order[i:i + state["bs"]]
            for j, ans in zip(batch_idx, _run_safe(batch_idx)):
                answers[j] = ans           # map lại đúng THỨ TỰ GỐC
            i += len(batch_idx)
            pbar.update(len(batch_idx))
        pbar.close()
        return answers
    finally:
        qa_tokenizer.padding_side = old_padding_side
        if was_training:
            qa_model.train()


@torch.no_grad()
def generate_answers_batch(questions, top_k_docs=TOP_K_ANSWER_CONTEXT, stage1_fn=hybrid_retrieve,
                            top_k_stage1=TOP_K_STAGE1, max_new_tokens=QA_MAX_NEW_TOKENS,
                            batch_size=INFER_BATCH_SIZE, desc="Stage-3 QA", progress=True):
    """Trả về (answers, all_top_chunks), cả hai là list CÙNG THỨ TỰ với `questions`.
    all_top_chunks[i]: list[(doc_id, score, chunk_text)] -- tối đa top_k_docs document, MỖI document
    đúng 1 chunk (điểm reranker cao nhất) dùng làm context cho câu hỏi thứ i."""
    # ---- Pha 1: retrieval từng câu (Stage-1 + Stage-2) -> build prompt ----
    prompts, all_top_chunks = [], []
    for q in tqdm(questions, desc=f"{desc} | retrieval", disable=not progress):
        top_chunks = get_top_chunks_for_answer(
            q, stage1_fn=stage1_fn, top_k_stage1=top_k_stage1, top_k_docs=top_k_docs
        )
        all_top_chunks.append(top_chunks)
        # ĐÃ SỬA: cắt bớt CONTEXT (không cắt cuối prompt) -- giống hệt lúc train/dev, xem Cell 21.
        prompts.append(build_qa_prompt_fit(q, build_qa_context(top_chunks)))

    # ---- Pha 2: generate theo batch ----
    answers = _generate_from_prompts(
        prompts, max_new_tokens=max_new_tokens, batch_size=batch_size,
        desc=f"{desc} | generate", progress=progress,
    )
    return answers, all_top_chunks


def generate_answer(question, top_k_docs=TOP_K_ANSWER_CONTEXT, stage1_fn=hybrid_retrieve,
                     top_k_stage1=TOP_K_STAGE1, max_new_tokens=QA_MAX_NEW_TOKENS):
    """Giữ nguyên API cũ cho 1 câu hỏi lẻ: trả về (answer_text, top_chunks). Thực chất chỉ là
    generate_answers_batch() với 1 câu hỏi, để CHỈ CÓ MỘT đường code sinh câu trả lời."""
    answers, all_top_chunks = generate_answers_batch(
        [question], top_k_docs=top_k_docs, stage1_fn=stage1_fn, top_k_stage1=top_k_stage1,
        max_new_tokens=max_new_tokens, batch_size=1, progress=False,
    )
    return answers[0], all_top_chunks[0]


In [ ]:
# Cell 24 — Sinh file submission.json CUỐI CÙNG của pipeline (Stage-3, định dạng QA)
#
# Đúng định dạng đề bài yêu cầu cho tác vụ sinh câu trả lời:
#   {"<qid>": {"answer": "<câu trả lời dạng chuỗi>"}, ...}
# (KHÔNG còn là list document_id như submission của Stage-2 cũ nữa — luồng đó đã bị xoá.)

def generate_submission(test_path, top_k_docs=TOP_K_ANSWER_CONTEXT,
                         stage1_fn=hybrid_retrieve, top_k_stage1=TOP_K_STAGE1,
                         batch_size=INFER_BATCH_SIZE):
    """Sinh submission Stage-3 cho một tệp test bất kỳ (public hoặc private).
    ĐÃ SỬA: sinh câu trả lời THEO BATCH (generate_answers_batch, Cell 23) thay vì từng câu một.
    Pha 1: với MỖI câu hỏi lấy top_k_docs document (mặc định 5) theo điểm reranker, mỗi document
    đúng 1 chunk điểm cao nhất (get_top_chunks_for_answer) rồi ghép prompt. Pha 2: gom toàn bộ
    prompt, sort theo độ dài, generate theo batch (num_beams=1). Thứ tự qid trong kết quả GIỮ
    NGUYÊN như file test. Trả về dict {"<qid>": {"answer": "..."}}."""
    test_raw = load_json(test_path)
    qids = [str(qid) for qid in test_raw.keys()]
    questions = [v["question"] for v in test_raw.values()]
    answers, _all_top_chunks = generate_answers_batch(
        questions, top_k_docs=top_k_docs, stage1_fn=stage1_fn, top_k_stage1=top_k_stage1,
        batch_size=batch_size, desc=f"Stage-3 QA {os.path.basename(test_path)}",
    )
    return {qid: {"answer": ans} for qid, ans in zip(qids, answers)}


def save_submission(submission, out_dir="DSC2026-IntelliJQK/outputs"):
    json_path = os.path.join(out_dir, "submission_7vileg17.json")
    zip_path = os.path.join(out_dir, "submission_7vileg17.zip")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=2)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(json_path, arcname="submission_7vileg17.json")
    print("Đã lưu:", json_path, "và", zip_path)


# Cell 25 — Chạy dự đoán cho Public Test (đổi sang PRIVATE_TEST_PATH khi vào giai đoạn Private)
if os.path.exists(PRIVATE_TEST_PATH):
    private_submission = generate_submission(PRIVATE_TEST_PATH)
    save_submission(private_submission)
    # xem thử vài kết quả đầu tiên
    for qid in list(private_submission.keys())[:3]:
        print(qid, private_submission[qid])
else:
    print(f"Chưa có file {PRIVATE_TEST_PATH} — cập nhật DATA_DIR khi đề bài công bố dữ liệu chính thức.")
